In [1]:
# chdir removed: nbconvert/Jupyter already start in this notebook's own directory
# (scripts/figures/), which is what the relative ../../nets and ../../figures paths assume.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import graph_tool.all as gt
import pandas as pd
import os
from matplotlib.colors import to_hex
import matplotlib
import warnings
import random
import re
import networkx as nx
import scipy as sp
import sys
# '..' is scripts/ (net_functions.py); '../pooling' holds the canonical
# lfc_data_loader.py; '../LFC' holds extract_lfc_csv.py, whose
# get_graph_props / get_selected_gains this notebook reuses so that the
# figures derive their numbers exactly the way the CSVs were built.
sys.path.insert(0, '..')
sys.path.insert(0, '../pooling')
sys.path.insert(0, '../LFC')
from lfc_data_loader import pooled_gains
matplotlib.use('Agg')
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.patches import FancyArrowPatch

warnings.filterwarnings("ignore")
plt.rc('text', usetex=True)
plt.rc('text.latex', preamble = r'\usepackage{mathptmx}')

def get_sorted_filenames(directory):
    """
    Natural-sort the .gt filenames in `directory`, returning their stems.

    Sorts by a digit-aware key but returns the ORIGINAL stems, never a name
    rebuilt from the parsed parts: reassembling would drop the zero-padding
    in the k{k}_p{p:.6f} names ('k16_p1.000000' -> 'k16_p1.0') and the
    caller would then try to load a file that does not exist.
    """
    def sort_key(filename):
        stem = os.path.splitext(filename)[0]
        parts = re.split(r'(\d+)', stem)
        return [int(part) if part.isdigit() else part for part in parts]

    return [os.path.splitext(f)[0] for f in sorted(os.listdir(directory), key=sort_key)]

def custom_mean(x):
    # Fill NaN with 0 for calculation
    x_filled = x.fillna(0)
    
    # If sum is 0, return NaN
    if x_filled.sum() == 0:
        return np.nan
    
    # Calculate mean over all seeds (including NaN positions treated as 0)
    return x_filled.sum() / len(x)

# --- p-selection and p-keyed graph lookup -------------------------------
#
# (1) The LFC panels plot 4 of the 100 p-values and key their legend on
#     C | T | l | R_g, so the four picks need to be spread across the
#     realized clustering range rather than across p. SPECIAL_P below holds
#     indices at four evenly separated CC values, with ws matched to mhk's
#     CC so the two networks line up curve for curve. The values are BAKED
#     IN, not recomputed per run; regenerate them with
#     `python scripts/figures/pick_special_p.py` (repo root) if the data
#     changes.
#
# (2) A .gt file is located by its stored `probability` property, never by
#     position in a sorted listing. Only the k{k}_p{p:.6f} names sort into
#     ascending-p order; the ws graphs are named by generation ID, which
#     natural-sorts by timestamp instead (45 of 99 adjacent pairs inverted
#     at k=16). Indexing positionally would pair a legend row with the
#     wrong eigenvalue histogram. gt_path_for_p() does the lookup.

SPECIAL_P = {
    # (net, k, N) ->  indices into the ascending-p axis, LOW CC -> HIGH CC.
    #
    # mhk sets the reference CC values: 4 points spaced evenly across its
    # own reachable range, which is the narrower of the two and therefore
    # the binding constraint. ws is then matched to those SAME CC values,
    # so a given curve position means the same clustering in both networks.
    # Matching on p would be meaningless -- p is a rewiring probability in
    # ws and a triad-formation probability in mhk.
    ('mhk', 16,  240): [ 0, 50, 83, 99],   # CC 0.122 / 0.218 / 0.318 / 0.413
    ('ws',  16,  240): [52, 37, 26, 17],   # CC 0.127 / 0.222 / 0.318 / 0.414
    ('mhk',  8,  240): [ 1, 42, 78, 99],   # CC 0.085 / 0.240 / 0.394 / 0.552
    ('ws',   8,  240): [54, 31, 15,  4],   # CC 0.085 / 0.234 / 0.393 / 0.548
    # ke is unchanged: LFC ke has a single p per seed, so an index pick
    # is not meaningful there (tracked separately for S16).
    # S17 / S18: mhk at larger N. CC(p) flattens as N grows -- at k=16 the
    # reachable range falls from [0.122, 0.413] at N=240 to [0.014, 0.368]
    # at N=5000 -- so the N=240 indices do not transfer. Solved on each N's
    # own curve (python pick_special_p.py).
    ('mhk', 16, 1000): [ 0, 50, 81, 99],   # CC 0.050 / 0.154 / 0.258 / 0.362
    ('mhk', 16, 5000): [ 0, 52, 82, 99],   # CC 0.014 / 0.133 / 0.247 / 0.368
    ('ke',  16,  240): [20, 90, 97, 99],
    ('ke',   8,  240): [20, 90, 97, 99],
}

_GT_BY_P_CACHE = {}


def gt_path_for_p(directory, p, tol=1e-6):
    """Path to the .gt in `directory` whose stored probability == p.

    Never index .gt files positionally: only the k{k}_p{p:.6f} names sort
    by p, and the ws graphs are named by generation ID instead.
    """
    if directory not in _GT_BY_P_CACHE:
        mapping = {}
        for f in sorted(os.listdir(directory)):
            if not f.endswith('.gt'):
                continue
            m = re.search(r'_p([0-9.]+)\.gt$', f)
            if m:
                mapping[float(m.group(1))] = directory + f
            else:
                # {ID}_{jobid}_{taskid}.gt -- p only exists inside the file
                g = gt.load_graph(directory + f)
                mapping[float(g.gp.probability)] = directory + f
                del g
        _GT_BY_P_CACHE[directory] = mapping
    mapping = _GT_BY_P_CACHE[directory]
    hit = min(mapping, key=lambda q: abs(q - p))
    if abs(hit - p) > tol:
        raise KeyError(f'no .gt with probability {p} in {directory} '
                       f'(closest {hit})')
    return mapping[hit]


In [3]:
import os
import warnings

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import FancyArrowPatch
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

from lfc_data_loader import pooled_props, pooled_polarization

warnings.filterwarnings("ignore")
plt.rc('text', usetex=True)
plt.rc('text.latex', preamble=r'\usepackage{mathptmx}')

ROOT = '../../nets'
MODEL = 'LTM'
NODES = 1000
K = 16
CASCADES = list(map(str, np.round(np.linspace(0.1, 0.9, 9), 1)))
SUB_PLOT_TAG = 'abcdefghij'


def pooled_mpol(network, k, t_b, perc, root=ROOT, th_exclude=(0.206000, 0.238667)):
    '''
    Pools run_ltm_cascade.py's per-seed polarization CSVs across every
    realization for (network, k, t_b, perc). Two-stage aggregation, indexed
    by (realization, network, p, th, seed) where 'realization' is the seed
    directory's own basename (e.g. '16_seed13', added by
    pooled_polarization) and 'seed' is the seed NODE within that realization
    (not the realization/seed directory).

    Grouped by 'realization', not the graph's own 'ID' (an
    int(time.time()*1000) millisecond timestamp stamped at generation time
    by generate_ltm.py) - 'ID' can collide between two
    different seeds generated in the same millisecond under concurrent
    SLURM array submission, so grouping by it would silently merge two
    distinct realizations into one. Confirmed via inventory: 9 such
    collisions out of 400 graph-writes in the mhk sweep alone (100 seeds x
    4 p-values), which would undercount e.g. p=0.0's realization count from
    100 to 98.
    'realization' can't collide this way since it's just the (already
    filesystem-unique) seed directory name.

    1. custom_mean collapses each realization's own selected seed-nodes
       (grouping by 'realization', i.e. within one realization) into a
       single per-realization mean, which is the quantity a
       single-realization SM figure would show.
    2. Only THEN do we take mean/std of those ~100 per-realization means,
       grouped by (p, th, network), for the +/- 1 std band.

    This matters because a single one-stage pool (mean/std taken directly
    across every seed-node from every realization at once) conflates two
    different sources of spread: node-to-node heterogeneity
    *within* one realization's top/bottom-percent selector group, and
    realization-to-realization variability *across* the ~100 independently
    generated networks. For ws (exactly k-regular by construction - every
    node has identical degree) the two are the same thing, so it didn't
    matter. For mhk (heavy-tailed degree distribution: at k=16 the top 10%
    selector alone spans seed-node degree ~53-81), the one-stage pool's
    spread was dominated by which hub sizes happened to be in the
    top/bottom slice, not by how reproducible the cascade behavior is
    across realizations - inflating the +/- 1 std band on Figures S3-S6
    far beyond what S7-S10 (ws) show, even though the underlying
    realization-to-realization reproducibility is much closer between the
    two once node heterogeneity is factored out first (median CV/std-over-
    mean at the 50%-cascade-size point drops from ~0.43-0.56 to ~0.14-0.17
    for mhk's four selectors, and ~0.08-0.10 to ~0.02 for ws's).

    `th_exclude` defaults to the two threshold values the SM cascade-grid
    figures (S3-S10) drop (0.206, 0.238667). Figures
    1/4/7/S11/S12 instead drop th=0.304 - pass th_exclude=(0.304,) for those.
    '''
    polarization = pooled_polarization(MODEL, NODES, network, k, t_b, perc, root=root)
    polarization.set_index(['realization', 'network', 'p', 'th', 'seed'], inplace=True)
    polarization = polarization[~polarization.index.get_level_values('th').isin(th_exclude)]

    per_realization = polarization.groupby(['realization', 'network', 'p', 'th']).agg(custom_mean)
    grouped = per_realization.groupby(['p', 'th', 'network'])
    return grouped.mean(), grouped.std()


def readiness(network, k, t_b, perc, root=ROOT):
    '''How many of the 100 realization seeds have a polarization CSV for
    this (network, k, t_b, perc) so far.'''
    import glob
    perc_label = int(perc) if float(perc).is_integer() else perc
    sign = '-' if t_b == 'bot' else ''
    label = f'{t_b}{sign}{perc_label}'
    seed_dirs = sorted(glob.glob(f'{root}/{MODEL}/{NODES}/{network}/{k}_seed*'))
    seed_dirs = [d for d in seed_dirs if os.path.isdir(d)]
    have = sum(1 for d in seed_dirs if os.path.exists(f'{d}_{label}.csv'))
    return have, len(seed_dirs)

In [4]:
def _corr_label(p_props):
    C_label = f"{p_props['CC'].mean():.2f}"
    T_label = f"{p_props['T'].mean():.2f}"
    l_label = f"{p_props['SP'].mean():.1f}"
    r_label = f"{(p_props['Rg'].mean() / 1000):.1f}" + r'\! \times \! 10^{3}'
    return r'${}  |  {}  |  {} \,  |  {} $'.format(C_label, T_label, l_label, r_label)


def generate_sm_figure(network, t_b, perc, counter, hist_ylim, inset_ylim, inset_yticks,
                        arrow_start, arrow_end, root=ROOT, fig_path='../../figures'):
    '''
    Pooled (+/- 1 std) SM cascade-grid figure (S3-S10): 9 polarization-speed-vs-threshold panels
    (one per cascade size 10%-90%) plus one normalized-Laplacian-eigenvalue
    density panel, for one (network, t_b, perc) node-selector combo, pooled
    across every realization seed found under `root`.

    The eigenvalue panel reads precompute_eig_norm.py's precomputed
    '{k}_p{p:.6f}_eig_norm.npy' (every seed's normalized-Laplacian spectrum
    already concatenated per p) rather than loading .gt files directly -
    pooling eigenvalues the way lfc_data_loader.pooled_eig_laplacians() does
    would mean opening ~100 individual .gt files per p interactively, which
    is exactly what that function's docstring warns not to do on a login
    node; precompute_eig_norm.py already did that work as a SLURM job.
    '''
    have, total = readiness(network, K, t_b, perc, root=root)
    print(f'{network} k={K} {t_b}{perc}%: {have}/{total} realization seeds ready')

    props = pooled_props(MODEL, NODES, network, K, root=root)
    mean, std = pooled_mpol(network, K, t_b, perc, root=root)
    # Order curves/legend entries by increasing clustering coefficient (mean
    # CC across all pooled seeds for that p), not by p itself - the two
    # orderings can differ since CC vs p need not be monotonic.
    unordered_p = mean.index.get_level_values('p').unique()
    cc_by_p = {p: props.xs(p, level='p')['CC'].mean() for p in unordered_p}
    probabilities = sorted(unordered_p, key=lambda p: cc_by_p[p])

    fig, axs = plt.subplots(figsize=(5.5 * 1.5 * 2, 2.31 * 6.5), ncols=3, nrows=4,
                             sharex=False, sharey=False, tight_layout=True)
    fig.subplots_adjust(wspace=0.4)

    axins = inset_axes(axs[3][1], width=1.5, height=1, loc='upper left',
                        bbox_to_anchor=(0.04, 1), bbox_transform=axs[3][1].transAxes)

    for i, cas in enumerate(CASCADES):
        ax = axs[i // 3, i % 3]
        ax.get_xaxis().get_major_formatter().set_scientific(False)

        for idx, p in enumerate(probabilities):
            label_string = _corr_label(props.xs(p, level='p'))

            m = mean.loc[(p, slice(None), network), cas].droplevel(['p', 'network']).sort_index()
            s = std.loc[(p, slice(None), network), cas].droplevel(['p', 'network']).sort_index()
            th = m.index.values
            lo = np.clip(m.values - s.values, 1e-6, None)
            hi = m.values + s.values

            color = 'black' if idx >= 4 else None
            line, = ax.plot(th, m.values, ls='-', label=label_string, color=color, linewidth=2)
            ax.fill_between(th, lo, hi, alpha=0.2, color=line.get_color())

        ax.text(-0.02, 1.03, fr'\textbf{{({SUB_PLOT_TAG[i]})}} Cascade size: {int(float(cas) * 100)}\% activated nodes',
                transform=ax.transAxes, fontsize=15)
        ax.set_yscale('log')
        ax.set_ylabel(r'Polarization Speed $(v)$', labelpad=2.5, fontsize=14)
        ax.set_xlabel(r'Threshold $( \theta )$', labelpad=2, math_fontfamily='cm', fontsize=14)
        ax.set_ylim([3e-5, 1.12])
        ax.set_xlim([-0.02, 0.52])
        ax.set_xticks([0, 0.1, 0.2, 0.3, 0.4, 0.5])
        ax.tick_params(axis='x', labelsize=13)
        ax.tick_params(axis='y', labelsize=13)
        ax.grid(True, which='major', ls=':')
        ax.annotate('', xy=(0.0, 5e-5), xytext=(0.22, 5e-5),
                    arrowprops=dict(facecolor='black', shrink=0.01, width=0.005, headwidth=3))
        ax.annotate('', xy=(0.5, 5e-5), xytext=(0.28, 5e-5),
                    arrowprops=dict(facecolor='black', shrink=0.01, width=0.005, headwidth=3))
        ax.text(0.1, 7e-5, 'Simple', fontsize=14, verticalalignment='center')
        ax.text(0.33, 7e-5, 'Complex', fontsize=14, verticalalignment='center')

    for idx, p in enumerate(probabilities):
        eig = np.load(f'{root}/{MODEL}/{NODES}/{network}/{K}_p{p:.6f}_eig_norm.npy')
        label_string = _corr_label(props.xs(p, level='p'))
        color = 'black' if idx >= 4 else None
        axs[3][1].hist(eig, bins=100, density=True, alpha=0.6, label=label_string, color=color)
        axins.hist(eig, bins=100, density=True, alpha=0.5, label=label_string, color=color)

    axs[3][1].text(-0.02, 1.03, fr'\textbf{{({SUB_PLOT_TAG[9]})}}', transform=axs[3][1].transAxes, fontsize=15)
    axs[3][1].set_ylabel(r'Density', labelpad=2.5, fontsize=14)
    axs[3][1].set_xlabel(r'Normalized Laplacian eigenvalues', labelpad=2, math_fontfamily='cm', fontsize=14)
    axs[3][1].set_ylim(hist_ylim)
    axs[3][1].set_xlim([-0.02, 2.05])
    axs[3][1].tick_params(axis='x', labelsize=13)
    axs[3][1].tick_params(axis='y', labelsize=13)

    axins.set_xlim(-0.02, 0.62)
    axins.set_ylim(*inset_ylim)
    axins.set_xticks([0.0, 0.2, 0.4, 0.6])
    axins.set_yticks(inset_yticks)
    axins.tick_params(axis='x', labelsize=5)
    axins.tick_params(axis='y', labelsize=5)
    axs[3][1].indicate_inset_zoom(axins, edgecolor='black')

    arrow = FancyArrowPatch(arrow_start, arrow_end, arrowstyle='->,head_width=2,head_length=5',
                             transform=axs[3][1].transAxes, color='black', linewidth=0.5, zorder=5)
    axs[3][1].add_patch(arrow)

    axs[3][0].remove()
    axs[3][2].axis('off')
    handles, labels = axs[3][1].get_legend_handles_labels()
    legend0 = axs[3][2].legend(handles, labels,
                                title=r'$C \;\, |\;\;\, T \;\,| \;\,\: \ell  \;\:\,  |\;\;\, R_{g}$',
                                title_fontsize=16, fontsize=14, loc=[-0.12, 0], frameon=False)
    legend0.get_title().set_position((1.5, 0))
    legend0.get_title().set_fontsize('14')
    legend0.get_title().set_fontweight('normal')

    os.makedirs(fig_path, exist_ok=True)
    fig.savefig(f'{fig_path}/Figure_S{counter}.pdf')
    plt.close(fig)
    print(f'{fig_path}/Figure_S{counter}.pdf saved')


# mhk histogram-panel axis limits (spectrum y-axis limit MHK(3.1))
MHK_HIST_KW = dict(hist_ylim=(0, 3.1), inset_ylim=(0, 0.5), inset_yticks=[0, 0.25, 0.5],
                    arrow_start=(0.13, 0.17), arrow_end=(0.2, 0.55))

# ws histogram-panel axis limits (spectrum y-axis limit WS(4))
WS_HIST_KW = dict(hist_ylim=(0, 4), inset_ylim=(0, 1), inset_yticks=[0, 0.5, 1],
                   arrow_start=(0.13, 0.25), arrow_end=(0.2, 0.55))

In [5]:
CAS = '0.3'
FIG1_S11_TH_EXCLUDE = (0.304,)


def _plot_polspeed_panel(ax, network, k, t_b, perc, ax_label, ylim_top=1.12, legend=True,
                          cas=CAS, root=ROOT):
    '''
    Single pooled (+/- 1 std) polarization-speed-vs-threshold panel at a
    fixed cascade size, for one (network, k, t_b, perc) combo. Same pooling
    machinery as generate_sm_figure's per-cascade panel, but drawn onto a
    caller-supplied ax (so Figures 1/4/7/S11/S12 can lay panels out in
    their own grids rather than the S3-S10 9-cascade grid), and excluding
    th=0.304 (see pooled_mpol's th_exclude) rather than S3-S10's pair.
    '''
    have, total = readiness(network, k, t_b, perc, root=root)
    print(f'{network} k={k} {t_b}{perc}%: {have}/{total} realization seeds ready')

    props = pooled_props(MODEL, NODES, network, k, root=root)
    mean, std = pooled_mpol(network, k, t_b, perc, root=root, th_exclude=FIG1_S11_TH_EXCLUDE)
    unordered_p = mean.index.get_level_values('p').unique()
    cc_by_p = {p: props.xs(p, level='p')['CC'].mean() for p in unordered_p}
    probabilities = sorted(unordered_p, key=lambda p: cc_by_p[p])

    ax.get_xaxis().get_major_formatter().set_scientific(False)
    for idx, p in enumerate(probabilities):
        label_string = _corr_label(props.xs(p, level='p'))

        m = mean.loc[(p, slice(None), network), cas].droplevel(['p', 'network']).sort_index()
        s = std.loc[(p, slice(None), network), cas].droplevel(['p', 'network']).sort_index()
        th = m.index.values
        lo = np.clip(m.values - s.values, 1e-6, None)
        hi = m.values + s.values

        color = 'black' if idx >= 4 else None
        line, = ax.plot(th, m.values, ls='-', label=label_string, color=color)
        ax.fill_between(th, lo, hi, alpha=0.2, color=line.get_color())

    ax.set_yscale('log')
    ax.set_ylabel(r'Polarization Speed $(v)$', labelpad=2.5)
    ax.set_xlabel(r'Threshold $( \theta )$', labelpad=2, math_fontfamily='cm')
    ax.set_ylim([3e-5, ylim_top])
    ax.set_xlim([-0.02, 0.52])
    ax.set_xticks([0, 0.1, 0.2, 0.3, 0.4, 0.5])
    ax.tick_params(axis='x', labelsize=7)
    ax.tick_params(axis='y', labelsize=7)
    ax.grid(True, which='major', ls=':')

    ax.annotate('', xy=(0.0, 5e-5), xytext=(0.22, 5e-5),
                arrowprops=dict(facecolor='black', shrink=0.01, width=0.005, headwidth=3))
    ax.annotate('', xy=(0.5, 5e-5), xytext=(0.28, 5e-5),
                arrowprops=dict(facecolor='black', shrink=0.01, width=0.005, headwidth=3))
    ax.text(0.1, 7e-5, 'Simple', fontsize=5, verticalalignment='center')
    ax.text(0.33, 7e-5, 'Complex', fontsize=5, verticalalignment='center')

    ax.text(-0.02, 1.03, ax_label, transform=ax.transAxes, fontsize=8)

    if legend:
        leg = ax.legend(title=r' $  C \;\, |\;\;\, T \;\,| \;\,\: \ell  \;\:\,  |\;\;\, R_{g} $',
                         framealpha=1, facecolor='white', loc=[1.1, 0], edgecolor='w',
                         borderpad=0.2, markerscale=0.8, handlelength=1.4, handletextpad=0.4, fontsize=7)
        leg.get_title().set_position((1.5, 0))
        leg.get_title().set_fontsize('7')


def _plot_eig_panel(ax, network, k, hist_ylim, inset_ylim, inset_yticks, arrow_start, arrow_end,
                     ax_label, inset_anchor=(0.04, 1), inset_w=1, inset_h=0.6, root=ROOT):
    '''
    Pooled normalized-Laplacian-eigenvalue density panel (+ zoomed inset)
    for one (network, k), reading the same precomputed
    '{k}_p{p:.6f}_eig_norm.npy' files generate_sm_figure's eigenvalue panel
    does. Only used by Figure 1 (c, d) among the five figures added here.
    '''
    props = pooled_props(MODEL, NODES, network, k, root=root)
    unordered_p = props.index.get_level_values('p').unique()
    cc_by_p = {p: props.xs(p, level='p')['CC'].mean() for p in unordered_p}
    probabilities = sorted(unordered_p, key=lambda p: cc_by_p[p])

    axins = inset_axes(ax, width=inset_w, height=inset_h, loc='upper left',
                        bbox_to_anchor=inset_anchor, bbox_transform=ax.transAxes)

    for idx, p in enumerate(probabilities):
        eig = np.load(f'{root}/{MODEL}/{NODES}/{network}/{k}_p{p:.6f}_eig_norm.npy')
        label_string = _corr_label(props.xs(p, level='p'))
        color = 'black' if idx >= 4 else None
        ax.hist(eig, bins=100, density=True, alpha=0.5, label=label_string, color=color)
        axins.hist(eig, bins=100, density=True, alpha=0.5, label=label_string, color=color)

    ax.set_ylabel(r'Density', labelpad=2.5)
    ax.set_xlabel(r'Normalized Laplacian Eigenvalues', labelpad=2, math_fontfamily='cm')
    ax.set_ylim(hist_ylim)
    ax.set_xlim([-0.02, 2.05])
    ax.set_xticks([0, 0.5, 1, 1.5, 2])
    ax.tick_params(axis='x', labelsize=7)
    ax.tick_params(axis='y', labelsize=7)
    ax.text(-0.02, 1.03, ax_label, transform=ax.transAxes, fontsize=8)

    axins.set_xlim(-0.02, 0.62)
    axins.set_ylim(*inset_ylim)
    axins.set_xticks([0.0, 0.2, 0.4, 0.6])
    axins.set_yticks(inset_yticks)
    axins.tick_params(axis='x', labelsize=5)
    axins.tick_params(axis='y', labelsize=5)
    ax.indicate_inset_zoom(axins, edgecolor='black')

    arrow = FancyArrowPatch(arrow_start, arrow_end, arrowstyle='->,head_width=2,head_length=5',
                             transform=ax.transAxes, color='black', linewidth=0.5, zorder=5)
    ax.add_patch(arrow)


def _widen_bottom_row(fig, axs, scale=1.20, gap=0.11, right_limit=0.98, left_limit=0.02):
    '''
    Ports Figure_generator_nets_manual.ipynb's Figure-1-only cosmetic
    tweak: widen and re-center the bottom two axes (the eigenvalue panels)
    with a fixed gap between them, since 2x2 subplots leaves them narrower
    than the top row's polarization-speed panels need. Must be called
    before plotting into axs[1, 0]/axs[1, 1] - remove()+add_axes() replaces
    those Axes objects.
    '''
    pos10 = axs[1, 0].get_position()
    pos11 = axs[1, 1].get_position()

    cL = pos10.x0 + pos10.width * 0.5
    cR = pos11.x0 + pos11.width * 0.5
    wL_new = pos10.width * scale
    wR_new = pos11.width * scale
    hL, hR = pos10.height, pos11.height
    yL, yR = pos10.y0, pos11.y0

    xL = cL - wL_new * 0.5
    xR = cR - wR_new * 0.5
    current_gap = xR - (xL + wL_new)
    delta = gap - current_gap
    xL -= delta / 2.0
    xR += delta / 2.0

    if xL < left_limit:
        shift = left_limit - xL
        xL += shift
        xR += shift
    if xR + wR_new > right_limit:
        shift = (xR + wR_new) - right_limit
        xL -= shift
        xR -= shift
        if xL < left_limit:
            xL = left_limit
            xR = max(xL + wL_new + gap, xL + wL_new)
            if xR + wR_new > right_limit:
                xR = right_limit - wR_new

    axs[1, 0].remove()
    axs[1, 0] = fig.add_axes([xL, yL, wL_new, hL])
    axs[1, 1].remove()
    axs[1, 1] = fig.add_axes([xR, yR, wR_new, hR])
    return axs


# Figure 1's eigenvalue-panel axis limits (differ from MHK_HIST_KW/WS_HIST_KW
# above, which are tuned for S3-S10 - Figure 1 uses a shared hist_ylim=6.8 for
# both networks, matching Figure_generator_nets_manual.ipynb's Figure-1 cell)
FIG1_WS_EIG_KW = dict(hist_ylim=(0, 6.8), inset_ylim=(0, 1), inset_yticks=[0, 0.5, 1],
                       arrow_start=(0.15, 0.15), arrow_end=(0.22, 0.5), inset_anchor=(0.04, 1))
FIG1_MHK_EIG_KW = dict(hist_ylim=(0, 6.8), inset_ylim=(0, 0.5), inset_yticks=[0, 0.25, 0.5],
                        arrow_start=(0.15, 0.08), arrow_end=(0.22, 0.5), inset_anchor=(0.05, 1))


In [6]:
def generate_figure1(root=ROOT, fig_path='../../figures'):
    # tight_layout=False, not True: with usetex on, a per-panel ax.legend()
    # call while the figure has an auto-relayout flag set forces a full
    # figure relayout (remeasuring every already-drawn text run via LaTeX)
    # on each subsequent legend/annotation call - confirmed to blow up
    # ~20x (10.9s -> 210.7s) between this figure's 1st and 2nd polspeed
    # panel alone. generate_sm_figure above avoids this by only calling
    # .legend() once, at the very end. No fig.tight_layout() call is added
    # here either (unlike generate_figure4/7/S11/S12 below): _widen_bottom_row
    # manually repositions axs[1,0]/axs[1,1] via absolute-coordinate
    # fig.add_axes(), which a later automatic relayout would undo.
    fig, axs = plt.subplots(figsize=(5.5 * 1.5, 2.31 * 2), ncols=2, nrows=2,
                             sharex=False, sharey=False, tight_layout=False)
    fig.subplots_adjust(wspace=0.4)

    # Widen/re-center the bottom (eigenvalue) row before plotting into it -
    # remove()+add_axes() replaces those Axes objects.
    _widen_bottom_row(fig, axs)

    _plot_polspeed_panel(axs[0][0], 'ws', 16, 'top', 5, r'\textbf{(a)}', ylim_top=1.12, root=root)
    _plot_polspeed_panel(axs[0][1], 'mhk', 16, 'top', 5, r'\textbf{(b)}', ylim_top=1.12, root=root)
    _plot_eig_panel(axs[1][0], 'ws', 16, ax_label=r'\textbf{(c)}', root=root, **FIG1_WS_EIG_KW)
    _plot_eig_panel(axs[1][1], 'mhk', 16, ax_label=r'\textbf{(d)}', root=root, **FIG1_MHK_EIG_KW)

    os.makedirs(fig_path, exist_ok=True)
    fig.savefig(f'{fig_path}/Figure_1.pdf')
    plt.close(fig)
    print(f'{fig_path}/Figure_1.pdf saved')


generate_figure1()

ws k=16 top5%: 100/100 realization seeds ready


mhk k=16 top5%: 100/100 realization seeds ready


../../figures/Figure_1.pdf saved


In [ ]:
'''
Figure 2
LFC Plotting
MHK + WS
Gain, Spectrums, Correlations
Figures 2-3

Reads from ../../nets/.

Seed usage differs by panel, which matters when reading the figure:
  (a) collective response  - POOLED over every realization seed on disk
                             (pooled_gains), drawn as the mean with a
                             +/-1 std band.
  (b) eigenvalue histogram - seed 1 only, one graph per curve.
  (c) correlations         - seed 1 only, over the k = 2..32 sweep.
  legend C | T | l | R_g   - seed 1 only (same props as panel c).

So the curves in (a) are 100-seed means while the legend labelling them
reports seed 1's structural values; they are close but not identical.
'''

def LFC_plot(net,Ks,t_b,perc,model = 'LFC', centrality = 'degree', output = None):
    plt.rc('text', usetex=True)
    plt.rc('text.latex', preamble = r'\usepackage{mathptmx}')
    # matplotlib.verbose.level = 'debug-annoying'
    for in_k in Ks:
        ix=pd.IndexSlice
        intended_k = in_k
        networks = [net]
        seed = 1
        nodes = 240
        degrees = range(2,33,2) 
        print(f'{in_k}/{nodes}/{t_b}/{perc}/{networks[0]}')

        print('Loading data ..,')


        d = {'ID':[],'freq':[],'p':[]}
        network_gains = pd.DataFrame(data=d)
        network_gains.set_index(['ID','freq','p'],inplace=True)

        e = {'ID':[],'p':[]}
        network_props = pd.DataFrame(data=e)
        network_props.set_index(['ID','p'],inplace=True)

        insert = True
        landscape = True

        # main_props = ['CC','SP','Rg']
        main_props = ['CC','T','SP','Rg']
        aux_props = ['rawCC','rawSP','rawRg','k','p','rawT']
        props = main_props + aux_props
        # prop_label= {'CC':r'$\bar{C}$','SP':r'$\bar{\ell}$','Rg':r'$\bar{R}_g$'}
        prop_label= {'CC':r'$\bar{C}$','SP':r'$\bar{\ell}$','Rg':r'$\bar{R}_g$','T':r'$\bar{T}$'}
        for network in networks:
            for k in degrees:
                # print(k)
                # new_network_gains = pd.read_csv(f'networks_new/{nodes}/{k}/{network}_corr_gains.csv',sep='\t',index_col=[1])
                # new_props = pd.read_csv(f'networks_new/{nodes}/{k}/{network}_props.csv',sep='\t',index_col=[0,1])

                new_network_gains = pd.read_csv(f'../../nets/{model}/{nodes}/{network}/{k}_seed{seed}_{t_b}_{perc}_corr_gains_{centrality}.csv',sep='\t',index_col=[1])
                new_props = pd.read_csv(f'../../nets/{model}/{nodes}/{network}/{k}_seed{seed}_props.csv',sep='\t',index_col=[0,1])

                new_props['p'] = new_props.index.get_level_values(1)

                new_props['k'] = k
                new_props['rawCC'] = new_props.CC
                new_props['rawSP'] = new_props.SP
                new_props['rawRg'] = new_props.Rg
                new_props['rawT'] = new_props['T']
                new_props.CC = new_props.CC/new_props.CC.max()
                new_props['T'] = new_props['T']/new_props['T'].max()
                new_props.Rg = new_props.Rg/new_props.Rg.min()
                new_props.SP = new_props.SP/new_props.SP.min()

                for f in new_network_gains.index.unique():
                    new_network_gains.loc[f,'normH2'] = (new_network_gains.loc[f].H2/new_network_gains.loc[f,'H2'].max())


                if 'k' in new_network_gains.columns:
                    new_network_gains.loc[network_gains.isnull().k,'k']=k
                else:
                    new_network_gains['k'] = k

                network_props = pd.concat([network_props, new_props.loc[:,props]])
                # network_props = network_props.append(new_props.loc[:,props])

                new_network_gains = new_network_gains.reset_index()
                new_network_gains['freq'] = new_network_gains.freq.apply(lambda x: round(x,5))

                new_network_gains.set_index(['ID','freq'],inplace=True)

                # network_gains = network_gains.append(new_network_gains)
                network_gains = pd.concat([network_gains, new_network_gains])

        network_props.fillna(value=0,inplace = True)
        network_gains.fillna(value=0,inplace = True)

        print('Getting correlations ..,')
        freq16 = network_gains.index.get_level_values(1).unique()
        freqAll = network_gains.loc[network_gains.k==[x for x in degrees if x != intended_k][0]].index.get_level_values(1).unique()

        gg = pd.MultiIndex.from_tuples(list(zip(main_props*len(freqAll),['H2']*len(freqAll)*len(main_props),sorted(list(freqAll)*len(main_props)))))
        corr_index_k = pd.MultiIndex.from_tuples(list(zip(main_props*len(freq16),['H2']*len(freq16)*len(main_props),sorted(list(freq16)*len(main_props)))))
        network_corr_k = pd.DataFrame(index=corr_index_k)
        network_corr_lim = pd.DataFrame(index=gg)

        correlations = ['spearman']

        max_sp_lim = network_props.SP.max()
        min_sp_lim = network_props.SP.min()

        max_cc_lim = network_props.CC.max()
        min_cc_lim = network_props.CC.min()

        max_rg = network_props.Rg.max()
        min_rg = network_props.Rg.min()

        if network == 'mhk':
            specialp = SPECIAL_P[(network, intended_k, nodes)]
            min_rg = 1.2
            max_rg = 2.2
            min_cc_lim = 0.6
            max_cc_lim = 1
        elif network == 'ws':
            min_cc_lim = 0.6
            max_cc_lim = 1
            min_rg = 1.2
            max_rg = 2.2
            specialp = SPECIAL_P[(network, intended_k, nodes)]

        elif network == 'ke':
            specialp = SPECIAL_P[(network, intended_k, nodes)]

        # network_props.query(f'{min_cc_lim} < CC < {max_cc_lim} and {min_sp_lim}  < SP < {max_sp_lim} and {min_rg} < Rg < {max_rg}').loc[:,('CC','SP','Rg')].corr(method='spearman').round(2)

        # query for  range of clutsering
        query_string = f'{min_cc_lim} < CC < {max_cc_lim} and {min_sp_lim} < SP < {max_sp_lim} and {min_rg} < Rg < {max_rg}'
        # query_string16 = f'k == {intended_k}'

        # lim_IDs = network_props.query(query_string).index.get_level_values(0)
        # temp_network_ins = network_gains.loc[ix[lim_IDs,:]]
        # corr_values_k = []
        corr_values_lim = []
        for coef in correlations:
            # ## Need to be sorted, because network_corr_... expects frequencies to be ordered
            # for f in freq16.sort_values():
            #     corr_values_k += list(network_props.query(query_string16).loc[:,main_props].corrwith(network_gains.loc[ix[:,f],'H2'],method=coef).values)

            for f in freqAll.sort_values():
                corr_values_lim += list(network_props.query(query_string).loc[:,main_props].corrwith(network_gains.loc[ix[:,f],'normH2'],method=coef).values)

        # network_corr_k.loc[corr_index_k,coef] = np.reshape(corr_values_k,(len(corr_values_k),1))
        network_corr_lim.loc[gg,coef] = np.reshape(corr_values_lim,(len(corr_values_lim),1))

        print(f'Plotting ..,')

        my_new_colors = ['darkslateblue','crimson','darkcyan','coral']


        fig,axs = plt.subplots(figsize=(5.5*1.2,2.1*1.7),ncols=2,nrows=2,sharex=False,sharey=False)
        fig.subplots_adjust(left=None, bottom=None, right=None, top=None, wspace=0.21, hspace=0.3)
        ## Getting correct plot setup
        gs = axs[0][0].get_gridspec()
        axs[0][0].remove()
        axs[1][0].remove()
        bigax = fig.add_subplot(gs[0:,0])

        ax_eig = axs[0][1]
        axins = inset_axes(ax_eig, width= .75, height= .4, loc='upper left',
                    bbox_to_anchor=(0.06, 1), bbox_transform=ax_eig.transAxes)

        ax_corr = axs[1][1]

        ## Formating collective frequency response
        bigax.set_xscale('log')
        bigax.set_yscale('log')
        bigax.set_xlabel(r'Frequency $(\omega)$',labelpad=2)
        bigax.set_ylabel(r'Collective response $(H^2(\omega))$',labelpad=2.5)



        specialk = intended_k
        directory_path = f'../../nets/{model}/{nodes}/{network}/{specialk}_seed{seed}/'
        sorted_filenames = get_sorted_filenames(directory_path)

        network_gains = network_gains.reset_index().set_index(['ID','freq','p'])
        prop_special = network_props.loc[network_props.k==specialk].groupby(level=1).mean()
        gain_special = network_gains.loc[network_gains.k==specialk].groupby(level=[2,1]).mean()
        pooled_gains_a = pooled_gains(model, nodes, network, specialk, t_b, perc, centrality=centrality, root='../../nets')
        pooled_gains_a = pooled_gains_a.reset_index().set_index(['ID', 'freq', 'p'])
        for th,p in enumerate(gain_special.index.get_level_values(0).unique()[specialp]):
            flag_l_label = 0

            C_label = str(prop_special.loc[p].rawCC.round(2))
            if len(C_label) < 4:
                C_label =  C_label + '0'
            l_label = str(prop_special.loc[p].rawSP.round(2))

            if len(l_label) < 5:
                flag_l_label=1

            r_label = str((prop_special.loc[p].rawRg.mean()/1000).round(1))
            # if len(r_label) > 3:
            #     r_label = r_label[:2]
            r_label = r_label + r'\! \times \! 10^{3}'

            if flag_l_label:
                label_string =fr'${C_label}  |  {prop_special.loc[p].rawT.round(2)} \;    |  {prop_special.loc[p].rawSP:.2f}  \, |  {r_label}$'

            pol_fig_legend_label = label_string 

            G = gt.load_graph(gt_path_for_p(directory_path, p))
            print(gt_path_for_p(directory_path, p))
            eig_lap = np.linalg.eigvalsh(gt.laplacian(G, norm=True).todense())

            raw_p = pooled_gains_a.xs(p, level='p')
            grouped = raw_p.groupby(level='freq')['H2']
            h2_mean = grouped.mean().sort_index()
            h2_std = grouped.std().sort_index()
            band_lo = (h2_mean - h2_std).clip(lower=1e-6)
            band_hi = h2_mean + h2_std

            line, = bigax.plot(h2_mean,label=label_string)
            bigax.fill_between(h2_mean.index, band_lo, band_hi, color=line.get_color(), alpha=0.35, linewidth=0, zorder=1)
            ax_eig.hist(eig_lap, bins=100, density=True, alpha=0.5)
            axins.hist(eig_lap, bins=100, density=True, alpha=0.5,label=pol_fig_legend_label)

        leg1 = bigax.legend(title=r'$ \;\,\;\;\; C \;\,\;\,\;\; | \;\,\,\;\, T  \;\:\,\;\,\,\;\,  | \,\;\,\;\ \ell   \,\;\,\ | \;\,\,\;\, R_{g} $', loc=[0.025,0.07],borderpad=0.2,markerscale=0.8,handlelength=0.9,handletextpad=0.4,fontsize=7)

        leg1.get_title().set_position((3.55, 0))
        leg1.get_title().set_fontsize('7')
        leg1.get_frame().set_facecolor('white')
        leg1.get_frame().set_alpha(1.0)
        leg1.get_frame().set_edgecolor('white')

        for idx,metric in enumerate(main_props):
            ax_corr.plot(network_corr_lim.loc[(metric,'H2')],label=prop_label[metric],zorder=[3,2,4,3,5][idx%5],c=my_new_colors[idx%5],linewidth=2,markersize=[5,5,5,5][idx%5],ls=[':', '-.', '--', 'solid'][idx%5],markeredgewidth=[1,1,1,1][idx%4],markerfacecolor='none')


        leg2=ax_corr.legend(title=r'$\bar{\chi}$',loc=[1.05,0.05],borderpad=0.2,markerscale=1,handlelength=3,handletextpad=0.4,fontsize=6)
        leg2.get_title().set_position((0, 0))
        leg2.get_title().set_fontsize('7')
        leg2.get_frame().set_facecolor('white')
        leg2.get_frame().set_alpha(1.0)
        leg2.get_frame().set_edgecolor('white')
        
        ## Setup axises
        


        ax_eig.set_ylabel(r'Density',labelpad=8)
        ax_eig.set_xlabel(r'Normalized Laplacian eigenvalues',labelpad=2,math_fontfamily='cm')
        ax_eig.set_ylim([0,3.7])
        ax_eig.set_xlim([-0.02,2.05])

        axins.set_xlim(-0.02, 0.62)
        axins.set_ylim(0, 1)
        # axins.set_ylabel(r'Density',labelpad=-8,fontsize=3,)
        # axins.set_xlabel(r'Normalized Eigenvalues',labelpad=-5,fontsize=3,math_fontfamily='cm')
        axins.set_xticks([0.0,0.2,0.4,0.6])
        axins.set_yticks([0,0.5,1])
        axins.tick_params(axis='x',labelsize=5)
        axins.tick_params(axis='y',labelsize=5)
        ax_eig.indicate_inset_zoom(axins, edgecolor="black")

        arrow = FancyArrowPatch(
        (0.13, 0.29),  # Start point - middle bottom of inset
        (0.25, 0.50),  # End point - straight down below inset
        arrowstyle='->,head_width=2,head_length=5',
        transform=ax_eig.transAxes,
        color='black',
        linewidth=0.5,
        zorder=5
        )
        ax_eig.add_patch(arrow)

        
        # ax_corr.set_ylabel(r'Correlation $r_s$',labelpad=2.5)
        ax_corr.set_ylabel(r'Correlation $r_s(\bar{H}^2(\omega),\bar{\chi})$',labelpad=2.5)
        ax_corr.set_xscale('log')
        ax_corr.set_xlabel(r'Frequency $( \omega )$',labelpad=2, math_fontfamily='cm')

        ## Set axis limits
        bigax.set_xlim([0.0001,2])
        bigax.set_ylim([0.017,520])

        ax_corr.set_ylim([-1.5,1.05])
        ax_corr.set_xlim([0.001,0.2])

        ## Title



        ## Indentifying letters
        bigax.text(-0.02,1.04,r'$\bf{(a)}$',transform=bigax.transAxes,fontsize=8)
        ax_eig.text(-0.02,1.04,r'$\bf{(b)}$',transform=ax_eig.transAxes,fontsize=8)
        ax_corr.text(-0.02,1.04,r'$\bf{(c)}$',transform=ax_corr.transAxes,fontsize=8)

        ax_corr.yaxis.set_major_locator(matplotlib.ticker.FixedLocator([-1,0,1]))

        ax_corr.yaxis.set_minor_locator(matplotlib.ticker.NullLocator())

        ## text boxes
        box_props = dict(alpha=1,facecolor='w',linewidth=0,zorder=100000,boxstyle='round',pad=0.4)
        bigax.grid(True, which="major", ls=":")
        ax_eig.grid(False)
        ax_corr.grid(False)

        bigax.tick_params(axis='x',labelsize=7)
        ax_eig.tick_params(axis='x',labelsize=7)
        ax_corr.tick_params(axis='x',labelsize=7)
        bigax.tick_params(axis='y',labelsize=7)
        ax_eig.tick_params(axis='y',labelsize=7)
        ax_corr.tick_params(axis='y',labelsize=7)

        ax_corr.fill_between(x=[0.015,2.9],y1=[-4,-4],y2=[-1.72,-1.72],facecolor='w',zorder=-10)


        bigax.annotate(
            '', xy=(1*10**-4, 2.5*10**-2), xytext=(8*10**-3, 2.5*10**-2),
            arrowprops=dict(facecolor='black', shrink=0.01,width=0.005, headwidth=3)
        )

        bigax.annotate(
            '', xy=(2, 2.5*10**-2), xytext=(2*10**-2, 2.5*10**-2),
            arrowprops=dict(facecolor='black', shrink=0.01,width=0.005, headwidth=3)
        )

        # Add text on the right saying "complex"
        bigax.text(8*10**-4, 3*10**-2, 'Simple', fontsize=5, verticalalignment='center')
        bigax.text(1*10**-1, 3*10**-2, 'Complex', fontsize=5, verticalalignment='center')




        ax_corr.annotate(
            '', xy=(10**-3, -1.3), xytext=(8*10**-3, -1.3),
            arrowprops=dict(facecolor='black', shrink=0.01,width=0.005, headwidth=3)
        )

        ax_corr.annotate(
            '', xy=(0.2, -1.3), xytext=(2*10**-2, -1.3),
            arrowprops=dict(facecolor='black', shrink=0.01,width=0.005, headwidth=3)
        )

        # Add text on the right saying "complex"
        ax_corr.text(2.5*10**-3, -1.2, 'Simple', fontsize=5, verticalalignment='center')
        ax_corr.text(4*10**-2, -1.2, 'Complex', fontsize=5, verticalalignment='center')


        plt.show()
        print(output)
        fig_path = f"../../figures"
        if not os.path.exists(fig_path):
            os.makedirs(fig_path)
        fig.savefig(f'{fig_path}/{output}.pdf')
        print(f'{fig_path}/{output}.pdf')
    return print('pdf file saved')


# model = 'LFC'
ks = [16]
for net in ['ws']:
    for t_b in ['top']:
        perc = int(5)
        LFC_plot(net,ks,t_b,perc,model = 'LFC', centrality = 'degree', output = "Figure_2")

In [8]:
'''
Figure 3
'''
ks = [16]
for net in ['mhk']:
    for t_b in ['top']:
        perc = int(5)
        LFC_plot(net,ks,t_b,perc,model = 'LFC', centrality = 'degree', output = "Figure_3")

16/240/top/5/mhk
Loading data ..,


Getting correlations ..,


Plotting ..,


../../nets/LFC/240/mhk/16_seed1/1784082303759_65580088_136.gt
../../nets/LFC/240/mhk/16_seed1/1784082324961_65580088_188.gt


../../nets/LFC/240/mhk/16_seed1/1784082325349_65580088_191.gt
../../nets/LFC/240/mhk/16_seed1/1784097305418_65635066_198.gt


Figure_3


../../figures/Figure_3.pdf
pdf file saved


In [9]:
def generate_figure4(root=ROOT, fig_path='../../figures'):
    # tight_layout=False + a single manual fig.tight_layout() at the end
    # (not tight_layout=True at creation) - see generate_figure1's comment:
    # a per-panel ax.legend() call while the figure's auto-relayout flag is
    # set forces a full-figure LaTeX-remeasure on every subsequent
    # legend/annotation call, confirmed to blow up ~20x panel-over-panel.
    fig, axs = plt.subplots(figsize=(5.5 * 1.25, 2.31), ncols=2, nrows=1,
                             sharex=False, sharey=False, tight_layout=False)
    fig.subplots_adjust(wspace=0.4)

    # Matching Figure_generator_nets_manual.ipynb's Figure-4 cell: legend
    # only on panel (b), not (a).
    _plot_polspeed_panel(axs[0], 'mhk', 16, 'top', 5, r'\textbf{(a)}', ylim_top=1, legend=False, root=root)
    _plot_polspeed_panel(axs[1], 'mhk', 16, 'bot', 5, r'\textbf{(b)}', ylim_top=1, legend=True, root=root)

    fig.tight_layout()
    os.makedirs(fig_path, exist_ok=True)
    fig.savefig(f'{fig_path}/Figure_4.pdf')
    plt.close(fig)
    print(f'{fig_path}/Figure_4.pdf saved')


generate_figure4()

mhk k=16 top5%: 100/100 realization seeds ready


mhk k=16 bot5%: 100/100 realization seeds ready


../../figures/Figure_4.pdf saved


In [ ]:
'''
Figure 5
LFC Plotting
top-bottom
Side-by-Side for one k and one network [top---bot]

Reads one realization (seed = 1, ws) from ../../nets/.
'''


save = True
seed = 1
model = 'LFC'
net = 'ws' #input("ws or mhk? ")
# Ks  = range(4,33,2) #[4,8,16,32]
perc = int(5)#[5,10,100]
Ks  = [16]
centrality = 'degree'
# def LFC_plot(net,Ks,t_b,perc,model = 'LFC'):
plt.rc('text', usetex=True)
plt.rc('text.latex', preamble = r'\usepackage{mathptmx}')
# matplotlib.verbose.level = 'debug-annoying'
for in_k in Ks:
    t_b = 'top'
    ix=pd.IndexSlice
    intended_k = in_k
    networks = [net]
    network = net
    nodes = 240
    
    print('Loading data ..,')

    d = {'ID':[],'freq':[],'p':[]}
    network_gains = pd.DataFrame(data=d)
    network_gains.set_index(['ID','freq','p'],inplace=True)

    e = {'ID':[],'p':[]}
    network_props = pd.DataFrame(data=e)
    network_props.set_index(['ID','p'],inplace=True)

    insert = True
    landscape = True

    # main_props = ['CC','SP','Rg']
    main_props = ['CC','T','SP','Rg']
    aux_props = ['rawCC','rawSP','rawRg','k','p','rawT']
    props = main_props + aux_props
    # prop_label= {'CC':r'$\bar{C}$','SP':r'$\bar{\ell}$','Rg':r'$\bar{R}_g$'}
    prop_label= {'CC':r'$\bar{C}$','SP':r'$\bar{\ell}$','Rg':r'$\bar{R}_g$','T':r'$\bar{T}$'}

    
    
    print(f'{in_k}/{nodes}/{t_b}/{perc}/{networks[0]}')
    k = in_k
    new_network_gains = pd.read_csv(f'../../nets/{model}/{nodes}/{network}/{k}_seed{seed}_{t_b}_{perc}_corr_gains_{centrality}.csv',sep='\t',index_col=[1])
    new_props = pd.read_csv(f'../../nets/{model}/{nodes}/{network}/{k}_seed{seed}_props.csv',sep='\t',index_col=[0,1])

    new_props['p'] = new_props.index.get_level_values(1)

    new_props['k'] = k
    new_props['rawCC'] = new_props.CC
    new_props['rawSP'] = new_props.SP
    new_props['rawRg'] = new_props.Rg
    new_props['rawT'] = new_props['T']
    new_props.CC = new_props.CC/new_props.CC.max()
    new_props['T'] = new_props['T']/new_props['T'].max()
    new_props.Rg = new_props.Rg/new_props.Rg.min()
    new_props.SP = new_props.SP/new_props.SP.min()

    for f in new_network_gains.index.unique():
        new_network_gains.loc[f,'normH2'] = (new_network_gains.loc[f].H2/new_network_gains.loc[f,'H2'].max())


    if 'k' in new_network_gains.columns:
        new_network_gains.loc[network_gains.isnull().k,'k']=k
    else:
        new_network_gains['k'] = k

        network_props = pd.concat([network_props, new_props.loc[:,props]])
        # network_props = network_props.append(new_props.loc[:,props])

        new_network_gains = new_network_gains.reset_index()
        new_network_gains['freq'] = new_network_gains.freq.apply(lambda x: round(x,5))

        new_network_gains.set_index(['ID','freq'],inplace=True)

        # network_gains = network_gains.append(new_network_gains)
        network_gains = pd.concat([network_gains, new_network_gains])

    network_props.fillna(value=0,inplace = True)
    network_gains.fillna(value=0,inplace = True)


    if network == 'mhk':
        specialp = SPECIAL_P[(network, intended_k, nodes)]

    elif network == 'ws':
        specialp = SPECIAL_P[(network, intended_k, nodes)]
    elif network == 'ke':
        specialp = SPECIAL_P[(network, intended_k, nodes)]

    print(f'Plotting ..,')

    my_new_colors = ['darkslateblue','darkcyan','coral','blue']
    
    fig,axs = plt.subplots(figsize=(5.5*1.2,2.1*1.7),ncols=2,nrows=1,sharex=False,sharey=False)
    fig.subplots_adjust(left=None, bottom=None, right=None, top=None, wspace=0.24, hspace=0.3)
    ## Getting correct plot setup
    gs = axs[0].get_gridspec()
    axs[0].remove()
    bigax = fig.add_subplot(gs[0:,0])

    bigax2 = axs[1]

    ## Formating collective frequency response
    bigax.set_xscale('log')
    bigax.set_yscale('log')
    bigax.set_xlabel(r'Frequency $(\omega)$',labelpad=2)
    bigax.set_ylabel(r'Collective response $(H^2)$',labelpad=2.5)
    
    bigax2.set_xscale('log')
    bigax2.set_yscale('log')
    bigax2.set_xlabel(r'Frequency $(\omega)$',labelpad=2)
    bigax2.set_ylabel(r'Collective response $(H^2)$',labelpad=2.5)


    specialk = intended_k
    # networks/{model}/{nodes}/{network}/{k}_props.csv
    directory_path = f'../../nets/{model}/{nodes}/{network}/{specialk}_seed{seed}/'
    sorted_filenames = get_sorted_filenames(directory_path)

    network_gains = network_gains.reset_index().set_index(['ID','freq','p'])
    prop_special = network_props.loc[network_props.k==specialk].groupby(level=1).mean()
    gain_special = network_gains.loc[network_gains.k==specialk].groupby(level=[2,1]).mean()
    pooled_gains_a = pooled_gains(model, nodes, network, specialk, t_b, perc, centrality=centrality, root='../../nets')
    pooled_gains_a = pooled_gains_a.reset_index().set_index(['ID', 'freq', 'p'])
    for th,p in enumerate(gain_special.index.get_level_values(0).unique()[specialp]):
        flag_l_label = 0

        C_label = str(prop_special.loc[p].rawCC.round(2))
        if len(C_label) < 4:
            C_label =  C_label + '0'
        l_label = str(prop_special.loc[p].rawSP.round(2))

        if len(l_label) < 5:
            flag_l_label=1

        r_label = str((prop_special.loc[p].rawRg.mean()/1000).round(1))
        # if len(r_label) > 3:
        #     r_label = r_label[:2]
        r_label = r_label + r'\! \times \! 10^{3}'


        if flag_l_label:
            label_string =fr'${C_label}  |  {prop_special.loc[p].rawT.round(2)} \;    |  {prop_special.loc[p].rawSP:.2f}  \, |  {r_label}$'
        # else:
        #     label_string =fr'${C_label}  |  {prop_special.loc[p].rawSP:.2f} \, |  {r_label}  \, |  {prop_special.loc[p][3]:.2f}$'

        pol_fig_legend_label = label_string 
        # print(network,p,intended_k)


        G = gt.load_graph(gt_path_for_p(directory_path, p))
        print(gt_path_for_p(directory_path, p))
        eig_lap = np.linalg.eigvalsh(gt.laplacian(G, norm=True).todense())

        raw_p = pooled_gains_a.xs(p, level='p')
        grouped = raw_p.groupby(level='freq')['H2']
        h2_mean = grouped.mean().sort_index()
        h2_std = grouped.std().sort_index()
        band_lo = (h2_mean - h2_std).clip(lower=1e-6)
        band_hi = h2_mean + h2_std
        line, = bigax.plot(h2_mean,label=label_string)
        bigax.fill_between(h2_mean.index, band_lo, band_hi, color=line.get_color(), alpha=0.35, linewidth=0, zorder=1)

    
    t_b = 'bot'
    
    d = {'ID':[],'freq':[],'p':[]}
    network_gains = pd.DataFrame(data=d)
    network_gains.set_index(['ID','freq','p'],inplace=True)
    
    print(f'{in_k}/{nodes}/{t_b}/{perc}/{networks[0]}')
    k = in_k
    new_network_gains = pd.read_csv(f'../../nets/{model}/{nodes}/{network}/{k}_seed{seed}_{t_b}_{perc}_corr_gains_{centrality}.csv',sep='\t',index_col=[1])

    for f in new_network_gains.index.unique():
        new_network_gains.loc[f,'normH2'] = (new_network_gains.loc[f].H2/new_network_gains.loc[f,'H2'].max())


    if 'k' in new_network_gains.columns:
        new_network_gains.loc[network_gains.isnull().k,'k']=k
    else:
        new_network_gains['k'] = k

        network_props = pd.concat([network_props, new_props.loc[:,props]])
        # network_props = network_props.append(new_props.loc[:,props])

        new_network_gains = new_network_gains.reset_index()
        new_network_gains['freq'] = new_network_gains.freq.apply(lambda x: round(x,5))

        new_network_gains.set_index(['ID','freq'],inplace=True)

        # network_gains = network_gains.append(new_network_gains)
        network_gains = pd.concat([network_gains, new_network_gains])

    network_props.fillna(value=0,inplace = True)
    network_gains.fillna(value=0,inplace = True)

    network_gains = network_gains.reset_index().set_index(['ID','freq','p'])
    prop_special = network_props.loc[network_props.k==specialk].groupby(level=1).mean()
    gain_special = network_gains.loc[network_gains.k==specialk].groupby(level=[2,1]).mean()
    pooled_gains_b = pooled_gains(model, nodes, network, specialk, t_b, perc, centrality=centrality, root='../../nets')
    pooled_gains_b = pooled_gains_b.reset_index().set_index(['ID', 'freq', 'p'])
    for th,p in enumerate(gain_special.index.get_level_values(0).unique()[specialp]):
        raw_p = pooled_gains_b.xs(p, level='p')
        grouped = raw_p.groupby(level='freq')['H2']
        h2_mean = grouped.mean().sort_index()
        h2_std = grouped.std().sort_index()
        band_lo = (h2_mean - h2_std).clip(lower=1e-6)
        band_hi = h2_mean + h2_std
        line, = bigax2.plot(h2_mean,label=label_string)
        bigax2.fill_between(h2_mean.index, band_lo, band_hi, color=line.get_color(), alpha=0.35, linewidth=0, zorder=1)

          

    leg1 = bigax.legend(title=r'$ \;\,\;\;\; C \;\,\;\,\;\; | \;\,\,\;\, T  \;\:\,\;\,\,\;\,  | \,\;\,\;\;\,\,\;\, \ell   \,\;\,\;\:\,\;\,  | \;\,\,\;\, R_{g} $', loc=[0.025,0.07],borderpad=0.2,markerscale=0.8,handlelength=0.9,handletextpad=0.4,fontsize=7)
    
    leg1.get_title().set_position((3.55, 0))
    leg1.get_title().set_fontsize('7')
    leg1.get_frame().set_facecolor('white')
    leg1.get_frame().set_alpha(1.0)
    leg1.get_frame().set_edgecolor('white')

    ## Set axis limits
    bigax.set_xlim([0.0001,2])
    bigax.set_ylim([0.017,520])
    bigax2.set_xlim([0.0001,2])
    bigax2.set_ylim([0.017,520])
    ## Title



    ## Indentifying letters
    bigax.text(-0.02,1.03,r'$\bf{(a)}$',transform=bigax.transAxes,fontsize=8)
    bigax2.text(-0.02,1.03,r'$\bf{(b)}$',transform=bigax2.transAxes,fontsize=8)

    ## text boxes
    box_props = dict(alpha=1,facecolor='w',linewidth=0,zorder=100000,boxstyle='round',pad=0.4)

    bigax.grid(True, which="major", ls=":")
    bigax2.grid(True, which="major", ls=":")

    bigax.tick_params(axis='x',labelsize=7)
    bigax.tick_params(axis='y',labelsize=7)
    bigax2.tick_params(axis='x',labelsize=7)
    bigax2.tick_params(axis='y',labelsize=7)

    ## Title
 

    ## text boxes
    box_props = dict(alpha=1,facecolor='w',linewidth=0,zorder=100000,boxstyle='round',pad=0.4)


    bigax.annotate(
        '', xy=(1*10**-4, 2.5*10**-2), xytext=(8*10**-3, 2.5*10**-2),
        arrowprops=dict(facecolor='black', shrink=0.01,width=0.005, headwidth=3)
    )

    bigax.annotate(
        '', xy=(2, 2.5*10**-2), xytext=(2*10**-2, 2.5*10**-2),
        arrowprops=dict(facecolor='black', shrink=0.01,width=0.005, headwidth=3)
    )

    # Add text on the right saying "complex"
    bigax.text(8*10**-4, 3*10**-2, 'Simple', fontsize=5, verticalalignment='center')
    bigax.text(1*10**-1, 3*10**-2, 'Complex', fontsize=5, verticalalignment='center')

    bigax2.annotate(
        '', xy=(1*10**-4, 2.5*10**-2), xytext=(8*10**-3, 2.5*10**-2),
        arrowprops=dict(facecolor='black', shrink=0.01,width=0.005, headwidth=3)
    )

    bigax2.annotate(
        '', xy=(2, 2.5*10**-2), xytext=(2*10**-2, 2.5*10**-2),
        arrowprops=dict(facecolor='black', shrink=0.01,width=0.005, headwidth=3)
    )

    # Add text on the right saying "complex"
    bigax2.text(8*10**-4, 3*10**-2, 'Simple', fontsize=5, verticalalignment='center')
    bigax2.text(1*10**-1, 3*10**-2, 'Complex', fontsize=5, verticalalignment='center')
    
    plt.show()
    if save:
        fig_path = f"../../figures"
        if not os.path.exists(fig_path):
            os.makedirs(fig_path)
        fig.savefig(f'{fig_path}/Figure_5.pdf')
        print(f'{fig_path}/{networks[0]}_{intended_k}_top_bot_{perc}_LFC_{centrality}.pdf')

In [ ]:
'''
Figure 6
LFC Plotting
top-bottom
Side-by-Side for one k and one network [top---bot]

Reads one realization (seed = 1, mhk) from ../../nets/.
'''


save = True
seed = 1
model = 'LFC'
net = 'mhk' #input("ws or mhk? ")
# Ks  = range(4,33,2) #[4,8,16,32]
perc = int(5)#[5,10,100]
Ks  = [16]
centrality = 'degree'
# def LFC_plot(net,Ks,t_b,perc,model = 'LFC'):
plt.rc('text', usetex=True)
plt.rc('text.latex', preamble = r'\usepackage{mathptmx}')
# matplotlib.verbose.level = 'debug-annoying'
for in_k in Ks:
    t_b = 'top'
    ix=pd.IndexSlice
    intended_k = in_k
    networks = [net]
    network = net
    nodes = 240
    
    print('Loading data ..,')

    d = {'ID':[],'freq':[],'p':[]}
    network_gains = pd.DataFrame(data=d)
    network_gains.set_index(['ID','freq','p'],inplace=True)

    e = {'ID':[],'p':[]}
    network_props = pd.DataFrame(data=e)
    network_props.set_index(['ID','p'],inplace=True)

    insert = True
    landscape = True

    # main_props = ['CC','SP','Rg']
    main_props = ['CC','T','SP','Rg']
    aux_props = ['rawCC','rawSP','rawRg','k','p','rawT']
    props = main_props + aux_props
    # prop_label= {'CC':r'$\bar{C}$','SP':r'$\bar{\ell}$','Rg':r'$\bar{R}_g$'}
    prop_label= {'CC':r'$\bar{C}$','SP':r'$\bar{\ell}$','Rg':r'$\bar{R}_g$','T':r'$\bar{T}$'}

    
    
    print(f'{in_k}/{nodes}/{t_b}/{perc}/{networks[0]}')
    k = in_k
    new_network_gains = pd.read_csv(f'../../nets/{model}/{nodes}/{network}/{k}_seed{seed}_{t_b}_{perc}_corr_gains_{centrality}.csv',sep='\t',index_col=[1])
    new_props = pd.read_csv(f'../../nets/{model}/{nodes}/{network}/{k}_seed{seed}_props.csv',sep='\t',index_col=[0,1])

    new_props['p'] = new_props.index.get_level_values(1)

    new_props['k'] = k
    new_props['rawCC'] = new_props.CC
    new_props['rawSP'] = new_props.SP
    new_props['rawRg'] = new_props.Rg
    new_props['rawT'] = new_props['T']
    new_props.CC = new_props.CC/new_props.CC.max()
    new_props['T'] = new_props['T']/new_props['T'].max()
    new_props.Rg = new_props.Rg/new_props.Rg.min()
    new_props.SP = new_props.SP/new_props.SP.min()

    for f in new_network_gains.index.unique():
        new_network_gains.loc[f,'normH2'] = (new_network_gains.loc[f].H2/new_network_gains.loc[f,'H2'].max())


    if 'k' in new_network_gains.columns:
        new_network_gains.loc[network_gains.isnull().k,'k']=k
    else:
        new_network_gains['k'] = k

        network_props = pd.concat([network_props, new_props.loc[:,props]])
        # network_props = network_props.append(new_props.loc[:,props])

        new_network_gains = new_network_gains.reset_index()
        new_network_gains['freq'] = new_network_gains.freq.apply(lambda x: round(x,5))

        new_network_gains.set_index(['ID','freq'],inplace=True)

        # network_gains = network_gains.append(new_network_gains)
        network_gains = pd.concat([network_gains, new_network_gains])

    network_props.fillna(value=0,inplace = True)
    network_gains.fillna(value=0,inplace = True)


    if network == 'mhk':
        specialp = SPECIAL_P[(network, intended_k, nodes)]

    elif network == 'ws':
        specialp = SPECIAL_P[(network, intended_k, nodes)]
    elif network == 'ke':
        specialp = SPECIAL_P[(network, intended_k, nodes)]

    print(f'Plotting ..,')

    my_new_colors = ['darkslateblue','darkcyan','coral','blue']
    
    fig,axs = plt.subplots(figsize=(5.5*1.2,2.1*1.7),ncols=2,nrows=1,sharex=False,sharey=False)
    fig.subplots_adjust(left=None, bottom=None, right=None, top=None, wspace=0.24, hspace=0.3)
    ## Getting correct plot setup
    gs = axs[0].get_gridspec()
    axs[0].remove()
    bigax = fig.add_subplot(gs[0:,0])

    bigax2 = axs[1]

    ## Formating collective frequency response
    bigax.set_xscale('log')
    bigax.set_yscale('log')
    bigax.set_xlabel(r'Frequency $(\omega)$',labelpad=2)
    bigax.set_ylabel(r'Collective response $(H^2)$',labelpad=2.5)
    
    bigax2.set_xscale('log')
    bigax2.set_yscale('log')
    bigax2.set_xlabel(r'Frequency $(\omega)$',labelpad=2)
    bigax2.set_ylabel(r'Collective response $(H^2)$',labelpad=2.5)


    specialk = intended_k
    # networks/{model}/{nodes}/{network}/{k}_props.csv
    directory_path = f'../../nets/{model}/{nodes}/{network}/{specialk}_seed{seed}/'
    sorted_filenames = get_sorted_filenames(directory_path)

    network_gains = network_gains.reset_index().set_index(['ID','freq','p'])
    prop_special = network_props.loc[network_props.k==specialk].groupby(level=1).mean()
    gain_special = network_gains.loc[network_gains.k==specialk].groupby(level=[2,1]).mean()
    pooled_gains_a = pooled_gains(model, nodes, network, specialk, t_b, perc, centrality=centrality, root='../../nets')
    pooled_gains_a = pooled_gains_a.reset_index().set_index(['ID', 'freq', 'p'])
    for th,p in enumerate(gain_special.index.get_level_values(0).unique()[specialp]):
        flag_l_label = 0

        C_label = str(prop_special.loc[p].rawCC.round(2))
        if len(C_label) < 4:
            C_label =  C_label + '0'
        l_label = str(prop_special.loc[p].rawSP.round(2))

        if len(l_label) < 5:
            flag_l_label=1

        r_label = str((prop_special.loc[p].rawRg.mean()/1000).round(1))
        # if len(r_label) > 3:
        #     r_label = r_label[:2]
        r_label = r_label + r'\! \times \! 10^{3}'


        if flag_l_label:
            label_string =fr'${C_label}  |  {prop_special.loc[p].rawT.round(2)} \;    |  {prop_special.loc[p].rawSP:.2f}  \, |  {r_label}$'
        # else:
        #     label_string =fr'${C_label}  |  {prop_special.loc[p].rawSP:.2f} \, |  {r_label}  \, |  {prop_special.loc[p][3]:.2f}$'

        pol_fig_legend_label = label_string 
        # print(network,p,intended_k)


        G = gt.load_graph(gt_path_for_p(directory_path, p))
        print(gt_path_for_p(directory_path, p))
        eig_lap = np.linalg.eigvalsh(gt.laplacian(G, norm=True).todense())

        raw_p = pooled_gains_a.xs(p, level='p')
        grouped = raw_p.groupby(level='freq')['H2']
        h2_mean = grouped.mean().sort_index()
        h2_std = grouped.std().sort_index()
        band_lo = (h2_mean - h2_std).clip(lower=1e-6)
        band_hi = h2_mean + h2_std
        line, = bigax.plot(h2_mean,label=label_string)
        bigax.fill_between(h2_mean.index, band_lo, band_hi, color=line.get_color(), alpha=0.35, linewidth=0, zorder=1)

    
    t_b = 'bot'
    
    d = {'ID':[],'freq':[],'p':[]}
    network_gains = pd.DataFrame(data=d)
    network_gains.set_index(['ID','freq','p'],inplace=True)
    
    print(f'{in_k}/{nodes}/{t_b}/{perc}/{networks[0]}')
    k = in_k
    new_network_gains = pd.read_csv(f'../../nets/{model}/{nodes}/{network}/{k}_seed{seed}_{t_b}_{perc}_corr_gains_{centrality}.csv',sep='\t',index_col=[1])

    for f in new_network_gains.index.unique():
        new_network_gains.loc[f,'normH2'] = (new_network_gains.loc[f].H2/new_network_gains.loc[f,'H2'].max())


    if 'k' in new_network_gains.columns:
        new_network_gains.loc[network_gains.isnull().k,'k']=k
    else:
        new_network_gains['k'] = k

        network_props = pd.concat([network_props, new_props.loc[:,props]])
        # network_props = network_props.append(new_props.loc[:,props])

        new_network_gains = new_network_gains.reset_index()
        new_network_gains['freq'] = new_network_gains.freq.apply(lambda x: round(x,5))

        new_network_gains.set_index(['ID','freq'],inplace=True)

        # network_gains = network_gains.append(new_network_gains)
        network_gains = pd.concat([network_gains, new_network_gains])

    network_props.fillna(value=0,inplace = True)
    network_gains.fillna(value=0,inplace = True)

    network_gains = network_gains.reset_index().set_index(['ID','freq','p'])
    prop_special = network_props.loc[network_props.k==specialk].groupby(level=1).mean()
    gain_special = network_gains.loc[network_gains.k==specialk].groupby(level=[2,1]).mean()
    pooled_gains_b = pooled_gains(model, nodes, network, specialk, t_b, perc, centrality=centrality, root='../../nets')
    pooled_gains_b = pooled_gains_b.reset_index().set_index(['ID', 'freq', 'p'])
    for th,p in enumerate(gain_special.index.get_level_values(0).unique()[specialp]):
        raw_p = pooled_gains_b.xs(p, level='p')
        grouped = raw_p.groupby(level='freq')['H2']
        h2_mean = grouped.mean().sort_index()
        h2_std = grouped.std().sort_index()
        band_lo = (h2_mean - h2_std).clip(lower=1e-6)
        band_hi = h2_mean + h2_std
        line, = bigax2.plot(h2_mean,label=label_string)
        bigax2.fill_between(h2_mean.index, band_lo, band_hi, color=line.get_color(), alpha=0.35, linewidth=0, zorder=1)

          

    leg1 = bigax.legend(title=r'$ \;\,\;\;\; C \;\,\;\,\;\; | \;\,\,\;\, T  \;\:\,\;\,\,\;\,  | \,\;\,\;\;\,\,\;\, \ell   \,\;\,\;\:\,\;\,  | \;\,\,\;\, R_{g} $', loc=[0.025,0.07],borderpad=0.2,markerscale=0.8,handlelength=0.9,handletextpad=0.4,fontsize=7)
    
    leg1.get_title().set_position((3.55, 0))
    leg1.get_title().set_fontsize('7')
    leg1.get_frame().set_facecolor('white')
    leg1.get_frame().set_alpha(1.0)
    leg1.get_frame().set_edgecolor('white')

    ## Set axis limits
    bigax.set_xlim([0.0001,2])
    bigax.set_ylim([0.017,520])
    bigax2.set_xlim([0.0001,2])
    bigax2.set_ylim([0.017,520])
    ## Title



    ## Indentifying letters
    bigax.text(-0.02,1.03,r'$\bf{(a)}$',transform=bigax.transAxes,fontsize=8)
    bigax2.text(-0.02,1.03,r'$\bf{(b)}$',transform=bigax2.transAxes,fontsize=8)

    ## text boxes
    box_props = dict(alpha=1,facecolor='w',linewidth=0,zorder=100000,boxstyle='round',pad=0.4)

    bigax.grid(True, which="major", ls=":")
    bigax2.grid(True, which="major", ls=":")

    bigax.tick_params(axis='x',labelsize=7)
    bigax.tick_params(axis='y',labelsize=7)
    bigax2.tick_params(axis='x',labelsize=7)
    bigax2.tick_params(axis='y',labelsize=7)

    ## Title
 

    ## text boxes
    box_props = dict(alpha=1,facecolor='w',linewidth=0,zorder=100000,boxstyle='round',pad=0.4)


    bigax.annotate(
        '', xy=(1*10**-4, 2.5*10**-2), xytext=(8*10**-3, 2.5*10**-2),
        arrowprops=dict(facecolor='black', shrink=0.01,width=0.005, headwidth=3)
    )

    bigax.annotate(
        '', xy=(2, 2.5*10**-2), xytext=(2*10**-2, 2.5*10**-2),
        arrowprops=dict(facecolor='black', shrink=0.01,width=0.005, headwidth=3)
    )

    # Add text on the right saying "complex"
    bigax.text(8*10**-4, 3*10**-2, 'Simple', fontsize=5, verticalalignment='center')
    bigax.text(1*10**-1, 3*10**-2, 'Complex', fontsize=5, verticalalignment='center')

    bigax2.annotate(
        '', xy=(1*10**-4, 2.5*10**-2), xytext=(8*10**-3, 2.5*10**-2),
        arrowprops=dict(facecolor='black', shrink=0.01,width=0.005, headwidth=3)
    )

    bigax2.annotate(
        '', xy=(2, 2.5*10**-2), xytext=(2*10**-2, 2.5*10**-2),
        arrowprops=dict(facecolor='black', shrink=0.01,width=0.005, headwidth=3)
    )

    # Add text on the right saying "complex"
    bigax2.text(8*10**-4, 3*10**-2, 'Simple', fontsize=5, verticalalignment='center')
    bigax2.text(1*10**-1, 3*10**-2, 'Complex', fontsize=5, verticalalignment='center')
    
    plt.show()
    if save:
        fig_path = f"../../figures"
        if not os.path.exists(fig_path):
            os.makedirs(fig_path)
        fig.savefig(f'{fig_path}/Figure_6.pdf')
        print(f'{fig_path}/{networks[0]}_{intended_k}_top_bot_{perc}_LFC_{centrality}.pdf')

In [12]:
def generate_figure7(root=ROOT, fig_path='../../figures'):
    # tight_layout=False + a single manual fig.tight_layout() at the end -
    # see generate_figure1's comment. This figure has 4 per-panel legends,
    # the worst case for the compounding relayout bug that fix avoids.
    fig, axs = plt.subplots(figsize=(5.5 * 1.5, 2.31 * 2), ncols=2, nrows=2,
                             sharex=False, sharey=False, tight_layout=False)
    fig.subplots_adjust(wspace=0.4)

    _plot_polspeed_panel(axs[0][0], 'ws', 8, 'top', 5, r'\textbf{(a)}', ylim_top=1.12, root=root)
    _plot_polspeed_panel(axs[0][1], 'ws', 16, 'top', 5, r'\textbf{(b)}', ylim_top=1.12, root=root)
    _plot_polspeed_panel(axs[1][0], 'mhk', 8, 'top', 5, r'\textbf{(c)}', ylim_top=1.12, root=root)
    _plot_polspeed_panel(axs[1][1], 'mhk', 16, 'top', 5, r'\textbf{(d)}', ylim_top=1.12, root=root)

    fig.tight_layout()
    os.makedirs(fig_path, exist_ok=True)
    fig.savefig(f'{fig_path}/Figure_7.pdf')
    plt.close(fig)
    print(f'{fig_path}/Figure_7.pdf saved')


generate_figure7()

ws k=8 top5%: 100/100 realization seeds ready


ws k=16 top5%: 100/100 realization seeds ready


mhk k=8 top5%: 100/100 realization seeds ready


mhk k=16 top5%: 100/100 realization seeds ready


../../figures/Figure_7.pdf saved


In [ ]:
'''
Figure 8
LFC Plotting
top_2Ks
Side-by-Side for one 2Ks and one network

Reads one realization (seed = 1, ws and mhk) from ../../nets/.
'''



model = 'LFC'
nets = ['ws', 'mhk'] #input("ws or mhk? ")
# Ks  = range(4,33,2) #[4,8,16,32]
perc = int(5)#[5,10,100]
t_b = 'top' #['top','bot','all']
Ks  = [8,16]
centrality = 'degree'

# def LFC_plot(net,Ks,t_b,perc,model = 'LFC'):
plt.rc('text', usetex=True)
plt.rc('text.latex', preamble = r'\usepackage{mathptmx}')
# matplotlib.verbose.level = 'debug-annoying'

fig,axs = plt.subplots(figsize=(5.5*1.2,2.1*1.7*1.8),ncols=2,nrows=2,sharex=False,sharey=False)
fig.subplots_adjust(left=None, bottom=None, right=None, top=None, wspace=0.24, hspace=0.3)
for selc, net in enumerate(nets):
    seed = 1
    for selector,in_k in enumerate(Ks):
        ix=pd.IndexSlice
        intended_k = in_k
        networks = [net]
        network = net
        nodes = 240

        def linepointstyle(order, color, marker):
            """Return style parameters for a
            solid line of a given color and
            order with points denoted by a
            given marker.
            """
            return {'zorder': order, 'lw': 1.5, 'c': color, 'ls': '-', 'marker': marker,
                    'markeredgecolor': color, 'markersize': 5, 'clip_on': False}

        def pointstyle(order, color, marker):
            """Return style parameters for a
            points of a given color and order,
            denoted by a given marker.
            """
            return {'zorder': order, 'color': color, 'marker': marker,
                    's': 30, 'clip_on': False}

        colors = ['r','b','g','y']
        markers = 's ^ o v'.split()
        lp_main = [linepointstyle(10 - i, colors[i], markers[i]) for i in range(4)]


        net_styles = [{'node_color': to_hex(p['c']),
                       'node_shape': '.',
                       'node_shape': p['marker'],
                       'width': 0.005, #p['lw'],
                       'edge_color': to_hex(p['c']),
                       'node_size': 0.01} for p in lp_main]


        ps_main = [pointstyle(10 - i, colors[i], markers[i]) for i in range(4)]

        print('Loading data ..,')


        d = {'ID':[],'freq':[],'p':[]}
        network_gains = pd.DataFrame(data=d)
        network_gains.set_index(['ID','freq','p'],inplace=True)

        e = {'ID':[],'p':[]}
        network_props = pd.DataFrame(data=e)
        network_props.set_index(['ID','p'],inplace=True)

        insert = True
        landscape = True

        # main_props = ['CC','SP','Rg']
        main_props = ['CC','T','SP','Rg']
        aux_props = ['rawCC','rawSP','rawRg','k','p','rawT']
        props = main_props + aux_props
        # prop_label= {'CC':r'$\bar{C}$','SP':r'$\bar{\ell}$','Rg':r'$\bar{R}_g$'}
        prop_label= {'CC':r'$\bar{C}$','SP':r'$\bar{\ell}$','Rg':r'$\bar{R}_g$','T':r'$\bar{T}$'}



        print(f'{in_k}/{nodes}/{t_b}/{perc}/{networks[0]}')
        k = in_k
        new_network_gains = pd.read_csv(f'../../nets/{model}/{nodes}/{network}/{k}_seed{seed}_{t_b}_{perc}_corr_gains_{centrality}.csv',sep='\t',index_col=[1])
        new_props = pd.read_csv(f'../../nets/{model}/{nodes}/{network}/{k}_seed{seed}_props.csv',sep='\t',index_col=[0,1])

        new_props['p'] = new_props.index.get_level_values(1)

        new_props['k'] = k
        new_props['rawCC'] = new_props.CC
        new_props['rawSP'] = new_props.SP
        new_props['rawRg'] = new_props.Rg
        new_props['rawT'] = new_props['T']
        new_props.CC = new_props.CC/new_props.CC.max()
        new_props['T'] = new_props['T']/new_props['T'].max()
        new_props.Rg = new_props.Rg/new_props.Rg.min()
        new_props.SP = new_props.SP/new_props.SP.min()

        for f in new_network_gains.index.unique():
            new_network_gains.loc[f,'normH2'] = (new_network_gains.loc[f].H2/new_network_gains.loc[f,'H2'].max())


        if 'k' in new_network_gains.columns:
            new_network_gains.loc[network_gains.isnull().k,'k']=k
        else:
            new_network_gains['k'] = k

            network_props = pd.concat([network_props, new_props.loc[:,props]])
            # network_props = network_props.append(new_props.loc[:,props])

            new_network_gains = new_network_gains.reset_index()
            new_network_gains['freq'] = new_network_gains.freq.apply(lambda x: round(x,5))

            new_network_gains.set_index(['ID','freq'],inplace=True)

            # network_gains = network_gains.append(new_network_gains)
            network_gains = pd.concat([network_gains, new_network_gains])

        network_props.fillna(value=0,inplace = True)
        network_gains.fillna(value=0,inplace = True)


        if network == 'mhk':
            specialp = SPECIAL_P[(network, intended_k, nodes)]

        elif network == 'ws':
            specialp = SPECIAL_P[(network, intended_k, nodes)]
        elif network == 'ke':
            specialp = SPECIAL_P[(network, intended_k, nodes)]

        print(f'Plotting ..,')

        my_new_colors = ['darkslateblue','darkcyan','coral','blue']

        cf = -3
        sf = 0



        ## Getting correct plot setup
        # bigax = axs[selc, selector]

        ## Formating collective frequency response
        axs[selc, selector].set_xscale('log')
        axs[selc, selector].set_yscale('log')
        axs[selc, selector].set_xlabel(r'Frequency $(\omega)$',labelpad=2)
        axs[selc, selector].set_ylabel(r'Collective response $(H^2)$',labelpad=2.5)

        specialk = intended_k
        # networks/{model}/{nodes}/{network}/{k}_props.csv
        directory_path = f'../../nets/{model}/{nodes}/{network}/{specialk}_seed{seed}/'
        sorted_filenames = get_sorted_filenames(directory_path)

        network_gains = network_gains.reset_index().set_index(['ID','freq','p'])
        prop_special = network_props.loc[network_props.k==specialk].groupby(level=1).mean()
        gain_special = network_gains.loc[network_gains.k==specialk].groupby(level=[2,1]).mean()
        pooled_gains_a = pooled_gains(model, nodes, network, specialk, t_b, perc, centrality=centrality, root='../../nets')
        pooled_gains_a = pooled_gains_a.reset_index().set_index(['ID', 'freq', 'p'])
        for th,p in enumerate(gain_special.index.get_level_values(0).unique()[specialp]):
            flag_l_label = 0

            C_label = str(prop_special.loc[p].rawCC.round(2))
            if len(C_label) < 4:
                C_label =  C_label + '0'
            l_label = str(prop_special.loc[p].rawSP.round(2))

            if len(l_label) < 5:
                flag_l_label=1

            r_label = str((prop_special.loc[p].rawRg.mean()/1000).round(1))
            # if len(r_label) > 3:
            #     r_label = r_label[:2]
            r_label = r_label + r'\! \times \! 10^{3}'


            if flag_l_label:
                label_string =fr'${C_label}  |  {prop_special.loc[p].rawT.round(2)} \;    |  {prop_special.loc[p].rawSP:.2f}  \, |  {r_label}$'
            # else:
            #     label_string =fr'${C_label}  |  {prop_special.loc[p].rawSP:.2f} \, |  {r_label}  \, |  {prop_special.loc[p][3]:.2f}$'

            pol_fig_legend_label = label_string 
            # print(network,p,intended_k)


            # G = gt.load_graph(gt_path_for_p(directory_path, p))
            # print(gt_path_for_p(directory_path, p))
            # eig_lap = np.linalg.eigvalsh(gt.spectral.laplacian(G, norm=True).todense())

            raw_p = pooled_gains_a.xs(p, level='p')
            grouped = raw_p.groupby(level='freq')['H2']
            h2_mean = grouped.mean().sort_index()
            h2_std = grouped.std().sort_index()
            band_lo = (h2_mean - h2_std).clip(lower=1e-6)
            band_hi = h2_mean + h2_std
            line, = axs[selc, selector].plot(h2_mean.iloc[sf:cf],label=label_string)
            axs[selc, selector].fill_between(h2_mean.iloc[sf:cf].index, band_lo.iloc[sf:cf], band_hi.iloc[sf:cf], color=line.get_color(), alpha=0.35, linewidth=0, zorder=1)



        leg1 = axs[selc, selector].legend(title=r'$ \;\,\;\;\; C \;\,\;\,\;\; | \;\,\,\;\, T  \;\:\,\;\,\,\;\,  | \,\;\,\;\;\,\,\;\, \ell   \,\;\,\;\:\,\;\,  | \;\,\,\;\, R_{g} $', loc=[0.025,0.1],borderpad=0.2,markerscale=0.8,handlelength=0.9,handletextpad=0.4,fontsize=7)

        leg1.get_title().set_position((3.55, 0))
        leg1.get_title().set_fontsize('7')
        leg1.get_frame().set_facecolor('white')
        leg1.get_frame().set_alpha(1.0)
        leg1.get_frame().set_edgecolor('white')

        ps_main[3]['s'] = 0.5
        ps_main[3]['zorder'] = 20

        ## Set axis limits
        axs[selc, selector].set_xlim([0.0001,2])
        axs[selc, selector].set_ylim([0.017,520])

        ## Title



        ## Indentifying letters
        axs[0,0].text(-0.02,1.03,r'$\bf{(a)}$',transform=axs[0,0].transAxes,fontsize=8)
        axs[0,1].text(-0.02,1.03,r'$\bf{(b)}$',transform=axs[0,1].transAxes,fontsize=8)
        axs[1,0].text(-0.02,1.03,r'$\bf{(c)}$',transform=axs[1,0].transAxes,fontsize=8)
        axs[1,1].text(-0.02,1.03,r'$\bf{(d)}$',transform=axs[1,1].transAxes,fontsize=8)
        ## text boxes
        box_props = dict(alpha=1,facecolor='w',linewidth=0,zorder=100000,boxstyle='round',pad=0.4)

        axs[selc, selector].grid(True, which="major", ls=":")

        axs[selc, selector].tick_params(axis='x',labelsize=7)
        axs[selc, selector].tick_params(axis='y',labelsize=7)

        ## Title


        ## text boxes
        box_props = dict(alpha=1,facecolor='w',linewidth=0,zorder=100000,boxstyle='round',pad=0.4)


        axs[selc, selector].annotate(
            '', xy=(1*10**-4, 2.5*10**-2), xytext=(8*10**-3, 2.5*10**-2),
            arrowprops=dict(facecolor='black', shrink=0.01,width=0.005, headwidth=3)
        )

        axs[selc, selector].annotate(
            '', xy=(2, 2.5*10**-2), xytext=(2*10**-2, 2.5*10**-2),
            arrowprops=dict(facecolor='black', shrink=0.01,width=0.005, headwidth=3)
        )

        # Add text on the right saying "complex"
        axs[selc, selector].text(8*10**-4, 3*10**-2, 'Simple', fontsize=5, verticalalignment='center')
        axs[selc, selector].text(1*10**-1, 3*10**-2, 'Complex', fontsize=5, verticalalignment='center')


plt.show()

fig_path = f"../../figures"
if not os.path.exists(fig_path):
    os.makedirs(fig_path)
fig.savefig(f'{fig_path}/Figure_8.pdf')

In [14]:
'''
Figure 9
celegans_metabolic vs. celegansneural, collective response + spectrum,
overlaid on shared axes (color = network, linestyle = top-bottom-5% degree
selection).

Reads ../../nets/real_world/*.gt - the Netzschleuder real-world networks
fetched by scripts/real_world/generate_real_world.py.
'''

from extract_lfc_csv import get_selected_gains
from matplotlib.lines import Line2D


def normalized_laplacian_eigenvalues(G):
    L = gt.laplacian(G, norm=True)
    return np.linalg.eigvalsh(L.todense())


def raw_gain_curve(G, t_b, perc=5):
    gains = get_selected_gains(G, t_b, perc)
    if gains is None:
        return None, None
    freqs = np.array(sorted(gains))
    h2 = np.array([gains[f] for f in freqs])
    return freqs, h2


def draw_response(ax, graphs):
    for G, name, color in graphs:
        curves = {}
        for t_b in ('top', 'bot'):
            freqs, h2 = raw_gain_curve(G, t_b)
            if freqs is not None:
                curves[t_b] = (freqs, h2)
        shared_max = max(h2.max() for _, h2 in curves.values())
        for t_b, style in [('top', 'solid'), ('bot', 'dashed')]:
            if t_b not in curves:
                continue
            freqs, h2 = curves[t_b]
            ax.plot(freqs, h2 / shared_max, linestyle=style, color=color)

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r'Frequency $(\omega)$', labelpad=2)
    ax.set_ylabel(r'Normalized collective response $(H^2 / H^2_{max})$', labelpad=2.5)

    color_handles = [Line2D([0], [0], color=color, label=name) for _, name, color in graphs]
    style_handles = [
        Line2D([0], [0], color='0.3', linestyle='solid', label='top 5\\%'),
        Line2D([0], [0], color='0.3', linestyle='dashed', label='bot 5\\%'),
    ]
    leg = ax.legend(handles=color_handles + style_handles, loc='lower left', fontsize=7,
                     borderpad=0.2, markerscale=0.8, handlelength=2.5, handletextpad=0.4)
    leg.get_frame().set_facecolor('white')
    leg.get_frame().set_alpha(1.0)
    leg.get_frame().set_edgecolor('white')

    ax.grid(True, which='major', ls=':')
    ax.tick_params(axis='x', labelsize=7)
    ax.tick_params(axis='y', labelsize=7)


def draw_spectrum(ax, graphs):
    for G, name, color in graphs:
        eig = normalized_laplacian_eigenvalues(G)
        ax.hist(eig, bins=100, density=True, alpha=0.5, color=color, label=name)

    ax.set_xlabel('Normalized Laplacian eigenvalues', labelpad=2)
    ax.set_ylabel('Density', labelpad=2.5)
    ax.set_xlim(-0.02, 2.05)

    leg = ax.legend(loc='upper left', fontsize=7, borderpad=0.2, handlelength=1.5, handletextpad=0.4)
    leg.get_frame().set_facecolor('white')
    leg.get_frame().set_alpha(1.0)
    leg.get_frame().set_edgecolor('white')

    ax.grid(True, which='major', ls=':')
    ax.tick_params(axis='x', labelsize=7)
    ax.tick_params(axis='y', labelsize=7)


rw_root = '../../nets/real_world'
NETWORKS = [
    (f'{rw_root}/scale_free/celegans_metabolic/celegans_metabolic.gt', 'C. elegans metabolic (2000)', 'tab:blue'),
    (f'{rw_root}/small_world/celegansneural/celegansneural.gt', 'C. elegans neurons (1986)', 'tab:orange'),
]
graphs = [(gt.load_graph(p), name, color) for p, name, color in NETWORKS]

fig, axs = plt.subplots(figsize=(5.5 * 1.2 * 1.5, 2.1 * 1.7), ncols=2, nrows=1)
fig.subplots_adjust(wspace=0.18)
for i, ax in enumerate(axs):
    ax.text(-0.02, 1.03, rf'$\bf({"ab"[i]})$', transform=ax.transAxes, fontsize=8)

draw_response(axs[0], graphs)
draw_spectrum(axs[1], graphs)

plt.show()

fig_path = f"../../figures"
if not os.path.exists(fig_path):
    os.makedirs(fig_path)
fig.savefig(f'{fig_path}/Figure_9.pdf', bbox_inches='tight')
print(f'saved {fig_path}/Figure_9.pdf')


saved ../../figures/Figure_9.pdf


In [ ]:
"""
Figure S1
MHK K = 4, P = 0.8 degree distribution

"""


# The mhk generator comes from net_functions.py, the single source shared
# with the generation pipeline, so this panel plots the same model the data
# was built from. It pins the realized average degree to exactly k -- the
# "Mean degree" line printed below reads 4.0000 at every N.
#
# It returns a networkx graph, so convert before graph_properties(), which
# expects graph_tool.
from net_functions import mhk_network, nx_to_gt


def graph_properties(G):
    """
    Compute degree statistics and clustering for a graph-tool graph.
    """

    degrees = np.array([v.out_degree() for v in G.vertices()])

    mean_degree = np.mean(degrees)
    max_degree = np.max(degrees)

    clustering = gt.global_clustering(G)[0]

    return degrees, mean_degree, max_degree, clustering


sizes = [100, 1000, 10000, 100000]

k = 4
p = 0.8

plt.figure(figsize=(7, 5))

for N in sizes:
    G = nx_to_gt(mhk_network(N=N, k=k, p=p, seed=1), expected_n=N)

    degrees, mean_degree, max_degree, clustering = graph_properties(G)

    values, counts = np.unique(degrees, return_counts=True)
    prob = counts / counts.sum()

    plt.loglog(
        values,
        prob,
        marker='o',
        linestyle='none',
        label=fr"$N=10^{{{int(np.log10(N))}}}$, $C={clustering:.3f}$"
    )

    print(f"N = {N}")
    print(f"Mean degree: {mean_degree:.3f}")
    print(f"Max degree: {max_degree}")
    print(f"Global clustering: {clustering:.3f}")
    print("-" * 40)

plt.xlabel(r'Degree $(k)$', labelpad=2, fontsize=14)
plt.ylabel(r'Probability $(P(k))$', labelpad=2.5, fontsize=14)
plt.grid(True, which='both', alpha=0.3)
plt.legend(fontsize=12)
plt.tick_params(axis='both', which='major', labelsize=14)
plt.tick_params(axis='both', which='minor', labelsize=5)
plt.tight_layout()
plt.savefig(f"../../figures/Figure_S1.pdf")
plt.show()


In [ ]:
'''
Figure S2
'''

# The Klemm-Eguiluz generator comes from net_functions.py, the single source
# shared with the generation pipeline. Two things to note about its
# interface and its rule:
#   * the second argument is the TARGET AVERAGE DEGREE k = 2m, not the
#     active-set size m, so an active set of 10 is requested as k = 20;
#   * an active node is deactivated with probability p_d(d_i) ~ 1/(m + d_i),
#     the degree-dependent rule Klemm-Eguiluz specifies, which is what makes
#     the degree distribution scale-free (tail exponent ~3.3).
from net_functions import ke_network

m = 10
n = 1000
K_TARGET = 2 * m

# Clustering vs N during KE growth. This re-walks net_functions.ke_network's
# growth loop rather than calling it, because the curve needs the clustering
# sampled after each node is added; the deactivation rule is identical.
rng = np.random.default_rng(1)
G = nx.complete_graph(m)
degree = dict(G.degree())
active_nodes = list(G.nodes())
cs = []
for i in range(m, n):
    for node in active_nodes:
        G.add_edge(node, i)
        degree[node] += 1
    degree[i] = len(active_nodes)
    active_nodes.append(i)
    weights = np.array([1.0 / (m + degree[a]) for a in active_nodes])
    weights /= weights.sum()
    active_nodes.remove(rng.choice(active_nodes, p=weights))
    cs.append(nx.average_clustering(G))

G = ke_network(100, K_TARGET, seed=1)
print(f'panel (a): N={G.number_of_nodes()} E={G.number_of_edges()} '
      f'<k>={2 * G.number_of_edges() / G.number_of_nodes():.4f} (target {K_TARGET})')

fig, axs = plt.subplots(ncols=3, figsize=(14, 4), gridspec_kw={'width_ratios': [2, 0.7, 3]})
nx.draw_circular(G, node_size=70, ax=axs[0])
axs[2].scatter(range(1, len(cs) + 1), cs, facecolors='none', edgecolors='r')

axs[0].text(-0.02, 1.03, r'\textbf{(a)}', transform=axs[0].transAxes, fontsize=16)
axs[2].text(-0.02, 1.03, r'\textbf{(b)}', transform=axs[2].transAxes, fontsize=16)

axs[2].set_xlabel(r'log($N$)', labelpad=2.5)
axs[2].set_ylabel('Clustering Coefficient', labelpad=2.5)
axs[2].set_xscale('log')
fig.delaxes(axs[1])
plt.savefig('../../figures/Figure_S2.pdf')


In [17]:
'''
LTM all cascade plotings for SM
Figures S3 S4 S5 S6
spectrum y-axis limt MKH(3.1)

Reads one realization (seed = 1) from ../../nets/.
'''

network_type = 'mhk'
model = 'LTM'
network = network_type
n = 1000
k = 16
seed = 1
# selector = ['top'] # centrality selector
# percentage = [int(n/20)] # number of nodes to choose
selector = ['top','top','bot','bot'] # centrality selector
percentage = [int(n/10),int(n/20),-int(n/20),-int(n/10)] # number of nodes to choose


# centrality = 'top'
# percen =  int(n/10)
save = True

ix = pd.IndexSlice
colors = ['darkslateblue','darkcyan','coral','blue']
#Toggle hatch
sub_plot_tag = 'abcdefghij'
## Pick which cascade sizes are consider, valid choises are 0.1,0.2,...,0.9
cascades = list(map(str,list(np.round(np.linspace(0.1,0.9,9),1))))

counter = 3
for centrality,percen in zip(selector,percentage):
    perc_pct = int(abs(percen)/n*100)
    selector_label = f'{centrality}-{perc_pct}' if centrality == 'bot' else f'{centrality}{perc_pct}'

    network_props = pd.read_csv(f'../../nets/{model}/{n}/{network_type}/{k}_seed{seed}_props.csv', sep='\t')
    network_props.set_index(['ID','p'],inplace=True)

    polarization = pd.read_csv(f'../../nets/{model}/{n}/{network_type}/{k}_seed{seed}_{selector_label}.csv', sep='\t')
    polarization.set_index(['ID','network','p','th','seed'],inplace=True)

    polarization = polarization[~polarization.index.get_level_values('th').isin([0.304])]
    mpol = polarization.groupby(['p','th','network']).agg(custom_mean)

    g = {'param1':[],'param2':[],'th':[]}

    fig,axs = plt.subplots(figsize=(5.5*1.5*2,2.31*6.5),ncols=3,nrows=4,sharex=False,sharey=False,tight_layout=True)
    fig.subplots_adjust(left=None, bottom=None, right=None, top=None, wspace=0.4, hspace=None)

    axins = inset_axes(axs[3][1], width= 1.5, height= 1, loc='upper left',
                bbox_to_anchor=(0.04, 1), bbox_transform=axs[3][1].transAxes)

    flag_l_label=0
    probabilities = np.sort(mpol.loc[ix[:,:,network],:].index.get_level_values(0).unique())[::-1]
    # for cas in cascades[:]: # different cascades 
    ### Plotting loop

    corr_fig_legend_label = [r'$C$',r'$\ell$',r'$R_g$',r'$T$']
    for i,cas in enumerate(cascades[:]):
        # print(f'Cascade size {cas,i}')
        ax = axs[i//3,i%3]
        ax.get_xaxis().get_major_formatter().set_scientific(False)

        for idx,p in enumerate(probabilities[::-1]):
            C_label = str(network_props.loc[ix[:,p],:]['CC'].mean().round(2))
            if len(C_label) < 4:
                C_label =  C_label + '0'

            T_label = str(network_props.loc[ix[:,p],:]['T'].mean().round(2))
            if len(T_label) < 4:
                T_label =  T_label + '0'

            l_label = str(network_props.loc[ix[:,p],:]['SP'].mean().round(1))
            if len(l_label) > 4:
                l_label = l_label[:3]
                flag_l_label=1
            # print(len(l_label))

            r_label = str((network_props.loc[ix[:,p],:]['Rg'].mean()/1000).round(1))
            # if len(r_label) > 3:
            #     r_label = r_label[:2]
            r_label = r_label + r'\! \times \! 10^{3}'

            label_string =r'${}  |  {}  |  {}  |  {} $'.format(C_label,T_label,l_label,r_label)



            pol_fig_legend_label = label_string
            #Make lines distingushable
            G = gt.load_graph(f'../../nets/{model}/{n}/{network_type}/{k}_seed{seed}/p{p:.6f}.gt')
            eig_lap = np.linalg.eigvalsh(gt.laplacian(G, norm=True).todense())

            ax.text(-0.02,1.03,fr'\textbf{{({sub_plot_tag[i]})}} Cascade size: {int(float(cas)*100)}\% activated nodes',transform=ax.transAxes,fontsize=15)
            # ax.text(-0.02, 1.03, f'({sub_plot_tag[i]}) Cascade size: {int(float(cas)*100)}\% activated nodes', 
                    # transform=ax.transAxes, fontsize=15, fontweight='bold')
            ax.set_yscale('log')
            ax.set_ylabel(r'Polarization Speed $(v)$',labelpad=2.5,fontsize = 14)
            ax.set_xlabel(r'Threshold $( \theta )$',labelpad=2,math_fontfamily='cm',fontsize = 14)
            ax.set_ylim([3*10**-5,1.12])
            ax.set_xlim([-0.02,0.52])
            ax.set_xticks([0,0.1,0.2,0.3,0.4,0.5])
            ax.tick_params(axis='x',labelsize=13)
            ax.tick_params(axis='y',labelsize=13)

            ax.grid(True, which="major", ls=":")

            ax.annotate(
                '', xy=(0.0, 5*10**-5), xytext=(0.22, 5*10**-5),
                arrowprops=dict(facecolor='black', shrink=0.01,width=0.005, headwidth=3)
            )
            
            ax.annotate(
                '', xy=(0.5, 5*10**-5), xytext=(0.28, 5*10**-5),
                arrowprops=dict(facecolor='black', shrink=0.01,width=0.005, headwidth=3)
            )
            
            # Add text on the right saying "complex"
            ax.text(0.1, 7*10**-5, 'Simple', fontsize=14, verticalalignment='center')
            ax.text(0.33, 7*10**-5, 'Complex', fontsize=14, verticalalignment='center')

            if idx < 4:
                ax.plot(mpol.loc[ix[p,:,network],f'{cas}'].index.get_level_values(1),mpol.loc[ix[p,:,network],f'{cas}'],ls='-',label=pol_fig_legend_label,linewidth=3)

            elif idx < 8:
                ax.plot(mpol.loc[ix[p,:,network],f'{cas}'].index.get_level_values(1),mpol.loc[ix[p,:,network],f'{cas}'],ls='-',label=pol_fig_legend_label,c='black', linewidth=3)

            if i==0:
                if idx < 4:
                    axs[3][1].hist(eig_lap, bins=100, density=True, alpha=0.6,label=pol_fig_legend_label)
                    axins.hist(eig_lap, bins=100, density=True, alpha=0.5,label=pol_fig_legend_label)

                elif idx < 8:
                    axs[3][1].hist(eig_lap, bins=100, density=True, alpha=0.6,label=pol_fig_legend_label,color='black')
                    axins.hist(eig_lap, bins=100, density=True, alpha=0.5,label=pol_fig_legend_label,color='black')

                axs[3][1].text(-0.02,1.03,fr'\textbf{{({sub_plot_tag[9]})}}',transform=axs[3][1].transAxes,fontsize=15)
                # axs[3][1].text(-0.02, 1.03, f'({sub_plot_tag[9]})',transform=axs[3][1].transAxes, fontsize=15, fontweight='bold')
                axs[3][1].set_ylabel(r'Density',labelpad=2.5,fontsize = 14)
                axs[3][1].set_xlabel(r'Normalized Laplacian eigenvalues',labelpad=2,math_fontfamily='cm',fontsize = 14)
                axs[3][1].set_ylim([0,3.1])  # WS [0,8], MHK [0,4]
                axs[3][1].set_xlim([-0.02,2.05])
                axs[3][1].tick_params(axis='x',labelsize=13)
                axs[3][1].tick_params(axis='y',labelsize=13)

    ## Legend and title
    axs[3][0].remove()
    axs[3][2].axis('off')

    # Turn axs[3][2] into an empty canvas
    axs[3][2].axis('off')
    
    # Get handles and labels from axs[3][1]
    handles, labels = axs[3][1].get_legend_handles_labels()
    
    # Create the legend in axs[3][2]
    legend0 = axs[3][2].legend(
        handles, labels,
        title=r'$C \;\, |\;\;\, T \;\,| \;\,\: \ell  \;\:\,  |\;\;\, R_{g}$',
        title_fontsize=16,
        fontsize=14,
        loc=[-0.12, 0],
        frameon=False
    )
    
    
    axins.set_xlim(-0.02, 0.62)
    axins.set_ylim(0, 0.5)
    # axins.set_ylabel(r'Density',labelpad=-8,fontsize=3,)
    # axins.set_xlabel(r'Normalized Eigenvalues',labelpad=-5,fontsize=3,math_fontfamily='cm')
    axins.set_xticks([0.0,0.2,0.4,0.6])
    axins.set_yticks([0,0.25,0.5])
    axins.tick_params(axis='x',labelsize=5)
    axins.tick_params(axis='y',labelsize=5)
    axs[3][1].indicate_inset_zoom(axins, edgecolor="black")
    
    arrow = FancyArrowPatch(
    (0.13, 0.17),  # Start point - middle bottom of inset
    (0.2, 0.55),  # End point - straight down below inset
    arrowstyle='->,head_width=2,head_length=5',
    transform=axs[3][1].transAxes,
    color='black',
    linewidth=0.5,
    zorder=5
    )
    axs[3][1].add_patch(arrow)


    
    
    legend0.get_title().set_position((1.5,0))
    legend0.get_title().set_fontsize('14')


# Then, set the title to have normal weight (if you don't want bold)
    legend0.get_title().set_fontweight('normal')

    # plt.show()
    # fig.savefig(f'../../figures/fig1/{network}/fig1_{cas}.pdf')
    if save:
        fig_path = f'../../figures/'
        if not os.path.exists(fig_path):
            os.makedirs(fig_path)
        fig.savefig(fig_path + f'Figure_S{counter}.pdf')
        counter +=1
        print(f'fig {network_type}_{n}_{k}_allcas_{model}_{centrality}{int(percen/n*100)}.pdf saved')
        plt.close()
    else:
        fig.show()


fig mhk_1000_16_allcas_LTM_top10.pdf saved


fig mhk_1000_16_allcas_LTM_top5.pdf saved


fig mhk_1000_16_allcas_LTM_bot-5.pdf saved


fig mhk_1000_16_allcas_LTM_bot-10.pdf saved


In [18]:
'''
LTM all cascade plotings for SM
Figures S7 S8 S9 S10
Spectrum y-axis limit for WS(4)

Reads one realization (seed = 1) from ../../nets/.
'''

network_type = 'ws'
model = 'LTM'
network = network_type
n = 1000
k = 16
seed = 1
# selector = ['top'] # centrality selector
# percentage = [int(n/20)] # number of nodes to choose
selector = ['top','top','bot','bot'] # centrality selector
percentage = [int(n/10),int(n/20),-int(n/20),-int(n/10)] # number of nodes to choose


# centrality = 'top'
# percen =  int(n/10)
save = True

ix = pd.IndexSlice
colors = ['darkslateblue','darkcyan','coral','blue']
#Toggle hatch
sub_plot_tag = 'abcdefghij'
## Pick which cascade sizes are consider, valid choises are 0.1,0.2,...,0.9
cascades = list(map(str,list(np.round(np.linspace(0.1,0.9,9),1))))

counter = 7
for centrality,percen in zip(selector,percentage):
    perc_pct = int(abs(percen)/n*100)
    selector_label = f'{centrality}-{perc_pct}' if centrality == 'bot' else f'{centrality}{perc_pct}'

    network_props = pd.read_csv(f'../../nets/{model}/{n}/{network_type}/{k}_seed{seed}_props.csv', sep='\t')
    network_props.set_index(['ID','p'],inplace=True)

    polarization = pd.read_csv(f'../../nets/{model}/{n}/{network_type}/{k}_seed{seed}_{selector_label}.csv', sep='\t')
    polarization.set_index(['ID','network','p','th','seed'],inplace=True)

    polarization = polarization[~polarization.index.get_level_values('th').isin([0.304])]
    mpol = polarization.groupby(['p','th','network']).agg(custom_mean)

    g = {'param1':[],'param2':[],'th':[]}

    fig,axs = plt.subplots(figsize=(5.5*1.5*2,2.31*6.5),ncols=3,nrows=4,sharex=False,sharey=False,tight_layout=True)
    fig.subplots_adjust(left=None, bottom=None, right=None, top=None, wspace=0.4, hspace=None)

    axins = inset_axes(axs[3][1], width= 1.5, height= 1, loc='upper left',
            bbox_to_anchor=(0.04, 1), bbox_transform=axs[3][1].transAxes)


    flag_l_label=0
    probabilities = np.sort(mpol.loc[ix[:,:,network],:].index.get_level_values(0).unique())
    # for cas in cascades[:]: # different cascades 
    ### Plotting loop

    corr_fig_legend_label = [r'$C$',r'$\ell$',r'$R_g$',r'$T$']
    for i,cas in enumerate(cascades[:]):
        # print(f'Cascade size {cas,i}')
        ax = axs[i//3,i%3]
        ax.get_xaxis().get_major_formatter().set_scientific(False)

        for idx,p in enumerate(probabilities[::-1]):
            C_label = str(network_props.loc[ix[:,p],:]['CC'].mean().round(2))
            if len(C_label) < 4:
                C_label =  C_label + '0'

            T_label = str(network_props.loc[ix[:,p],:]['T'].mean().round(2))
            if len(T_label) < 4:
                T_label =  T_label + '0'

            l_label = str(network_props.loc[ix[:,p],:]['SP'].mean().round(1))
            if len(l_label) > 4:
                l_label = l_label[:3]
                flag_l_label=1
            # print(len(l_label))

            r_label = str((network_props.loc[ix[:,p],:]['Rg'].mean()/1000).round(1))
            # if len(r_label) > 3:
            #     r_label = r_label[:2]
            r_label = r_label + r'\! \times \! 10^{3}'

            label_string =r'${}  |  {}  |  {}  |  {} $'.format(C_label,T_label,l_label,r_label)



            pol_fig_legend_label = label_string
            #Make lines distingushable
            G = gt.load_graph(f'../../nets/{model}/{n}/{network_type}/{k}_seed{seed}/p{p:.6f}.gt')
            eig_lap = np.linalg.eigvalsh(gt.laplacian(G, norm=True).todense())

            ax.text(-0.02,1.03,fr'\textbf{{({sub_plot_tag[i]})}} Cascade size: {int(float(cas)*100)}\% activated nodes',transform=ax.transAxes,fontsize=15)
            # ax.text(-0.02, 1.03, f'({sub_plot_tag[i]}) Cascade size: {int(float(cas)*100)}\% activated nodes', 
                    # transform=ax.transAxes, fontsize=15, fontweight='bold')
            ax.set_yscale('log')
            ax.set_ylabel(r'Polarization Speed $(v)$',labelpad=2.5,fontsize = 14)
            ax.set_xlabel(r'Threshold $( \theta )$',labelpad=2,math_fontfamily='cm',fontsize = 14)
            ax.set_ylim([3*10**-5,1.12])
            ax.set_xlim([-0.02,0.52])
            ax.set_xticks([0,0.1,0.2,0.3,0.4,0.5])
            ax.tick_params(axis='x',labelsize=13)
            ax.tick_params(axis='y',labelsize=13)

            ax.grid(True, which="major", ls=":")

            ax.annotate(
                '', xy=(0.0, 5*10**-5), xytext=(0.22, 5*10**-5),
                arrowprops=dict(facecolor='black', shrink=0.01,width=0.005, headwidth=3)
            )
            
            ax.annotate(
                '', xy=(0.5, 5*10**-5), xytext=(0.28, 5*10**-5),
                arrowprops=dict(facecolor='black', shrink=0.01,width=0.005, headwidth=3)
            )
            
            # Add text on the right saying "complex"
            ax.text(0.1, 7*10**-5, 'Simple', fontsize=14, verticalalignment='center')
            ax.text(0.33, 7*10**-5, 'Complex', fontsize=14, verticalalignment='center')

            if idx < 4:
                ax.plot(mpol.loc[ix[p,:,network],f'{cas}'].index.get_level_values(1),mpol.loc[ix[p,:,network],f'{cas}'],ls='-',label=pol_fig_legend_label,linewidth=3)

            elif idx < 8:
                ax.plot(mpol.loc[ix[p,:,network],f'{cas}'].index.get_level_values(1),mpol.loc[ix[p,:,network],f'{cas}'],ls='-',label=pol_fig_legend_label,c='black', linewidth=3)

            if i==0:
                if idx < 4:
                    axs[3][1].hist(eig_lap, bins=100, density=True, alpha=0.6,label=pol_fig_legend_label)
                    axins.hist(eig_lap, bins=100, density=True, alpha=0.5,label=pol_fig_legend_label)

                elif idx < 8:
                    axs[3][1].hist(eig_lap, bins=100, density=True, alpha=0.6,label=pol_fig_legend_label,color='black')
                    axins.hist(eig_lap, bins=100, density=True, alpha=0.5,label=pol_fig_legend_label,color='black')

                axs[3][1].text(-0.02,1.03,fr'\textbf{{({sub_plot_tag[9]})}}',transform=axs[3][1].transAxes,fontsize=15)
                # axs[3][1].text(-0.02, 1.03, f'({sub_plot_tag[9]})',transform=axs[3][1].transAxes, fontsize=15, fontweight='bold')
                axs[3][1].set_ylabel(r'Density',labelpad=2.5,fontsize = 14)
                axs[3][1].set_xlabel(r'Normalized Laplacian eigenvalues',labelpad=2,math_fontfamily='cm',fontsize = 14)
                axs[3][1].set_ylim([0,4])  # WS [0,8], MHK [0,4]
                axs[3][1].set_xlim([-0.02,2.05])
                axs[3][1].tick_params(axis='x',labelsize=13)
                axs[3][1].tick_params(axis='y',labelsize=13)


    axins.set_xlim(-0.02, 0.62)
    axins.set_ylim(0, 1)
    # axins.set_ylabel(r'Density',labelpad=-8,fontsize=3,)
    # axins.set_xlabel(r'Normalized Eigenvalues',labelpad=-5,fontsize=3,math_fontfamily='cm')
    axins.set_xticks([0.0,0.2,0.4,0.6])
    axins.set_yticks([0,0.5,1])
    axins.tick_params(axis='x',labelsize=5)
    axins.tick_params(axis='y',labelsize=5)
    axs[3][1].indicate_inset_zoom(axins, edgecolor="black")
    
    arrow = FancyArrowPatch(
    (0.13, 0.25),  # Start point - middle bottom of inset
    (0.2, 0.55),  # End point - straight down below inset
    arrowstyle='->,head_width=2,head_length=5',
    transform=axs[3][1].transAxes,
    color='black',
    linewidth=0.5,
    zorder=5
    )
    axs[3][1].add_patch(arrow)

    ## Legend and title
    axs[3][0].remove()
    axs[3][2].axis('off')

    # Turn axs[3][2] into an empty canvas
    axs[3][2].axis('off')
    
    # Get handles and labels from axs[3][1]
    handles, labels = axs[3][1].get_legend_handles_labels()
    
    # Create the legend in axs[3][2]
    legend0 = axs[3][2].legend(
        handles, labels,
        title=r'$C \;\, |\;\;\, T \;\,| \;\,\: \ell  \;\:\,  |\;\;\, R_{g}$',
        title_fontsize=16,
        fontsize=14,
        loc=[-0.12, 0],
        frameon=False
    )
    
    
    legend0.get_title().set_position((1.5,0))
    legend0.get_title().set_fontsize('14')


# Then, set the title to have normal weight (if you don't want bold)
    legend0.get_title().set_fontweight('normal')

    # plt.show()
    # fig.savefig(f'../../figures/fig1/{network}/fig1_{cas}.pdf')
    if save:
        fig_path = f'../../figures'
        if not os.path.exists(fig_path):
            os.makedirs(fig_path)
        # fig.savefig(fig_path + f'/{network_type}_{n}_{k}_allcas_{model}_{centrality}{int(percen/n*100)}.pdf')
        fig.savefig(fig_path + f'/Figure_S{counter}.pdf')
        print(f'fig {network_type}_{n}_{k}_allcas_{model}_{centrality}{int(percen/n*100)}.pdf saved')
        counter += 1
        plt.close()
    else:
        fig.show()


fig ws_1000_16_allcas_LTM_top10.pdf saved


fig ws_1000_16_allcas_LTM_top5.pdf saved


fig ws_1000_16_allcas_LTM_bot-5.pdf saved


fig ws_1000_16_allcas_LTM_bot-10.pdf saved


In [19]:
def generate_figureS11(root=ROOT, fig_path='../../figures'):
    # tight_layout=False + a single manual fig.tight_layout() at the end -
    # see generate_figure1's comment.
    fig, axs = plt.subplots(figsize=(5.5 * 1.25, 2.31), ncols=2, nrows=1,
                             sharex=False, sharey=False, tight_layout=False)
    fig.subplots_adjust(wspace=0.4)

    # Structural twin of Figure 4 but for ws instead of mhk - matching
    # Figure_generator_nets_manual.ipynb's Figure-S11 cell: legend only on
    # panel (b), not (a).
    _plot_polspeed_panel(axs[0], 'ws', 16, 'top', 5, r'\textbf{(a)}', ylim_top=1, legend=False, root=root)
    _plot_polspeed_panel(axs[1], 'ws', 16, 'bot', 5, r'\textbf{(b)}', ylim_top=1, legend=True, root=root)

    fig.tight_layout()
    os.makedirs(fig_path, exist_ok=True)
    fig.savefig(f'{fig_path}/Figure_S11.pdf')
    plt.close(fig)
    print(f'{fig_path}/Figure_S11.pdf saved')


generate_figureS11()

ws k=16 top5%: 100/100 realization seeds ready


ws k=16 bot5%: 100/100 realization seeds ready


../../figures/Figure_S11.pdf saved


In [20]:
def generate_figureS12(root=ROOT, fig_path='../../figures'):
    # tight_layout=False + a single manual fig.tight_layout() at the end -
    # see generate_figure1's comment. This figure has 4 per-panel legends,
    # the worst case for the compounding relayout bug that fix avoids.
    fig, axs = plt.subplots(figsize=(5.5 * 1.5, 2.31 * 2), ncols=2, nrows=2,
                             sharex=False, sharey=False, tight_layout=False)
    fig.subplots_adjust(wspace=0.4)

    _plot_polspeed_panel(axs[0][0], 'ws', 8, 'bot', 5, r'\textbf{(a)}', ylim_top=1.12, root=root)
    _plot_polspeed_panel(axs[0][1], 'ws', 16, 'bot', 5, r'\textbf{(b)}', ylim_top=1.12, root=root)
    _plot_polspeed_panel(axs[1][0], 'mhk', 8, 'bot', 5, r'\textbf{(c)}', ylim_top=1.12, root=root)
    _plot_polspeed_panel(axs[1][1], 'mhk', 16, 'bot', 5, r'\textbf{(d)}', ylim_top=1.12, root=root)

    fig.tight_layout()
    os.makedirs(fig_path, exist_ok=True)
    fig.savefig(f'{fig_path}/Figure_S12.pdf')
    plt.close(fig)
    print(f'{fig_path}/Figure_S12.pdf saved')


generate_figureS12()

ws k=8 bot5%: 100/100 realization seeds ready


ws k=16 bot5%: 100/100 realization seeds ready


mhk k=8 bot5%: 100/100 realization seeds ready


mhk k=16 bot5%: 100/100 realization seeds ready


../../figures/Figure_S12.pdf saved


In [21]:
"""
S13 for detailed top and bottom leaders for extreme plots

Reads one realization (seed = 1, mhk and ws) from ../../nets/.
"""



def make_original_label(prop_special, p):
    C_label = str(prop_special.loc[p].rawCC.round(2))

    if len(C_label) < 4:
        C_label = C_label + '0'

    r_label = str((prop_special.loc[p].rawRg.mean() / 1000).round(1))
    r_label = r_label + r'\! \times \! 10^{3}'

    label_string = (
        fr'${C_label}  |  {prop_special.loc[p].rawT.round(2)} \;'
        fr'    |  {prop_special.loc[p].rawSP:.2f}  \, |  {r_label}$'
    )

    return label_string


def LFC_plot_mhk_ws_combined_sidebyside(Ks, t_b_list, perc, model='LFC', centrality='degree'):
    """
    Create side-by-side figures for different t_b values.
    t_b_list: list of ['top', 'bot'] to plot side by side
    """
    
    plt.rc('text', usetex=True)
    plt.rc('text.latex', preamble=r'\usepackage{mathptmx}')

    nodes = 240
    degrees = range(2, 33, 2)

    for in_k in Ks:

        intended_k = in_k

        print(f'Combined mhk + ws (side-by-side) / k={intended_k}/{nodes}/{perc}')
        
        # Create figure with 2 subplots side-by-side
        fig, axes = plt.subplots(1, 2, figsize=(5.5 * 1.2 * 2, 3 * 1.7))
        
        # Original matplotlib color cycle
        original_colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
        
        for ax_idx, t_b in enumerate(t_b_list):
            
            bigax = axes[ax_idx]
            
            print(f'Processing t_b={t_b} ..,')
            
            # Same original Figure (a) formatting
            bigax.set_xscale('log')
            bigax.set_yscale('log')
            bigax.set_xlabel(r'Frequency $(\omega)$', labelpad=2)
            bigax.set_ylabel(r'Collective response $(H^2(\omega))$', labelpad=2.5)
            
            for network in ['mhk', 'ws']:

                seed = 1

                print(f'Processing {network} for t_b={t_b} ..,')

                d = {'ID': [], 'freq': [], 'p': []}
                network_gains = pd.DataFrame(data=d)
                network_gains.set_index(['ID', 'freq', 'p'], inplace=True)

                e = {'ID': [], 'p': []}
                network_props = pd.DataFrame(data=e)
                network_props.set_index(['ID', 'p'], inplace=True)

                main_props = ['CC', 'T', 'SP', 'Rg']
                aux_props = ['rawCC', 'rawSP', 'rawRg', 'k', 'p', 'rawT']
                props = main_props + aux_props

                for k in degrees:

                    new_network_gains = pd.read_csv(
                        f'../../nets/{model}/{nodes}/{network}/{k}_seed{seed}_{t_b}_{perc}_corr_gains_{centrality}.csv',
                        sep='\t',
                        index_col=[1]
                    )

                    new_props = pd.read_csv(
                        f'../../nets/{model}/{nodes}/{network}/{k}_seed{seed}_props.csv',
                        sep='\t',
                        index_col=[0, 1]
                    )

                    new_props['p'] = new_props.index.get_level_values(1)
                    new_props['k'] = k

                    new_props['rawCC'] = new_props.CC
                    new_props['rawSP'] = new_props.SP
                    new_props['rawRg'] = new_props.Rg
                    new_props['rawT'] = new_props['T']

                    new_props.CC = new_props.CC / new_props.CC.max()
                    new_props['T'] = new_props['T'] / new_props['T'].max()
                    new_props.Rg = new_props.Rg / new_props.Rg.min()
                    new_props.SP = new_props.SP / new_props.SP.min()

                    for f in new_network_gains.index.unique():
                        new_network_gains.loc[f, 'normH2'] = (
                            new_network_gains.loc[f].H2 /
                            new_network_gains.loc[f, 'H2'].max()
                        )

                    if 'k' in new_network_gains.columns:
                        new_network_gains.loc[network_gains.isnull().k, 'k'] = k
                    else:
                        new_network_gains['k'] = k

                    network_props = pd.concat([network_props, new_props.loc[:, props]])

                    new_network_gains = new_network_gains.reset_index()
                    new_network_gains['freq'] = new_network_gains.freq.apply(lambda x: round(x, 5))
                    new_network_gains.set_index(['ID', 'freq'], inplace=True)

                    network_gains = pd.concat([network_gains, new_network_gains])

                network_props.fillna(value=0, inplace=True)
                network_gains.fillna(value=0, inplace=True)

                if network == 'mhk':
                    specialp = [20, 85, 97, 99]
                    network_linestyle = '--'
                elif network == 'ws':
                    specialp = [51, 30, 12, 7]
                    network_linestyle = '-'

                specialk = intended_k

                network_gains = network_gains.reset_index().set_index(['ID', 'freq', 'p'])

                prop_special = (
                    network_props.loc[network_props.k == specialk]
                    .groupby(level=1)
                    .mean()
                )

                gain_special = (
                    network_gains.loc[network_gains.k == specialk]
                    .groupby(level=[2, 1])
                    .mean()
                )

                all_p_values = gain_special.index.get_level_values(0).unique()
                special_p_values = all_p_values[specialp]

                p_min = min(special_p_values)
                p_max = max(special_p_values)

                # Same colors as original subplot (a):
                color_for_p = {}

                for th, p0 in enumerate(special_p_values):
                    color_for_p[p0] = original_colors[th % len(original_colors)]

                plot_cases = [
                    {
                        'p': p_min,
                        'color': color_for_p[p_min],
                        'linestyle': network_linestyle
                    },
                    {
                        'p': p_max,
                        'color': color_for_p[p_max],
                        'linestyle': network_linestyle
                    }
                ]

                pooled_gains_a = pooled_gains(model, nodes, network, specialk, t_b, perc, centrality=centrality, root='../../nets')
                pooled_gains_a = pooled_gains_a.reset_index().set_index(['ID', 'freq', 'p'])

                for case in plot_cases:

                    p = case['p']
                    color = case['color']
                    linestyle = case['linestyle']

                    net_label = network.upper()

                    label_string = (
                        r'\makebox[0.75cm][l]{$\mathrm{' + net_label + r'}$}'
                        + make_original_label(prop_special, p)
                    )

                    raw_p = pooled_gains_a.xs(p, level='p')
                    grouped = raw_p.groupby(level='freq')['H2']
                    h2_mean = grouped.mean().sort_index()
                    h2_std = grouped.std().sort_index()
                    band_lo = (h2_mean - h2_std).clip(lower=1e-6)
                    band_hi = h2_mean + h2_std

                    bigax.plot(
                        h2_mean,
                        label=label_string,
                        color=color,
                        linestyle=linestyle
                    )
                    bigax.fill_between(h2_mean.index, band_lo, band_hi, color=color, alpha=0.35, linewidth=0, zorder=1)

            # Legend only on LEFT plot (ax_idx == 0)
            if ax_idx == 0:
                leg1 = bigax.legend(
                    title=(
                        r'\makebox[0.75cm][l]{$\mathrm{Net}$}'
                        r'$ \;\;\; C \;\,\;\,\;\; | \;\,\,\;\, T'
                        r'  \;\:\,\;\,\,\;\,  | \,\;\,\;\ \ell'
                        r'   \,\;\, | \;\,\,\;\, R_{g} $'
                    ),
                    loc=[0.025, 0.1],
                    borderpad=0.2,
                    markerscale=0.8,
                    handlelength=2,
                    handletextpad=0.4,
                    fontsize=7
                )

                leg1.get_title().set_position((3.55, 0))
                leg1.get_title().set_fontsize('7')
                leg1.get_frame().set_facecolor('white')
                leg1.get_frame().set_alpha(1.0)
                leg1.get_frame().set_edgecolor('white')

            # Same original axis limits (both subplots)
            bigax.set_xlim([0.0001, 2])
            bigax.set_ylim([0.017, 520])

            # Panel labels (a) and (b) on both subplots
            panel_label = chr(97 + ax_idx)  # 'a' for ax_idx=0, 'b' for ax_idx=1
            bigax.text(
                -0.02,
                1.04,
                rf'$\bf{{({panel_label})}}$',
                transform=bigax.transAxes,
                fontsize=8
            )

            # Same original grid and ticks
            bigax.grid(True, which="major", ls=":")

            bigax.tick_params(axis='x', labelsize=7)
            bigax.tick_params(axis='y', labelsize=7)

            # Same original Simple/Complex arrows
            bigax.annotate(
                '',
                xy=(1 * 10 ** -4, 2.5 * 10 ** -2),
                xytext=(8 * 10 ** -3, 2.5 * 10 ** -2),
                arrowprops=dict(
                    facecolor='black',
                    shrink=0.01,
                    width=0.005,
                    headwidth=3
                )
            )

            bigax.annotate(
                '',
                xy=(2, 2.5 * 10 ** -2),
                xytext=(2 * 10 ** -2, 2.5 * 10 ** -2),
                arrowprops=dict(
                    facecolor='black',
                    shrink=0.01,
                    width=0.005,
                    headwidth=3
                )
            )

            bigax.text(
                8 * 10 ** -4,
                3 * 10 ** -2,
                'Simple',
                fontsize=5,
                verticalalignment='center'
            )

            bigax.text(
                1 * 10 ** -1,
                3 * 10 ** -2,
                'Complex',
                fontsize=5,
                verticalalignment='center'
            )

        plt.tight_layout()
        plt.show()

        fig_path = f"../../figures/"

        if not os.path.exists(fig_path):
            os.makedirs(fig_path)

        save_file = f'{fig_path}/Figure_S13.pdf'
        print(f'{fig_path}/mhk_ws_{intended_k}_{perc}_LFC_degree_figure_a_sidebyside.pdf')
        fig.savefig(save_file, bbox_inches='tight')

        print(save_file)


# Run
ks = [16]
perc = int(5)

LFC_plot_mhk_ws_combined_sidebyside(
    ks,
    ['top', 'bot'],  # Both t_b values in one figure
    perc,
    model='LFC',
    centrality='degree'
)


Combined mhk + ws (side-by-side) / k=16/240/5
Processing t_b=top ..,
Processing mhk for t_b=top ..,


Processing ws for t_b=top ..,


Processing t_b=bot ..,
Processing mhk for t_b=bot ..,


Processing ws for t_b=bot ..,


../../figures//mhk_ws_16_5_LFC_degree_figure_a_sidebyside.pdf


../../figures//Figure_S13.pdf


In [ ]:
'''
Figure S14

Reads one realization (seed = 1, ws) from ../../nets/.
'''


model = 'LFC'
seed = 1
net = 'ws' #input("ws or mhk? ")
# Ks  = range(4,33,2) #[4,8,16,32]
perc = int(5)#[5,10,100]
Ks  = [16]
centrality = 'degree'
# def LFC_plot(net,Ks,t_b,perc,model = 'LFC'):
plt.rc('text', usetex=True)
plt.rc('text.latex', preamble = r'\usepackage{mathptmx}')
# matplotlib.verbose.level = 'debug-annoying'
for in_k in Ks:
    ix=pd.IndexSlice
    intended_k = in_k
    networks = [net]
    network = net
    nodes = 240
    
    print('Loading data ..,')
    if network == 'mhk':
        specialp = SPECIAL_P[(network, intended_k, nodes)]
    elif network == 'ws':
        specialp = SPECIAL_P[(network, intended_k, nodes)]
    elif network == 'ke':
        specialp = SPECIAL_P[(network, intended_k, nodes)]

    fig,axs = plt.subplots(figsize=(5.5*1.2,2.1*1.7*2),ncols=2,nrows=2,sharex=False,sharey=False)
    fig.subplots_adjust(left=None, bottom=None, right=None, top=None, wspace=0.24, hspace=0.3)
    ## Getting correct plot setup
    axs = axs.flatten()
    
    ## Formating collective frequency response
   
    
    for thi ,i in enumerate(specialp):
        # gs = axs[thi].get_gridspec()
        bigax = axs[thi]
        bigax.set_xscale('log')
        bigax.set_yscale('log')
        bigax.set_xlabel(r'Frequency $(\omega)$',labelpad=2)
        bigax.set_ylabel(r'Collective response $(H^2)$',labelpad=2.5)
        for thl, t_b in enumerate (['top','bot']):
            d = {'ID':[],'freq':[],'p':[]}
            network_gains = pd.DataFrame(data=d)
            network_gains.set_index(['ID','freq','p'],inplace=True)
        
            e = {'ID':[],'p':[]}
            network_props = pd.DataFrame(data=e)
            network_props.set_index(['ID','p'],inplace=True)
        
            insert = True
            landscape = True
        
            # main_props = ['CC','SP','Rg']
            main_props = ['CC','T','SP','Rg']
            aux_props = ['rawCC','rawSP','rawRg','k','p','rawT']
            props = main_props + aux_props
            # prop_label= {'CC':r'$\bar{C}$','SP':r'$\bar{\ell}$','Rg':r'$\bar{R}_g$'}
            prop_label= {'CC':r'$\bar{C}$','SP':r'$\bar{\ell}$','Rg':r'$\bar{R}_g$','T':r'$\bar{T}$'}
            print(f'{in_k}/{nodes}/{t_b}/{perc}/{networks[0]}')
            
            k = in_k
            new_network_gains = pd.read_csv(f'../../nets/{model}/{nodes}/{network}/{k}_seed{seed}_{t_b}_{perc}_corr_gains_{centrality}.csv',sep='\t',index_col=[1])
            new_props = pd.read_csv(f'../../nets/{model}/{nodes}/{network}/{k}_seed{seed}_props.csv',sep='\t',index_col=[0,1])
        
            new_props['p'] = new_props.index.get_level_values(1)
        
            new_props['k'] = k
            new_props['rawCC'] = new_props.CC
            new_props['rawSP'] = new_props.SP
            new_props['rawRg'] = new_props.Rg
            new_props['rawT'] = new_props['T']
            new_props.CC = new_props.CC/new_props.CC.max()
            new_props['T'] = new_props['T']/new_props['T'].max()
            new_props.Rg = new_props.Rg/new_props.Rg.min()
            new_props.SP = new_props.SP/new_props.SP.min()
        
            for f in new_network_gains.index.unique():
                new_network_gains.loc[f,'normH2'] = (new_network_gains.loc[f].H2/new_network_gains.loc[f,'H2'].max())
        
        
            if 'k' in new_network_gains.columns:
                new_network_gains.loc[network_gains.isnull().k,'k']=k
            else:
                new_network_gains['k'] = k
        
                network_props = pd.concat([network_props, new_props.loc[:,props]])
                # network_props = network_props.append(new_props.loc[:,props])
        
                new_network_gains = new_network_gains.reset_index()
                new_network_gains['freq'] = new_network_gains.freq.apply(lambda x: round(x,5))
        
                new_network_gains.set_index(['ID','freq'],inplace=True)
        
                # network_gains = network_gains.append(new_network_gains)
                network_gains = pd.concat([network_gains, new_network_gains])
        
            network_props.fillna(value=0,inplace = True)
            network_gains.fillna(value=0,inplace = True)
        
            print(f'Plotting ..,')
        
            my_new_colors = ['tab:blue','tab:orange','tab:green','tab:red']
        
            lin_style = ['solid',':']
           
            
            specialk = intended_k
            # networks/{model}/{nodes}/{network}/{k}_props.csv
            directory_path = f'../../nets/{model}/{nodes}/{network}/{specialk}_seed{seed}/'
            sorted_filenames = get_sorted_filenames(directory_path)
        
            network_gains = network_gains.reset_index().set_index(['ID','freq','p'])
            prop_special = network_props.loc[network_props.k==specialk].groupby(level=1).mean()
            gain_special = network_gains.loc[network_gains.k==specialk].groupby(level=[2,1]).mean()
            pooled_gains_a = pooled_gains(model, nodes, network, specialk, t_b, perc, centrality=centrality, root='../../nets')
            pooled_gains_a = pooled_gains_a.reset_index().set_index(['ID', 'freq', 'p'])
            for th,p in enumerate(gain_special.index.get_level_values(0).unique()[[i]]):
                flag_l_label = 0
        
                C_label = str(prop_special.loc[p].rawCC.round(2))
                if len(C_label) < 4:
                    C_label =  C_label + '0'
                l_label = str(prop_special.loc[p].rawSP.round(2))
        
                if len(l_label) < 5:
                    flag_l_label=1
        
                r_label = str((prop_special.loc[p].rawRg.mean()/1000).round(1))
                # if len(r_label) > 3:
                #     r_label = r_label[:2]
                r_label = r_label + r'\! \times \! 10^{3}'
        
        
                if flag_l_label:
                    label_string =fr'${C_label}  |  {prop_special.loc[p].rawT.round(2)} \;    |  {prop_special.loc[p].rawSP:.2f}  \, |  {r_label}$'
                # else:
                #     label_string =fr'${C_label}  |  {prop_special.loc[p].rawSP:.2f} \, |  {r_label}  \, |  {prop_special.loc[p][3]:.2f}$'
        
                pol_fig_legend_label = label_string 
                # print(network,p,intended_k)
        
                raw_p = pooled_gains_a.xs(p, level='p')
                grouped = raw_p.groupby(level='freq')['H2']
                h2_mean = grouped.mean().sort_index()
                h2_std = grouped.std().sort_index()
                band_lo = (h2_mean - h2_std).clip(lower=1e-6)
                band_hi = h2_mean + h2_std
                line, = bigax.plot(h2_mean, label=label_string, color=my_new_colors[thi], linestyle=lin_style[thl])
                bigax.fill_between(h2_mean.index, band_lo, band_hi, color=my_new_colors[thi], alpha=0.35, linewidth=0, zorder=1)
        
                leg1 = bigax.legend(title=r'$ \;\,\;\;\; C \;\,\;\,\;\; | \;\,\,\;\, T  \;\:\,\;\,\,\;\,  | \,\;\,\;\;\,\,\;\, \ell   \,\;\,\;\:\,\;\,  | \;\,\,\;\, R_{g}    \,\;\,\;\:\,\;\,$', loc=[0.025,0.07],borderpad=0.2,markerscale=0.8,handlelength=0.9,handletextpad=0.4,fontsize=7)
                
                leg1.get_title().set_position((3.55, 0))
                leg1.get_title().set_fontsize('7')
                leg1.get_frame().set_facecolor('white')
                leg1.get_frame().set_alpha(1.0)
                leg1.get_frame().set_edgecolor('white')
                
                ## Set axis limits
                bigax.set_xlim([0.0001,2])
                bigax.set_ylim([0.017,520])
                ## Title
            
            
                sub_ind = 'abcd'
                ## Indentifying letters
                bigax.text(-0.02,1.03,rf'$\bf({sub_ind[thi]})$',transform=bigax.transAxes,fontsize=8)
            
                ## text boxes
                box_props = dict(alpha=1,facecolor='w',linewidth=0,zorder=100000,boxstyle='round',pad=0.4)
            
                bigax.grid(True, which="major", ls=":")
            
                bigax.tick_params(axis='x',labelsize=7)
                bigax.tick_params(axis='y',labelsize=7)
            
                ## Title
             
            
                ## text boxes
                box_props = dict(alpha=1,facecolor='w',linewidth=0,zorder=100000,boxstyle='round',pad=0.4)
            
            
                bigax.annotate(
                    '', xy=(1*10**-4, 2.5*10**-2), xytext=(8*10**-3, 2.5*10**-2),
                    arrowprops=dict(facecolor='black', shrink=0.01,width=0.005, headwidth=3)
                )
            
                bigax.annotate(
                    '', xy=(2, 2.5*10**-2), xytext=(2*10**-2, 2.5*10**-2),
                    arrowprops=dict(facecolor='black', shrink=0.01,width=0.005, headwidth=3)
                )
                bigax.text(8*10**-4, 3*10**-2, 'Simple', fontsize=5, verticalalignment='center')
                bigax.text(1*10**-1, 3*10**-2, 'Complex', fontsize=5, verticalalignment='center')
    plt.show()
    
    fig_path = f"../../figures"
    if not os.path.exists(fig_path):
        os.makedirs(fig_path)
    fig.savefig(f'{fig_path}/Figure_S14.pdf')
    print(f'{fig_path}/{networks[0]}_{intended_k}_top_bot_{perc}_LFC_{centrality}_indv.pdf')

In [ ]:
'''
Figure S15

Reads one realization (seed = 1, mhk) from ../../nets/.
'''


model = 'LFC'
seed = 1
net = 'mhk' #input("ws or mhk? ")
# Ks  = range(4,33,2) #[4,8,16,32]
perc = int(5)#[5,10,100]
Ks  = [16]
centrality = 'degree'
# def LFC_plot(net,Ks,t_b,perc,model = 'LFC'):
plt.rc('text', usetex=True)
plt.rc('text.latex', preamble = r'\usepackage{mathptmx}')
# matplotlib.verbose.level = 'debug-annoying'
for in_k in Ks:
    ix=pd.IndexSlice
    intended_k = in_k
    networks = [net]
    network = net
    nodes = 240
    
    print('Loading data ..,')
    if network == 'mhk':
        specialp = SPECIAL_P[(network, intended_k, nodes)]
    elif network == 'ws':
        specialp = SPECIAL_P[(network, intended_k, nodes)]
    elif network == 'ke':
        specialp = SPECIAL_P[(network, intended_k, nodes)]

    fig,axs = plt.subplots(figsize=(5.5*1.2,2.1*1.7*2),ncols=2,nrows=2,sharex=False,sharey=False)
    fig.subplots_adjust(left=None, bottom=None, right=None, top=None, wspace=0.24, hspace=0.3)
    ## Getting correct plot setup
    axs = axs.flatten()
    
    ## Formating collective frequency response
   
    
    for thi ,i in enumerate(specialp):
        # gs = axs[thi].get_gridspec()
        bigax = axs[thi]
        bigax.set_xscale('log')
        bigax.set_yscale('log')
        bigax.set_xlabel(r'Frequency $(\omega)$',labelpad=2)
        bigax.set_ylabel(r'Collective response $(H^2)$',labelpad=2.5)
        for thl, t_b in enumerate (['top','bot']):
            d = {'ID':[],'freq':[],'p':[]}
            network_gains = pd.DataFrame(data=d)
            network_gains.set_index(['ID','freq','p'],inplace=True)
        
            e = {'ID':[],'p':[]}
            network_props = pd.DataFrame(data=e)
            network_props.set_index(['ID','p'],inplace=True)
        
            insert = True
            landscape = True
        
            # main_props = ['CC','SP','Rg']
            main_props = ['CC','T','SP','Rg']
            aux_props = ['rawCC','rawSP','rawRg','k','p','rawT']
            props = main_props + aux_props
            # prop_label= {'CC':r'$\bar{C}$','SP':r'$\bar{\ell}$','Rg':r'$\bar{R}_g$'}
            prop_label= {'CC':r'$\bar{C}$','SP':r'$\bar{\ell}$','Rg':r'$\bar{R}_g$','T':r'$\bar{T}$'}
            print(f'{in_k}/{nodes}/{t_b}/{perc}/{networks[0]}')
            
            k = in_k
            new_network_gains = pd.read_csv(f'../../nets/{model}/{nodes}/{network}/{k}_seed{seed}_{t_b}_{perc}_corr_gains_{centrality}.csv',sep='\t',index_col=[1])
            new_props = pd.read_csv(f'../../nets/{model}/{nodes}/{network}/{k}_seed{seed}_props.csv',sep='\t',index_col=[0,1])
        
            new_props['p'] = new_props.index.get_level_values(1)
        
            new_props['k'] = k
            new_props['rawCC'] = new_props.CC
            new_props['rawSP'] = new_props.SP
            new_props['rawRg'] = new_props.Rg
            new_props['rawT'] = new_props['T']
            new_props.CC = new_props.CC/new_props.CC.max()
            new_props['T'] = new_props['T']/new_props['T'].max()
            new_props.Rg = new_props.Rg/new_props.Rg.min()
            new_props.SP = new_props.SP/new_props.SP.min()
        
            for f in new_network_gains.index.unique():
                new_network_gains.loc[f,'normH2'] = (new_network_gains.loc[f].H2/new_network_gains.loc[f,'H2'].max())
        
        
            if 'k' in new_network_gains.columns:
                new_network_gains.loc[network_gains.isnull().k,'k']=k
            else:
                new_network_gains['k'] = k
        
                network_props = pd.concat([network_props, new_props.loc[:,props]])
                # network_props = network_props.append(new_props.loc[:,props])
        
                new_network_gains = new_network_gains.reset_index()
                new_network_gains['freq'] = new_network_gains.freq.apply(lambda x: round(x,5))
        
                new_network_gains.set_index(['ID','freq'],inplace=True)
        
                # network_gains = network_gains.append(new_network_gains)
                network_gains = pd.concat([network_gains, new_network_gains])
        
            network_props.fillna(value=0,inplace = True)
            network_gains.fillna(value=0,inplace = True)
        
            print(f'Plotting ..,')
        
            my_new_colors = ['tab:blue','tab:orange','tab:green','tab:red']
        
            lin_style = ['solid',':']
           
            
            specialk = intended_k
            # networks/{model}/{nodes}/{network}/{k}_props.csv
            directory_path = f'../../nets/{model}/{nodes}/{network}/{specialk}_seed{seed}/'
            sorted_filenames = get_sorted_filenames(directory_path)
        
            network_gains = network_gains.reset_index().set_index(['ID','freq','p'])
            prop_special = network_props.loc[network_props.k==specialk].groupby(level=1).mean()
            gain_special = network_gains.loc[network_gains.k==specialk].groupby(level=[2,1]).mean()
            pooled_gains_a = pooled_gains(model, nodes, network, specialk, t_b, perc, centrality=centrality, root='../../nets')
            pooled_gains_a = pooled_gains_a.reset_index().set_index(['ID', 'freq', 'p'])
            for th,p in enumerate(gain_special.index.get_level_values(0).unique()[[i]]):
                flag_l_label = 0
        
                C_label = str(prop_special.loc[p].rawCC.round(2))
                if len(C_label) < 4:
                    C_label =  C_label + '0'
                l_label = str(prop_special.loc[p].rawSP.round(2))
        
                if len(l_label) < 5:
                    flag_l_label=1
        
                r_label = str((prop_special.loc[p].rawRg.mean()/1000).round(1))
                # if len(r_label) > 3:
                #     r_label = r_label[:2]
                r_label = r_label + r'\! \times \! 10^{3}'
        
        
                if flag_l_label:
                    label_string =fr'${C_label}  |  {prop_special.loc[p].rawT.round(2)} \;    |  {prop_special.loc[p].rawSP:.2f}  \, |  {r_label}$'
                # else:
                #     label_string =fr'${C_label}  |  {prop_special.loc[p].rawSP:.2f} \, |  {r_label}  \, |  {prop_special.loc[p][3]:.2f}$'
        
                pol_fig_legend_label = label_string 
                # print(network,p,intended_k)
        
                raw_p = pooled_gains_a.xs(p, level='p')
                grouped = raw_p.groupby(level='freq')['H2']
                h2_mean = grouped.mean().sort_index()
                h2_std = grouped.std().sort_index()
                band_lo = (h2_mean - h2_std).clip(lower=1e-6)
                band_hi = h2_mean + h2_std
                line, = bigax.plot(h2_mean, label=label_string, color=my_new_colors[thi], linestyle=lin_style[thl])
                bigax.fill_between(h2_mean.index, band_lo, band_hi, color=my_new_colors[thi], alpha=0.35, linewidth=0, zorder=1)
        
                leg1 = bigax.legend(title=r'$ \;\,\;\;\; C \;\,\;\,\;\; | \;\,\,\;\, T  \;\:\,\;\,\,\;\,  | \,\;\,\;\;\,\,\;\, \ell   \,\;\,\;\:\,\;\,  | \;\,\,\;\, R_{g}    \,\;\,\;\:\,\;\,$', loc=[0.025,0.07],borderpad=0.2,markerscale=0.8,handlelength=0.9,handletextpad=0.4,fontsize=7)
                
                leg1.get_title().set_position((3.55, 0))
                leg1.get_title().set_fontsize('7')
                leg1.get_frame().set_facecolor('white')
                leg1.get_frame().set_alpha(1.0)
                leg1.get_frame().set_edgecolor('white')
                
                ## Set axis limits
                bigax.set_xlim([0.0001,2])
                bigax.set_ylim([0.017,520])
                ## Title
            
            
                sub_ind = 'abcd'
                ## Indentifying letters
                bigax.text(-0.02,1.03,rf'$\bf({sub_ind[thi]})$',transform=bigax.transAxes,fontsize=8)
            
                ## text boxes
                box_props = dict(alpha=1,facecolor='w',linewidth=0,zorder=100000,boxstyle='round',pad=0.4)
            
                bigax.grid(True, which="major", ls=":")
            
                bigax.tick_params(axis='x',labelsize=7)
                bigax.tick_params(axis='y',labelsize=7)
            
                ## Title
             
            
                ## text boxes
                box_props = dict(alpha=1,facecolor='w',linewidth=0,zorder=100000,boxstyle='round',pad=0.4)
            
            
                bigax.annotate(
                    '', xy=(1*10**-4, 2.5*10**-2), xytext=(8*10**-3, 2.5*10**-2),
                    arrowprops=dict(facecolor='black', shrink=0.01,width=0.005, headwidth=3)
                )
            
                bigax.annotate(
                    '', xy=(2, 2.5*10**-2), xytext=(2*10**-2, 2.5*10**-2),
                    arrowprops=dict(facecolor='black', shrink=0.01,width=0.005, headwidth=3)
                )
                bigax.text(8*10**-4, 3*10**-2, 'Simple', fontsize=5, verticalalignment='center')
                bigax.text(1*10**-1, 3*10**-2, 'Complex', fontsize=5, verticalalignment='center')
    plt.show()
    
    fig_path = f"../../figures"
    if not os.path.exists(fig_path):
        os.makedirs(fig_path)
    fig.savefig(f'{fig_path}/Figure_S15.pdf')
    print(f'{fig_path}/{networks[0]}_{intended_k}_top_bot_{perc}_LFC_{centrality}_indv.pdf')

In [ ]:
'''
Figure S16
LTM and LFC [16] [top and bot] HCSF
'''
print('Plotting LTM')
n = 1000
k = 16
cas = 0.3
seed = 1

model_type = 'LTM'
network_root = f'top{int(n/20)}'
network_class = 'ke'
network = network_class


save = True
hatching = False
pick_props = {'mhk':[99,60,10,0],'ws':[0,15,23,30,65],'ke':[0,10,30,20]}

ix = pd.IndexSlice
colors = ['darkslateblue','darkcyan','coral','blue']
#Toggle hatch

## Pick which cascade sizes are consider, valid choises are 0.1,0.2,...,0.9
cascades = list(map(str,list(np.round(np.linspace(0.1,0.9,9),1))))

# --- ke has no p axis --------------------------------------------------
# The Klemm-Eguiluz model has no rewiring / triad-formation parameter, so
# the pipeline stores ONE graph per (k, seed); a p-labelled sweep would just
# be repeated draws of a single ensemble. There is therefore no p axis here
# to index with `specialp`.
#
# The four curves below are four REALIZATIONS, not four p values.
# That is a real comparison, not a fallback: ke's growth is strongly
# stochastic, and across seeds 1-4 at k=16 the shortest path runs
# 1.97-2.88 and the spectral gap 0.41-1.36 (LFC, N=240). The legend still
# reads C | T | l | R_g -- those now vary by realization rather than by p.
KE_SEEDS = [1, 2, 3, 4]


def _ke_prop_label(row):
    """Legend entry 'C | T | l | R_g' from one props row."""
    return (fr'${row["CC"]:.2f}  |  {row["T"]:.2f} \;    |  {row["SP"]:.2f}  \, |  '
            fr'{row["Rg"] / 1000:.1f}\! \times \! 10^{{3}}$')


def ltm_curves(selector):
    """(threshold, polarization speed, label) per realization seed."""
    out = []
    for s in KE_SEEDS:
        props = pd.read_csv(
            f'../../nets/{model_type}/{n}/{network_class}/{k}_seed{s}_props.csv', sep='\t')
        pol = pd.read_csv(
            f'../../nets/{model_type}/{n}/{network_class}/{k}_seed{s}_{selector}.csv', sep='\t')
        pol = pol[~pol['th'].isin([0.304])]
        m = pol.groupby('th')[f'{cas}'].agg(custom_mean)
        row = props.iloc[0]
        out.append((row['CC'], m.index.values, m.values, _ke_prop_label(row)))
    # curves ordered by clustering, low -> high
    return [t[1:] for t in sorted(out, key=lambda t: t[0])]


def lfc_curves(t_b):
    """(H2 series indexed by frequency, label) per realization seed."""
    out = []
    for s in KE_SEEDS:
        gains = pd.read_csv(
            f'../../nets/{model}/{nodes}/{network}/{k}_seed{s}_{t_b}_{perc}'
            f'_corr_gains_{centrality}.csv', sep='\t')
        props = pd.read_csv(
            f'../../nets/{model}/{nodes}/{network}/{k}_seed{s}_props.csv', sep='\t')
        # H2 is plotted as stored, with NO (100 / perc) rescaling. The CSV
        # holds H2 already AVERAGED over the selected leaders, and each
        # leader's H2 is itself a whole-network sum, so scaling a 5% leader
        # sample up to 100% would be double counting -- a mean needs no
        # sample-size correction. The bound confirms it: H2 = ||h||^2 over
        # the N-1 followers and at omega -> 0 every follower tracks the
        # leader exactly (h_i -> 1), so H2 <= N - 1 = 239, and the stored
        # data peaks at 238.97.
        h2 = gains.set_index('freq')['H2'].sort_index()
        row = props.iloc[0]
        out.append((row['CC'], h2, _ke_prop_label(row)))
    # curves ordered by clustering, low -> high
    return [t[1:] for t in sorted(out, key=lambda t: t[0])]

ltm_top = ltm_curves('top5')



g = {'param1':[],'param2':[],'th':[]}
# network_corr = pd.DataFrame(data=g)
# network_corr.set_index(['param1','param2','th'],inplace=True)

fig,axs = plt.subplots(figsize=(5.5*1.25,2.31*2),ncols=2,nrows=2,sharex=False,sharey=False,tight_layout=True)
fig.subplots_adjust(left=None, bottom=None, right=None, top=None, wspace=0.4, hspace=None)


for th_x, speed, _ in ltm_top:
    axs[0,0].plot(th_x, speed, ls='-')
        
# Set scale stuff
axs[0,0].set_yscale('log')
axs[0,0].set_ylabel(r'Polarization Speed $(v)$',labelpad=2.5)
axs[0,0].set_xlabel(r'Threshold $( \theta )$',labelpad=2,math_fontfamily='cm')
axs[0,0].set_ylim([3*10**-5,1.12])
axs[0,0].set_xlim([-0.02,0.52])
axs[0,0].set_xticks([0,0.1,0.2,0.3,0.4,0.5])
axs[0,0].tick_params(axis='x',labelsize=7)
axs[0,0].tick_params(axis='y',labelsize=7)

axs[0,0].annotate(
    '', xy=(0.0, 5*10**-5), xytext=(0.22, 5*10**-5),
    arrowprops=dict(facecolor='black', shrink=0.01,width=0.005, headwidth=3)
)

axs[0,0].annotate(
    '', xy=(0.5, 5*10**-5), xytext=(0.28, 5*10**-5),
    arrowprops=dict(facecolor='black', shrink=0.01,width=0.005, headwidth=3)
)

# Add text on the right saying "complex"
axs[0,0].text(0.1, 7*10**-5, 'Simple', fontsize=5, verticalalignment='center')
axs[0,0].text(0.33, 7*10**-5, 'Complex', fontsize=5, verticalalignment='center')


n = 1000
k = 16
cas = 0.3

# model_type = 'LTM'
network_root = f'bot{-int(n/20)}'
# network_class = 'mhk'
# network = network_class


ltm_bot = ltm_curves('bot-5')

for th_x, speed, label_string in ltm_bot:
    axs[0,1].plot(th_x, speed, ls='-', label=label_string)

# Set scale stuff

axs[0,1].set_yscale('log')
axs[0,1].set_ylabel(r'Polarization Speed $(v)$',labelpad=2.5)
axs[0,1].set_xlabel(r'Threshold $( \theta )$',labelpad=2,math_fontfamily='cm')
axs[0,1].set_ylim([3*10**-5,1.12])
axs[0,1].set_xlim([-0.02,0.52])
axs[0,1].set_xticks([0,0.1,0.2,0.3,0.4,0.5])
axs[0,1].tick_params(axis='x',labelsize=7)
axs[0,1].tick_params(axis='y',labelsize=7)

## Legend and title
legend1 = axs[0,1].legend(title=r' $  C \;\, |\;\;\, T \;\,| \;\,\: \ell  \;\:\,  |\;\;\, R_{g} $', framealpha=1, facecolor='white',loc=[1.1,0],edgecolor='w',borderpad=0.2,markerscale=0.8,handlelength=1.4,handletextpad=0.4,fontsize=7)
legend1.get_title().set_position((1.5,0))
legend1.get_title().set_fontsize('7')



axs[0,1].annotate(
    '', xy=(0.0, 5*10**-5), xytext=(0.22, 5*10**-5),
    arrowprops=dict(facecolor='black', shrink=0.01,width=0.005, headwidth=3)
)

axs[0,1].annotate(
    '', xy=(0.5, 5*10**-5), xytext=(0.28, 5*10**-5),
    arrowprops=dict(facecolor='black', shrink=0.01,width=0.005, headwidth=3)
)

# Add text on the right saying "complex"
axs[0,1].text(0.1, 7*10**-5, 'Simple', fontsize=5, verticalalignment='center')
axs[0,1].text(0.33, 7*10**-5, 'Complex', fontsize=5, verticalalignment='center')

axs[0,0].grid(True, which="major", ls=":")
axs[0,1].grid(True, which="major", ls=":")


'''

LFC network setting

'''
print('Plotting LFC ...')
nodes = 240
model = 'LFC'
net = 'ke' #input("ws or mhk? ")
# Ks  = range(4,33,2) #[4,8,16,32]
perc = int(5)#[5,10,100]
centrality = 'degree'
t_b_s = ['top','bot'] #['top','bot','all']
Ks  = 16

# def LFC_plot(net,Ks,t_b,perc,model = 'LFC'):
plt.rc('text', usetex=True)
plt.rc('text.latex', preamble = r'\usepackage{mathptmx}')
# matplotlib.verbose.level = 'debug-annoying'

for selector,t_b in enumerate(t_b_s):
    ix=pd.IndexSlice
    intended_k = Ks
    networks = [net]
    network = net
    nodes = 240

    k = Ks
    my_new_colors = ['darkslateblue','darkcyan','coral','blue']

    ## Formating collective frequency response
    axs[1][selector].set_xscale('log')
    axs[1][selector].set_yscale('log')
    axs[1][selector].set_xlabel(r'Frequency $(\omega)$',labelpad=2)
    axs[1][selector].set_ylabel(r'Collective response $(H^2)$',labelpad=2.5)

    for h2, label_string in lfc_curves(t_b):
        if selector == 0:
            axs[1][selector].plot(h2)
        else:
            axs[1][selector].plot(h2, label=label_string)


        
    leg1 = axs[1][1].legend(title=r' $  C \;\, |\;\;\, T \;\,| \;\,\: \ell  \;\:\,  |\;\;\, R_{g} $', loc=[1.1,0],borderpad=0.2,markerscale=0.8,handlelength=0.9,handletextpad=0.4,fontsize=7)
    
    leg1.get_title().set_position((3.55, 0))
    leg1.get_title().set_fontsize('7')
    leg1.get_frame().set_facecolor('white')
    leg1.get_frame().set_alpha(1.0)
    leg1.get_frame().set_edgecolor('white')
    
    ## Set axis limits
    axs[1][selector].set_xlim([0.0001,2])
    axs[1][selector].set_ylim([0.017,520])

    ## Indentifying letters
    axs[0][0].text(-0.02,1.03,r'$\bf{(a)}$',transform=axs[0][0].transAxes,fontsize=8)
    axs[0][1].text(-0.02,1.03,r'$\bf{(b)}$',transform=axs[0][1].transAxes,fontsize=8)
    axs[1][0].text(-0.02,1.03,r'$\bf{(c)}$',transform=axs[1][0].transAxes,fontsize=8)
    axs[1][1].text(-0.02,1.03,r'$\bf{(d)}$',transform=axs[1][1].transAxes,fontsize=8)
    
    axs[1][selector].grid(True, which="major", ls=":")
    

    axs[1][selector].tick_params(axis='x',labelsize=7)
    axs[1][selector].tick_params(axis='y',labelsize=7)

    ## Title
 
    axs[1][selector].annotate(
        '', xy=(1*10**-4, 2.5*10**-2), xytext=(8*10**-3, 2.5*10**-2),
        arrowprops=dict(facecolor='black', shrink=0.01,width=0.005, headwidth=3)
    )

    axs[1][selector].annotate(
        '', xy=(2, 2.5*10**-2), xytext=(2*10**-2, 2.5*10**-2),
        arrowprops=dict(facecolor='black', shrink=0.01,width=0.005, headwidth=3)
    )

    # Add text on the right saying "complex"
    axs[1][selector].text(8*10**-4, 3.5*10**-2, 'Simple', fontsize=5, verticalalignment='center')
    axs[1][selector].text(1*10**-1, 3.5*10**-2, 'Complex', fontsize=5, verticalalignment='center')


if save:
    fig_path = f'../../figures'
    if not os.path.exists(fig_path):
        os.makedirs(fig_path)
    fig.savefig(fig_path + f'/Figure_S16.pdf')
    print('Figure_S16 saved')
else:
    fig.show()

In [ ]:
'''
Figure S17, S18
1k and 5k MHK models
'''

def LFC_plot(nodes,net,Ks,t_b,perc,model = 'LFC', centrality = 'degree', output = None):
    plt.rc('text', usetex=True)
    plt.rc('text.latex', preamble = r'\usepackage{mathptmx}')
    # matplotlib.verbose.level = 'debug-annoying'
    for in_k in Ks:
        ix=pd.IndexSlice
        intended_k = in_k
        networks = [net]
        seed = 1   # L5/L6 are single-realization sweeps (1 seed, 100 p)
        degrees = range(2,33,2) 
        print(f'{in_k}/{nodes}/{t_b}/{perc}/{networks[0]}')

        print('Loading data ..,')


        d = {'ID':[],'freq':[],'p':[]}
        network_gains = pd.DataFrame(data=d)
        network_gains.set_index(['ID','freq','p'],inplace=True)

        e = {'ID':[],'p':[]}
        network_props = pd.DataFrame(data=e)
        network_props.set_index(['ID','p'],inplace=True)

        insert = True
        landscape = True

        # main_props = ['CC','SP','Rg']
        main_props = ['CC','T','SP','Rg']
        aux_props = ['rawCC','rawSP','rawRg','k','p','rawT']
        props = main_props + aux_props
        # prop_label= {'CC':r'$\bar{C}$','SP':r'$\bar{\ell}$','Rg':r'$\bar{R}_g$'}
        prop_label= {'CC':r'$\bar{C}$','SP':r'$\bar{\ell}$','Rg':r'$\bar{R}_g$','T':r'$\bar{T}$'}
        for network in networks:
            for k in degrees:
                # print(k)
                # new_network_gains = pd.read_csv(f'networks_new/{nodes}/{k}/{network}_corr_gains.csv',sep='\t',index_col=[1])
                # new_props = pd.read_csv(f'networks_new/{nodes}/{k}/{network}_props.csv',sep='\t',index_col=[0,1])

                # The N=1000/5000 sweeps live in the realization tree like
                # everything else here, so they use the '<k>_seed<seed>'
                # naming that extract_lfc_csv.py writes.
                new_network_gains = pd.read_csv(f'../../nets/{model}/{nodes}/{network}/{k}_seed{seed}_{t_b}_{perc}_corr_gains_{centrality}.csv',sep='\t',index_col=[1])
                new_props = pd.read_csv(f'../../nets/{model}/{nodes}/{network}/{k}_seed{seed}_props.csv',sep='\t',index_col=[0,1])

                new_props['p'] = new_props.index.get_level_values(1)

                new_props['k'] = k
                new_props['rawCC'] = new_props.CC
                new_props['rawSP'] = new_props.SP
                new_props['rawRg'] = new_props.Rg
                new_props['rawT'] = new_props['T']
                new_props.CC = new_props.CC/new_props.CC.max()
                new_props['T'] = new_props['T']/new_props['T'].max()
                new_props.Rg = new_props.Rg/new_props.Rg.min()
                new_props.SP = new_props.SP/new_props.SP.min()

                for f in new_network_gains.index.unique():
                    new_network_gains.loc[f,'normH2'] = (new_network_gains.loc[f].H2/new_network_gains.loc[f,'H2'].max())


                if 'k' in new_network_gains.columns:
                    new_network_gains.loc[network_gains.isnull().k,'k']=k
                else:
                    new_network_gains['k'] = k

                network_props = pd.concat([network_props, new_props.loc[:,props]])
                # network_props = network_props.append(new_props.loc[:,props])

                new_network_gains = new_network_gains.reset_index()
                new_network_gains['freq'] = new_network_gains.freq.apply(lambda x: round(x,5))

                new_network_gains.set_index(['ID','freq'],inplace=True)

                # network_gains = network_gains.append(new_network_gains)
                network_gains = pd.concat([network_gains, new_network_gains])

        network_props.fillna(value=0,inplace = True)
        network_gains.fillna(value=0,inplace = True)

        print('Getting correlations ..,')
        freq16 = network_gains.index.get_level_values(1).unique()
        freqAll = network_gains.loc[network_gains.k==[x for x in degrees if x != intended_k][0]].index.get_level_values(1).unique()

        gg = pd.MultiIndex.from_tuples(list(zip(main_props*len(freqAll),['H2']*len(freqAll)*len(main_props),sorted(list(freqAll)*len(main_props)))))
        corr_index_k = pd.MultiIndex.from_tuples(list(zip(main_props*len(freq16),['H2']*len(freq16)*len(main_props),sorted(list(freq16)*len(main_props)))))
        network_corr_k = pd.DataFrame(index=corr_index_k)
        network_corr_lim = pd.DataFrame(index=gg)

        correlations = ['spearman']

        max_sp_lim = network_props.SP.max()
        min_sp_lim = network_props.SP.min()

        max_cc_lim = network_props.CC.max()
        min_cc_lim = network_props.CC.min()

        max_rg = network_props.Rg.max()
        min_rg = network_props.Rg.min()

        if network == 'mhk':
            specialp = SPECIAL_P[(network, intended_k, nodes)]
            min_rg = 1.2
            max_rg = 2.2
            min_cc_lim = 0.6
            max_cc_lim = 1
        elif network == 'ws':
            min_cc_lim = 0.6
            max_cc_lim = 1
            min_rg = 1.2
            max_rg = 2.2
            specialp = SPECIAL_P[(network, intended_k, nodes)]

        elif network == 'ke':
            specialp = SPECIAL_P[(network, intended_k, nodes)]

        # network_props.query(f'{min_cc_lim} < CC < {max_cc_lim} and {min_sp_lim}  < SP < {max_sp_lim} and {min_rg} < Rg < {max_rg}').loc[:,('CC','SP','Rg')].corr(method='spearman').round(2)

        # query for  range of clutsering
        query_string = f'{min_cc_lim} < CC < {max_cc_lim} and {min_sp_lim} < SP < {max_sp_lim} and {min_rg} < Rg < {max_rg}'
        # query_string16 = f'k == {intended_k}'

        # lim_IDs = network_props.query(query_string).index.get_level_values(0)
        # temp_network_ins = network_gains.loc[ix[lim_IDs,:]]
        # corr_values_k = []
        corr_values_lim = []
        for coef in correlations:
            # ## Need to be sorted, because network_corr_... expects frequencies to be ordered
            # for f in freq16.sort_values():
            #     corr_values_k += list(network_props.query(query_string16).loc[:,main_props].corrwith(network_gains.loc[ix[:,f],'H2'],method=coef).values)

            for f in freqAll.sort_values():
                corr_values_lim += list(network_props.query(query_string).loc[:,main_props].corrwith(network_gains.loc[ix[:,f],'normH2'],method=coef).values)

        # network_corr_k.loc[corr_index_k,coef] = np.reshape(corr_values_k,(len(corr_values_k),1))
        network_corr_lim.loc[gg,coef] = np.reshape(corr_values_lim,(len(corr_values_lim),1))

        print(f'Plotting ..,')

        my_new_colors = ['darkslateblue','crimson','darkcyan','coral']


        fig,axs = plt.subplots(figsize=(5.5*1.2,2.1*1.7),ncols=2,nrows=2,sharex=False,sharey=False)
        fig.subplots_adjust(left=None, bottom=None, right=None, top=None, wspace=0.21, hspace=0.3)
        ## Getting correct plot setup
        gs = axs[0][0].get_gridspec()
        axs[0][0].remove()
        axs[1][0].remove()
        bigax = fig.add_subplot(gs[0:,0])

        ax_eig = axs[0][1]
        axins = inset_axes(ax_eig, width= .75, height= .4, loc='upper left',
                    bbox_to_anchor=(0.07, 1), bbox_transform=ax_eig.transAxes)

        ax_corr = axs[1][1]

        ## Formating collective frequency response
        bigax.set_xscale('log')
        bigax.set_yscale('log')
        bigax.set_xlabel(r'Frequency $(\omega)$',labelpad=2)
        bigax.set_ylabel(r'Collective response $(H^2(\omega))$',labelpad=2.5)



        specialk = intended_k
        directory_path = f'../../nets/{model}/{nodes}/{network}/{specialk}_seed{seed}/'
        sorted_filenames = get_sorted_filenames(directory_path)

        network_gains = network_gains.reset_index().set_index(['ID','freq','p'])
        prop_special = network_props.loc[network_props.k==specialk].groupby(level=1).mean()
        # H2 as stored, with NO (100 / perc) rescaling -- same reasoning as
        # the S16 cell. The CSV holds H2 already AVERAGED over the selected
        # leaders and each leader's H2 is itself a whole-network sum, so no
        # sample-size correction applies. H2 = ||h||^2 over the N-1 followers
        # and each h_i -> 1 as omega -> 0, so H2 <= N - 1 (999 at N=1000,
        # 4999 at N=5000), which the stored data honours.
        gain_special = network_gains.loc[network_gains.k==specialk].groupby(level=[2,1]).mean()
        for th,p in enumerate(gain_special.index.get_level_values(0).unique()[specialp]):
            flag_l_label = 0

            C_label = str(prop_special.loc[p].rawCC.round(2))
            if len(C_label) < 4:
                C_label =  C_label + '0'
            l_label = str(prop_special.loc[p].rawSP.round(2))

            if len(l_label) < 5:
                flag_l_label=1

            r_label = str((prop_special.loc[p].rawRg.mean()/1000).round(1))
            # if len(r_label) > 3:
            #     r_label = r_label[:2]
            r_label = r_label + r'\! \times \! 10^{3}'

            if flag_l_label:
                label_string =fr'${C_label}  |  {prop_special.loc[p].rawT.round(2)} \;    |  {prop_special.loc[p].rawSP:.2f}  \, |  {r_label}$'

            pol_fig_legend_label = label_string 

            G = gt.load_graph(gt_path_for_p(directory_path, p))
            print(gt_path_for_p(directory_path, p))
            eig_lap = np.linalg.eigvalsh(gt.laplacian(G, norm=True).todense())

            bigax.plot(gain_special.loc[p].H2,label=label_string)
            ax_eig.hist(eig_lap, bins=100, density=True, alpha=0.5)
            axins.hist(eig_lap, bins=100, density=True, alpha=0.5,label=pol_fig_legend_label)

        leg1 = bigax.legend(title=r'$ \;\,\;\;\; C \;\,\;\,\;\; | \;\,\,\;\, T  \;\:\,\;\,\,\;\,  | \,\;\,\;\ \ell   \,\;\,\ | \;\,\,\;\, R_{g} $', loc=[0.025,0.07],borderpad=0.2,markerscale=0.8,handlelength=0.9,handletextpad=0.4,fontsize=6)

        leg1.get_title().set_position((3.55, 0))
        leg1.get_title().set_fontsize('6')
        leg1.get_frame().set_facecolor('white')
        leg1.get_frame().set_alpha(1.0)
        leg1.get_frame().set_edgecolor('white')

        for idx,metric in enumerate(main_props):
            ax_corr.plot(network_corr_lim.loc[(metric,'H2')],label=prop_label[metric],zorder=[3,2,4,3,5][idx%5],c=my_new_colors[idx%5],linewidth=2,markersize=[5,5,5,5][idx%5],ls=[':', '-.', '--', 'solid'][idx%5],markeredgewidth=[1,1,1,1][idx%4],markerfacecolor='none')


        leg2=ax_corr.legend(title=r'$\bar{\chi}$',loc=[1.05,0.05],borderpad=0.2,markerscale=1,handlelength=2,handletextpad=0.4,fontsize=6)
        leg2.get_title().set_position((0, 0))
        leg2.get_title().set_fontsize('7')
        leg2.get_frame().set_facecolor('white')
        leg2.get_frame().set_alpha(1.0)
        leg2.get_frame().set_edgecolor('white')
        
        ## Setup axises
        


        ax_eig.set_ylabel(r'Density',labelpad=8)
        ax_eig.set_xlabel(r'Normalized Laplacian eigenvalues',labelpad=2,math_fontfamily='cm')
        ax_eig.set_ylim([0,3.7])
        ax_eig.set_xlim([-0.02,2.05])

        axins.set_xlim(-0.02, 0.62)
        axins.set_ylim(0, 0.5)
        # axins.set_ylabel(r'Density',labelpad=-8,fontsize=3,)
        # axins.set_xlabel(r'Normalized Eigenvalues',labelpad=-5,fontsize=3,math_fontfamily='cm')
        axins.set_xticks([0.0,0.2,0.4,0.6])
        axins.set_yticks([0,0.25,0.5])
        axins.tick_params(axis='x',labelsize=5)
        axins.tick_params(axis='y',labelsize=5)
        ax_eig.indicate_inset_zoom(axins, edgecolor="black")

        arrow = FancyArrowPatch(
        (0.13, 0.16),  # Start point - middle bottom of inset
        (0.25, 0.50),  # End point - straight down below inset
        arrowstyle='->,head_width=2,head_length=5',
        transform=ax_eig.transAxes,
        color='black',
        linewidth=0.5,
        zorder=5
        )
        ax_eig.add_patch(arrow)

        
        # ax_corr.set_ylabel(r'Correlation $r_s$',labelpad=2.5)
        ax_corr.set_ylabel(r'Correlation $r_s(\bar{H}^2(\omega),\bar{\chi})$',labelpad=2.5)
        ax_corr.set_xscale('log')
        ax_corr.set_xlabel(r'Frequency $( \omega )$',labelpad=2, math_fontfamily='cm')

        ## Set axis limits
        bigax.set_xlim([0.0001,2])
        # upper limit tracks the N - 1 ceiling rather than a fixed number,
        # so S17 (N=1000) and S18 (N=5000) are each framed to their own bound
        bigax.set_ylim([0.017, 2 * nodes])

        ax_corr.set_ylim([-1.5,1.05])
        ax_corr.set_xlim([0.001,0.2])

        ## Title



        ## Indentifying letters
        bigax.text(-0.02,1.04,r'$\bf{(a)}$',transform=bigax.transAxes,fontsize=8)
        ax_eig.text(-0.02,1.04,r'$\bf{(b)}$',transform=ax_eig.transAxes,fontsize=8)
        ax_corr.text(-0.02,1.04,r'$\bf{(c)}$',transform=ax_corr.transAxes,fontsize=8)

        ax_corr.yaxis.set_major_locator(matplotlib.ticker.FixedLocator([-1,0,1]))

        ax_corr.yaxis.set_minor_locator(matplotlib.ticker.NullLocator())

        ## text boxes
        box_props = dict(alpha=1,facecolor='w',linewidth=0,zorder=100000,boxstyle='round',pad=0.4)
        bigax.grid(True, which="major", ls=":")
        ax_eig.grid(False)
        ax_corr.grid(False)

        bigax.tick_params(axis='x',labelsize=7)
        ax_eig.tick_params(axis='x',labelsize=7)
        ax_corr.tick_params(axis='x',labelsize=7)
        bigax.tick_params(axis='y',labelsize=7)
        ax_eig.tick_params(axis='y',labelsize=7)
        ax_corr.tick_params(axis='y',labelsize=7)

        ax_corr.fill_between(x=[0.015,2.9],y1=[-4,-4],y2=[-1.72,-1.72],facecolor='w',zorder=-10)


        bigax.annotate(
            '', xy=(1*10**-4, 2.5*10**-2), xytext=(8*10**-3, 2.5*10**-2),
            arrowprops=dict(facecolor='black', shrink=0.01,width=0.005, headwidth=3)
        )

        bigax.annotate(
            '', xy=(2, 2.5*10**-2), xytext=(2*10**-2, 2.5*10**-2),
            arrowprops=dict(facecolor='black', shrink=0.01,width=0.005, headwidth=3)
        )

        # Add text on the right saying "complex"
        bigax.text(8*10**-4, 3*10**-2, 'Simple', fontsize=5, verticalalignment='center')
        bigax.text(1*10**-1, 3*10**-2, 'Complex', fontsize=5, verticalalignment='center')




        ax_corr.annotate(
            '', xy=(10**-3, -1.3), xytext=(8*10**-3, -1.3),
            arrowprops=dict(facecolor='black', shrink=0.01,width=0.005, headwidth=3)
        )

        ax_corr.annotate(
            '', xy=(0.2, -1.3), xytext=(2*10**-2, -1.3),
            arrowprops=dict(facecolor='black', shrink=0.01,width=0.005, headwidth=3)
        )

        # Add text on the right saying "complex"
        ax_corr.text(2.5*10**-3, -1.2, 'Simple', fontsize=5, verticalalignment='center')
        ax_corr.text(4*10**-2, -1.2, 'Complex', fontsize=5, verticalalignment='center')


        plt.show()
        print(output)
        fig_path = f"../../figures"
        if not os.path.exists(fig_path):
            os.makedirs(fig_path)
        fig.savefig(f'{fig_path}/{output}.pdf')
        print(f'{fig_path}/{output}.pdf')
    return print('pdf file saved')


output_map = {1000: 'Figure_S17', 5000: 'Figure_S18'}
ks = [16]
for nodes in [1000, 5000]:
    for net in ['mhk']:
        for t_b in ['top']:
            perc = int(5)
            LFC_plot(nodes, net, ks, t_b, perc, model='LFC', centrality='degree', output=output_map[nodes])

In [26]:
'''
Figure S19
Collective response + spectrum for 4 of the 8 real-world networks not
already shown in Figure 9 (celegansneural, dolphins, football, uni_email).
Continued in Figure S20.

Reads ../../nets/real_world/*.gt (same data as Figure 9, see that cell's note).
'''

from extract_lfc_csv import get_selected_gains, get_graph_props


def normalized_laplacian_eigenvalues(G):
    L = gt.laplacian(G, norm=True)
    return np.linalg.eigvalsh(L.todense())


def raw_gain_curve(G, t_b, perc=5):
    gains = get_selected_gains(G, t_b, perc)
    if gains is None:
        return None, None
    freqs = np.array(sorted(gains))
    h2 = np.array([gains[f] for f in freqs])
    return freqs, h2


def characteristics_label(G):
    props = get_graph_props(G)
    rg = props['Rg']
    exp = int(np.floor(np.log10(abs(rg)))) if rg else 0
    mantissa = rg / 10**exp
    rg_label = rf'{mantissa:.1f}\!\times\!10^{{{exp}}}'
    return (rf'$C={props["CC"]:.2f} \,|\, T={props["T"]:.2f} \,|\, '
            rf'\ell={props["SP"]:.2f} \,|\, R_g={rg_label}$')


def draw_response(ax, G, title):
    curves = {}
    for t_b in ('top', 'bot'):
        freqs, h2 = raw_gain_curve(G, t_b)
        if freqs is not None:
            curves[t_b] = (freqs, h2)
    shared_max = max(h2.max() for _, h2 in curves.values())
    for t_b, style in [('top', 'solid'), ('bot', 'dashed')]:
        if t_b not in curves:
            continue
        freqs, h2 = curves[t_b]
        ax.plot(freqs, h2 / shared_max, linestyle=style, linewidth=1.3, label=f'{t_b} 5\\%')

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r'Frequency $(\omega)$', labelpad=2, fontsize=9)
    ax.set_ylabel(r'Normalized collective response $(H^2 / H^2_{max})$', labelpad=2.5, fontsize=9)
    ax.set_title(title, fontsize=11)

    leg = ax.legend(title=characteristics_label(G), loc='lower left', fontsize=8,
                     borderpad=0.2, markerscale=0.8, handlelength=2.0, handletextpad=0.4)
    leg.get_title().set_fontsize(8)
    leg.get_frame().set_facecolor('white')
    leg.get_frame().set_alpha(1.0)
    leg.get_frame().set_edgecolor('white')

    ax.grid(True, which='major', ls=':')
    ax.tick_params(axis='x', labelsize=8)
    ax.tick_params(axis='y', labelsize=8)


def draw_spectrum(ax, G):
    eig = normalized_laplacian_eigenvalues(G)
    ax.hist(eig, bins=100, density=True, alpha=0.5, color='tab:blue')
    ax.set_xlabel('Normalized Laplacian eigenvalues', labelpad=2, fontsize=9)
    ax.set_ylabel('Density', labelpad=2.5, fontsize=9)
    ax.set_xlim(-0.02, 2.05)
    ax.grid(True, which='major', ls=':')
    ax.tick_params(axis='x', labelsize=8)
    ax.tick_params(axis='y', labelsize=8)


rw_root = '../../nets/real_world'
NETWORKS = [
    (f'{rw_root}/small_world/celegansneural/celegansneural.gt', 'C. elegans neurons (1986)'),
    (f'{rw_root}/small_world/dolphins/dolphins.gt', 'Dolphin social network'),
    (f'{rw_root}/small_world/football/football.gt', 'NCAA college football 2000'),
    (f'{rw_root}/scale_free/uni_email/uni_email.gt', 'Email network (Uni. R-V, Spain, 2003)'),
]
graphs = [gt.load_graph(p) for p, _ in NETWORKS]
titles = [label for _, label in NETWORKS]
nrows = len(NETWORKS)

fig, axs = plt.subplots(figsize=(11, 13), nrows=nrows, ncols=2)
fig.subplots_adjust(wspace=0.35, hspace=0.3)

for row in range(nrows):
    for col, letter in enumerate('ab'):
        axs[row][col].text(-0.02, 1.06, rf'$\bf({row + 1}{letter})$',
                            transform=axs[row][col].transAxes, fontsize=10)
    draw_response(axs[row][0], graphs[row], titles[row])
    draw_spectrum(axs[row][1], graphs[row])

plt.show()

fig_path = f"../../figures"
if not os.path.exists(fig_path):
    os.makedirs(fig_path)
fig.savefig(f'{fig_path}/Figure_S19.pdf', bbox_inches='tight')
print(f'saved {fig_path}/Figure_S19.pdf')


saved ../../figures/Figure_S19.pdf


In [27]:
'''
Figure S20
Collective response + spectrum for the other 4 of the 8 real-world
networks not already shown in Figure 9 (collins_yeast, polblogs,
faa_routes, interactome_yeast). Continues Figure S19.

Reads ../../nets/real_world/*.gt (same data as Figure 9, see that cell's note).
'''

from extract_lfc_csv import get_selected_gains, get_graph_props


def normalized_laplacian_eigenvalues(G):
    L = gt.laplacian(G, norm=True)
    return np.linalg.eigvalsh(L.todense())


def raw_gain_curve(G, t_b, perc=5):
    gains = get_selected_gains(G, t_b, perc)
    if gains is None:
        return None, None
    freqs = np.array(sorted(gains))
    h2 = np.array([gains[f] for f in freqs])
    return freqs, h2


def characteristics_label(G):
    props = get_graph_props(G)
    rg = props['Rg']
    exp = int(np.floor(np.log10(abs(rg)))) if rg else 0
    mantissa = rg / 10**exp
    rg_label = rf'{mantissa:.1f}\!\times\!10^{{{exp}}}'
    return (rf'$C={props["CC"]:.2f} \,|\, T={props["T"]:.2f} \,|\, '
            rf'\ell={props["SP"]:.2f} \,|\, R_g={rg_label}$')


def draw_response(ax, G, title):
    curves = {}
    for t_b in ('top', 'bot'):
        freqs, h2 = raw_gain_curve(G, t_b)
        if freqs is not None:
            curves[t_b] = (freqs, h2)
    shared_max = max(h2.max() for _, h2 in curves.values())
    for t_b, style in [('top', 'solid'), ('bot', 'dashed')]:
        if t_b not in curves:
            continue
        freqs, h2 = curves[t_b]
        ax.plot(freqs, h2 / shared_max, linestyle=style, linewidth=1.3, label=f'{t_b} 5\\%')

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r'Frequency $(\omega)$', labelpad=2, fontsize=9)
    ax.set_ylabel(r'Normalized collective response $(H^2 / H^2_{max})$', labelpad=2.5, fontsize=9)
    ax.set_title(title, fontsize=11)

    leg = ax.legend(title=characteristics_label(G), loc='lower left', fontsize=8,
                     borderpad=0.2, markerscale=0.8, handlelength=2.0, handletextpad=0.4)
    leg.get_title().set_fontsize(8)
    leg.get_frame().set_facecolor('white')
    leg.get_frame().set_alpha(1.0)
    leg.get_frame().set_edgecolor('white')

    ax.grid(True, which='major', ls=':')
    ax.tick_params(axis='x', labelsize=8)
    ax.tick_params(axis='y', labelsize=8)


def draw_spectrum(ax, G):
    eig = normalized_laplacian_eigenvalues(G)
    ax.hist(eig, bins=100, density=True, alpha=0.5, color='tab:blue')
    ax.set_xlabel('Normalized Laplacian eigenvalues', labelpad=2, fontsize=9)
    ax.set_ylabel('Density', labelpad=2.5, fontsize=9)
    ax.set_xlim(-0.02, 2.05)
    ax.grid(True, which='major', ls=':')
    ax.tick_params(axis='x', labelsize=8)
    ax.tick_params(axis='y', labelsize=8)


rw_root = '../../nets/real_world'
NETWORKS = [
    (f'{rw_root}/scale_free/collins_yeast/collins_yeast.gt', 'Collins yeast interactome (2007)'),
    (f'{rw_root}/scale_free/polblogs/polblogs.gt', 'Political blogs network (2004)'),
    (f'{rw_root}/scale_free/faa_routes/faa_routes.gt', 'FAA Preferred Routes (2010)'),
    (f'{rw_root}/scale_free/interactome_yeast/interactome_yeast.gt', 'Coulomb yeast interactome (2005)'),
]
graphs = [gt.load_graph(p) for p, _ in NETWORKS]
titles = [label for _, label in NETWORKS]
nrows = len(NETWORKS)

fig, axs = plt.subplots(figsize=(11, 13), nrows=nrows, ncols=2)
fig.subplots_adjust(wspace=0.35, hspace=0.3)

for row in range(nrows):
    for col, letter in enumerate('ab'):
        axs[row][col].text(-0.02, 1.06, rf'$\bf({row + 1}{letter})$',
                            transform=axs[row][col].transAxes, fontsize=10)
    draw_response(axs[row][0], graphs[row], titles[row])
    draw_spectrum(axs[row][1], graphs[row])

plt.show()

fig_path = f"../../figures"
if not os.path.exists(fig_path):
    os.makedirs(fig_path)
fig.savefig(f'{fig_path}/Figure_S20.pdf', bbox_inches='tight')
print(f'saved {fig_path}/Figure_S20.pdf')


saved ../../figures/Figure_S20.pdf


In [ ]:
'''
Figure S21
'''

from scipy.integrate import solve_ivp
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize
import matplotlib.patheffects as pe
import graph_tool.spectral

GRAPH_PATH = '../../nets/LFC/240/mhk/16_seed1/k16_p1.000000.gt'
# omega=5e-2 was quasi-static: every follower tracked the leader almost
# in phase, so the whole network's response amplitudes spanned only
# 1.12x and no choice of three nodes could be told apart. At 5e-1 the
# spread is 7.1x, which separates the curves while keeping the weakest
# one clearly visible on shared linear axes (omega=1 would spread them
# 19x, dropping the weak curve to ~5% of the strong one's height).
OMEGA = 5e-1
N_PERIODS = 5
T_MAX = N_PERIODS * 2 * np.pi / OMEGA
N_EVAL = 2000 * N_PERIODS

# --- panel (b): LTM cascade constants ---
LTM_GRAPH_PATH = '../../nets/LTM/1000/ws/16_seed1/p0.193600.gt'
LTM_THRESHOLD = np.linspace(0.01, 0.5, 16)
LTM_THETA = LTM_THRESHOLD[10]  # 0.3367
# The seed is DERIVED from the graph, not hardcoded - see select_ltm_seed()
# below. A literal node id would be tied to one particular graph and would
# silently become an ordinary low-degree node if the plotted p ever moved.
LTM_LAYOUT_SEED = 1  # sfdp_layout's only stochastic step - fixed for reproducibility
LTM_NODE_SIZE = 260  # standalone-figure default; this cell overrides via node_size=170 below
LTM_EXAMPLE_STEP = 14  # which step label the annotation arrow points at
LTM_STEP_LABEL_FS = 6
LTM_LABEL_FS = 9
LTM_TICK_FS = 8
LTM_LEGEND_FS = 8
LTM_SEED_COLOR = 'black'
LTM_EDGE_COLOR = '#888888'
LTM_EDGE_ALPHA = 0.5
LTM_EDGE_LINEWIDTH = 0.7


def build_normalized_laplacian(G):
    L = gt.laplacian(G, norm=False)
    L = (L / L.diagonal()).T  # random-walk normalization D^-1 L
    return L.toarray()


def driven_system(G, leader):
    """Split the normalized Laplacian into the follower block A and the
    leader-coupling column B, plus the follower->row index map."""
    L = build_normalized_laplacian(G)
    N = G.num_vertices()
    ida = np.arange(N) != leader
    idb = np.arange(N) == leader
    return L[np.ix_(ida, ida)], L[np.ix_(ida, idb)].flatten(), np.arange(N)[ida]


def steady_state_amplitude(G, leader, omega=None):
    """Per-follower steady-state amplitude |X_i|, from (A + i*omega*I) X = -B."""
    if omega is None:
        omega = OMEGA
    A, B, _ = driven_system(G, leader)
    X = np.linalg.solve(A.astype(complex) + 1j * omega * np.eye(A.shape[0]), -B)
    return np.abs(X)


def select_leader_and_followers(G):
    """Leader = highest-degree node; followers = three nodes whose response
    amplitudes at OMEGA are evenly separated in log, spanning the full range.

    Selection is on response amplitude rather than shortest-path distance
    from the leader (near / mid / far). Distance does not discriminate on
    this graph: the hub has degree 89 and the network has radius 3, so 149
    of 239 followers sit at distance 2 --
    the three picks came out at amplitude 0.4007 / 0.3985 / 0.4002, three
    curves drawn on top of each other.

    Selecting on the response itself guarantees separation. The distance
    story survives in the legend rather than driving the choice: amplitude
    correlates -0.75 with distance, yet two of the three nodes picked here
    sit at the SAME distance from the leader and still differ ~2.7x, which
    is precisely the point -- hop count alone does not set how strongly a
    node follows.
    """
    degrees = G.get_total_degrees(G.get_vertices())
    leader = int(np.argmax(degrees))
    dist = gt.shortest_distance(G, source=G.vertex(leader)).a.copy()

    amp = steady_state_amplitude(G, leader)
    _, _, follower_ids = driven_system(G, leader)

    log_amp = np.log10(amp)
    lo, hi = float(log_amp.min()), float(log_amp.max())
    picks = {}
    for label, frac in (('strong', 1.0), ('medium', 0.5), ('weak', 0.0)):
        j = int(np.argmin(np.abs(log_amp - (lo + (hi - lo) * frac))))
        picks[label] = int(follower_ids[j])

    return leader, picks, dist, degrees


def simulate(G, leader):
    A, B, follower_ids = driven_system(G, leader)
    index_of = {node: i for i, node in enumerate(follower_ids)}

    def u(t):
        return np.sin(OMEGA * t)

    def rhs(t, x):
        return -(A @ x + B * u(t))

    # Steady-state phasor X: (A + i*omega*I) X = -B, x_ss(t) = Im(X e^{i*omega*t}).
    # Starting at x_f(0) = Im(X) puts the integration exactly on the periodic
    # orbit already, so there's no startup transient to plot.
    A_complex = A.astype(complex) + 1j * OMEGA * np.eye(A.shape[0])
    X = np.linalg.solve(A_complex, -B)
    x0 = np.imag(X)

    t_eval = np.linspace(0, T_MAX, N_EVAL)
    sol = solve_ivp(rhs, (0, T_MAX), x0, t_eval=t_eval, method='RK45', rtol=1e-8, atol=1e-10)

    return t_eval, u(t_eval), sol.y, index_of


def linear_threshold_model(G, threshold, seed_nodes, init_spread=True, max_iter=None):
    '''
    Verbatim logic from run_ltm_cascade.py's linear_threshold_model,
    specialized to a single threshold value and returning just the
    per-node infection_step array (not graph_tool's grouped vector
    property, which only makes sense when sweeping many thresholds at
    once).
    '''
    if max_iter is None:
        max_iter = G.num_vertices()

    degree_dist = G.get_out_degrees(G.get_vertices())
    T = np.array((gt.adjacency(G).T.toarray() / degree_dist).T)

    infected = np.zeros(G.num_vertices(), dtype=int)
    infection_step = np.full(G.num_vertices(), np.inf, dtype=float)
    infected[seed_nodes] = 1
    infection_step[seed_nodes] = -1

    if init_spread:
        infected[T.dot(infected) > 0] = 1
        infection_step[np.logical_and(infected > 0, np.isinf(infection_step))] = 0
        i = 1
    else:
        i = 0
    while (not all(infected) and (i < max_iter) and i - 1 in infection_step):
        infected[T.dot(infected) >= threshold] = 1
        infection_step[np.logical_and(infected > 0, np.isinf(infection_step))] = i
        i += 1

    return infection_step


def select_ltm_seed(G, threshold):
    """Highest-degree node, ties broken toward the largest cascade.

    ws k=16 is near-regular -- degrees run 11-21 about a mean of 16, so the
    maximum is only 1.31x the mean and SIX nodes tie for it (299, 384, 768,
    816, 909, 976). Which one np.argmax returns is an accident of node
    ordering, and it matters: at this threshold their cascades range from
    24 nodes / 1 step to 193 nodes / 69 steps. The tie is therefore broken
    explicitly rather than left to chance.

    (Contrast panel (a)'s MHK network, which is scale-free: there the
    maximum degree is 5.6x the mean and unique, so "the highest-degree
    node" needs no tie-break to be well defined.)
    """
    deg = G.get_out_degrees(G.get_vertices())
    tied = np.flatnonzero(deg == deg.max())
    return max(
        (int(np.isfinite(linear_threshold_model(G, threshold, [int(s)])).sum()), int(s))
        for s in tied
    )[1]


def run_cascade(G):
    seed_node = select_ltm_seed(G, LTM_THETA)
    infection_step = linear_threshold_model(G, LTM_THETA, [seed_node])
    return seed_node, infection_step


def draw_cascade(ax, G, seed_node, infection_step, seed_size_mult=1.6, node_size=None):
    '''
    Draws the cascade network diagram onto a caller-provided ax (no fig
    creation, no title, no tight_layout/savefig). seed_size_mult scales the
    seed star relative to node_size (defaults to LTM_NODE_SIZE if not
    given).
    '''
    if node_size is None:
        node_size = LTM_NODE_SIZE
    seed_mask = infection_step == -1
    step_mask = (infection_step >= 0) & np.isfinite(infection_step)
    keep_mask = seed_mask | step_mask  # drop never-infected nodes entirely

    # Layout computed on just the displayed (seed + infected) subgraph, not
    # the full 1000-node graph - these ~137 nodes have much more room to
    # spread out on their own than when sfdp has to simultaneously place
    # the other ~863 never-shown nodes too.
    keep_prop = G.new_vertex_property('bool', vals=keep_mask.astype(bool))
    sub = gt.GraphView(G, vfilt=keep_prop)
    # sfdp_layout's only randomness is its initial layout, drawn via
    # numpy.random (not graph_tool's own RNG) - seeding both here is what
    # actually makes the node positions (and so the figure) reproducible.
    np.random.seed(LTM_LAYOUT_SEED)
    gt.seed_rng(LTM_LAYOUT_SEED)
    # sfdp_layout's force computation is OpenMP-parallelized - seeding the
    # RNG alone does NOT make it reproducible, since floating-point
    # summation order varies with thread scheduling (verified: with the
    # default 4 threads, two back-to-back calls with the same seed gave
    # a max coordinate difference of ~125 units; forcing 1 thread gives
    # an exact match, diff 0.0). Forcing single-threaded here is what
    # actually makes the node arrangement identical across runs.
    gt.openmp_set_num_threads(1)
    pos = gt.sfdp_layout(sub)
    xy = np.zeros((G.num_vertices(), 2))
    for v in sub.vertices():
        xy[int(v)] = pos[v].a

    edges = np.array([(int(e.source()), int(e.target())) for e in sub.edges()])
    segments = xy[edges]

    max_step = int(np.max(infection_step[step_mask]))

    fig = ax.figure

    edge_lc = LineCollection(segments, colors=LTM_EDGE_COLOR, linewidths=LTM_EDGE_LINEWIDTH,
                              alpha=LTM_EDGE_ALPHA, zorder=1)
    ax.add_collection(edge_lc)

    cmap = plt.cm.plasma
    norm = Normalize(vmin=0, vmax=max_step)
    sc = ax.scatter(xy[step_mask, 0], xy[step_mask, 1], s=node_size,
                     c=infection_step[step_mask], cmap=cmap, norm=norm,
                     linewidths=0.4, edgecolors='black', zorder=3)

    ax.scatter(xy[seed_mask, 0], xy[seed_mask, 1], s=node_size * seed_size_mult,
               marker='*', color=LTM_SEED_COLOR, edgecolors='white', linewidths=0.6,
               zorder=4, label='seed')

    label_stroke = [pe.withStroke(linewidth=1.2, foreground='white')]
    for node in np.flatnonzero(step_mask):
        ax.text(xy[node, 0], xy[node, 1], str(int(infection_step[node])),
                ha='center', va='center', fontsize=LTM_STEP_LABEL_FS, color='black',
                path_effects=label_stroke, zorder=5)
    for node in np.flatnonzero(seed_mask):
        ax.text(xy[node, 0], xy[node, 1], 'S',
                ha='center', va='center', fontsize=LTM_STEP_LABEL_FS, color='white',
                fontweight='bold', zorder=5)

    cbar = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.02)
    cbar.set_label('LTM infection step', fontsize=LTM_LABEL_FS)
    cbar.ax.tick_params(labelsize=LTM_TICK_FS)
    # Force a tick at the top of the scale - the default locator's ticks
    # don't reliably land exactly on vmax, so the max step wouldn't
    # otherwise be labeled.
    auto_ticks = [tk for tk in cbar.get_ticks() if tk < max_step - 0.05 * max_step]
    cbar.set_ticks(auto_ticks + [max_step])

    # Worked example pointing at one representative node, explaining how to
    # read the numbers/colors (in place of the removed in-axes caption box).
    # Point at a node carrying the label LTM_EXAMPLE_STEP, picking the one
    # nearest the TOP-LEFT of the layout so it sits under the annotation and
    # the arrow stays short. Falls back to the middle of the step range if
    # that particular step happens to be empty in this cascade.
    cand = np.flatnonzero(step_mask & (infection_step == LTM_EXAMPLE_STEP))
    if cand.size == 0:
        pool = np.flatnonzero(step_mask)
        cand = pool[np.argsort(np.abs(infection_step[pool] - max_step / 2))[:1]]
    x0, x1 = xy[keep_mask, 0].min(), xy[keep_mask, 0].max()
    y0, y1 = xy[keep_mask, 1].min(), xy[keep_mask, 1].max()
    sx, sy = (x1 - x0) or 1.0, (y1 - y0) or 1.0
    # distance from the top-left corner, in normalized layout coordinates
    corner = (xy[cand, 0] - x0) / sx + (y1 - xy[cand, 1]) / sy
    example_node = int(cand[np.argmin(corner)])
    ax.annotate(
        rf'label = LTM step at infection'
        '\n'
        rf'(color encodes the same value)',
        xy=(xy[example_node, 0], xy[example_node, 1]), xycoords='data',
        xytext=(0.02, 0.98), textcoords='axes fraction',
        fontsize=LTM_LABEL_FS, va='top', ha='left',
        arrowprops=dict(arrowstyle='->', color='black', linewidth=0.8),
        zorder=6
    )

    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    leg = ax.legend(loc='lower left', fontsize=LTM_LEGEND_FS, borderpad=0.4, handletextpad=0.4)
    leg.get_frame().set_facecolor('white')
    leg.get_frame().set_alpha(1.0)
    leg.get_frame().set_edgecolor('white')


G = gt.load_graph(GRAPH_PATH)
leader, picks, dist, degrees = select_leader_and_followers(G)
leader_degree = int(degrees[leader])
t, leader_signal, x, index_of = simulate(G, leader)

colors = {'strong': 'tab:red', 'medium': 'tab:orange', 'weak': 'tab:blue'}
leader_label = (rf'leader (node {leader}, deg {leader_degree}): '
                rf'$u(t)=\sin(\omega t)$, $\omega={OMEGA}$')

fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(7, 8.3),
                                      gridspec_kw={'height_ratios': [1, 1.5]})

ax_top.axhline(0, color='gray', linewidth=0.6)
ax_top.plot(t, leader_signal, color='black', linewidth=1.0, linestyle='dashed',
            alpha=0.6, label=leader_label)
peak_amps = {}
for label in ('strong', 'medium', 'weak'):
    node = picks[label]
    d = int(dist[node])
    response = x[index_of[node]]
    peak = np.max(np.abs(response))
    peak_amps[label] = peak
    ax_top.plot(t, response, color=colors[label], linewidth=1.3,
                label=f'{label} (node {node}, deg {int(degrees[node])}, '
                      f'dist {d}, peak {peak:.1e})')
ax_top.set_xlabel(r'Time $t$', labelpad=2, fontsize=9)
ax_top.set_ylabel(r'Node response $x_i(t)$', labelpad=2.5, fontsize=9)
ax_top.grid(True, which='major', ls=':')
ax_top.tick_params(axis='x', labelsize=8)
ax_top.tick_params(axis='y', labelsize=8)
leg = ax_top.legend(loc='lower left', fontsize=6, borderpad=0.3, handlelength=2.0,
                     handletextpad=0.4)
leg.get_frame().set_facecolor('white')
leg.get_frame().set_alpha(1.0)
leg.get_frame().set_edgecolor('white')
ax_top.text(-0.08, 1.03, r'\textbf{(a)}', transform=ax_top.transAxes, fontsize=9)

G_ltm = gt.load_graph(LTM_GRAPH_PATH)
seed_node_ltm, infection_step_ltm = run_cascade(G_ltm)
draw_cascade(ax_bot, G_ltm, seed_node_ltm, infection_step_ltm, seed_size_mult=1.6, node_size=170)
ax_bot.text(-0.02, 1.0, r'\textbf{(b)}', transform=ax_bot.transAxes, fontsize=9)

fig.tight_layout()
plt.show()

fig_path = f"../../figures"
if not os.path.exists(fig_path):
    os.makedirs(fig_path)
fig.savefig(f'{fig_path}/Figure_S21.pdf', bbox_inches='tight')
print(f'saved {fig_path}/Figure_S21.pdf')
